# Know Your Images
## Exploring and Preparing the Wildwood Farm Park Image Dataset

Simone Bonato & Susanne Suter, FHNW

Before training a deep learning model, you need to understand the data it will learn from. As Andrej Karpathy puts it, "become one with the data."

In computer vision, this means looking at your images - not just their labels or statistics. Explore the dataset, inspect representative examples, and look for unusual patterns, class imbalances, labeling errors, and differences in image quality.

Understanding your images helps you make informed decisions about preprocessing, data augmentation, model selection, and evaluation. It also helps you identify potential problems before they affect model performance.

**Your goal:** Get to know your dataset well enough to explain what makes the classification task easy or difficult - and what your model will need to learn.

*Inspired by Andrej Karpathy's [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/), Step 1: Become one with the data.*


**A note about this notebook**

This notebook was developed with AI assistance and subsequently reviewed, edited, and extended by the authors. While we have made every effort to ensure its accuracy, not every detail could be thoroughly verified.

If you encounter errors, unexpected behavior, or anything that seems incorrect or confusing, please let us know. Your feedback is especially valuable as this notebook is being used for the first time.

In [ ]:

# @title Learning Objectives

from IPython.display import HTML

HTML(r"""
<div class="callout callout-banner"
     style="background:#E5E7EB;color:#374151;border-left:6px solid #6B7280;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<div style="font-size:22px;font-weight:bold;line-height:1.3;">
Learning Objectives: Know Your Images
</div>

<p style="margin:6px 0 0 0;">
<strong>By the end of this notebook, you will be able to:</strong>
</p>

<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">
Explore and analyze image datasets qualitatively and quantitatively,
and assess their suitability for a given task.
</li>
<li style="margin:2px 0;">
Identify and assess common dataset issues, including duplicates,
incorrect labels, and invalid annotations.
</li>
<li style="margin:2px 0;">
Select and justify appropriate data splits and evaluation metrics,
considering class imbalance, object size, and data leakage across
computer vision tasks.
</li>
</ol>

<p style="margin:8px 0 0 0;">
Activities are marked 🟢 core · 🔴 optional
</p>

</div>
""")

**What Do You Need to Know for the Exam and Your Project?**

Focus on understanding the key concepts and applying them to new datasets. You should be able to explore and analyze your own image dataset, identify relevant characteristics and potential problems, interpret observations, and justify decisions about data preparation and evaluation.

You do **not** need to memorize specific dataset statistics, thresholds, code implementations, or individual examples. The activities help you develop the competencies needed for both the exam and your own project.

In [ ]:

# @title How to Use This Notebook
from IPython.display import HTML

HTML(r"""
<div class="callout" style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">How to Use This Notebook</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">This notebook is built around interactive activities. Explore the images, experiment with visualizations, and reflect on your observations.</li>
<li style="margin:2px 0;">The notebook runs in Google Colab. A CPU is sufficient for most activities, although computing image embeddings can take longer without a GPU.</li>
<li style="margin:2px 0;">A separate notebook guides you through exploring your own project dataset.</li>
</ul>
</div>
""")

### Setup


> ▶️ **Run the next three cells, in order.**
>
> Continue reading while they execute. The last cell downloads the Play Dataset (~115 MB), which may take 1–2 minutes on the first run. Subsequent runs use the cache.


In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import os
import random
import sys
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt  # noqa: F401  (every later section plots with plt)
import numpy as np
from PIL import Image as _Image

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated tests: tiny data, CPU, seconds. Never set this by hand.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:  # Jupytext turns the commented magic below into a real `%pip install -q imagehash` (N-021)
    pass
    %pip install -q imagehash

# --- Theme (SPEC L-003): white background, dark teal text, light grid, one colour per class everywhere ---
CLASSES = ["cat", "dog", "bird", "horse", "sheep", "zebra"]
CLASS_COLORS = {
    "cat": "#E07A5F",
    "dog": "#3D405B",
    "bird": "#F2CC8F",
    "horse": "#81B29A",
    "sheep": "#9C89B8",
    "zebra": "#2A9D8F",
}
INK = "#21434A"
mpl.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "text.color": INK,
        "axes.labelcolor": INK,
        "axes.edgecolor": INK,
        "axes.titlecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.grid": True,
        "grid.color": "#DDE5E7",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": mpl.cycler(color=[CLASS_COLORS[c] for c in CLASSES]),
        "figure.dpi": 100,
        "font.size": 11,
    }
)

# --- Results every section writes into (SPEC §9) ---
RESULTS = {}
RESULTS["class_colors"] = dict(CLASS_COLORS)


# --- use(): run a learner's function next to the reference solution (N-029) ---
def _is_stub(fn) -> bool:
    """True if fn's body is only a docstring plus `...` or `pass` (the learner has not written it yet)."""
    try:
        node = ast.parse(textwrap.dedent(inspect.getsource(fn))).body[0]
        body = node.body
    except (OSError, TypeError, SyntaxError, IndexError, AttributeError):
        return False
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, "value", None), str):
        body = body[1:]  # skip the docstring
    return all(
        isinstance(n, ast.Pass) or (isinstance(n, ast.Expr) and getattr(n.value, "value", None) is Ellipsis)
        for n in body
    )


def _same(a, b) -> bool:
    if hasattr(a, "equals"):  # pandas
        return bool(a.equals(b))
    try:
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            return bool(np.array_equal(a, b))
        return bool(a == b)
    except Exception:
        return False


def _short(x, n=60) -> str:
    s = repr(x)
    return s if len(s) <= n else s[: n - 1] + "…"


def _diff(mine, ref) -> str:
    """One short line on how `mine` differs from `ref` (e.g. 'cat: 12 vs 14')."""
    if isinstance(mine, dict) and isinstance(ref, dict):
        keys = list(ref) + [k for k in mine if k not in ref]
        bad = [k for k in keys if not _same(mine.get(k, "—"), ref.get(k, "—"))]
        return ", ".join(f"{k}: {_short(mine.get(k, '—'))} vs {_short(ref.get(k, '—'))}" for k in bad[:4])
    if isinstance(mine, (list, tuple, set)) and isinstance(ref, (list, tuple, set)):
        try:
            extra, missing = set(mine) - set(ref), set(ref) - set(mine)
            parts = []
            if missing:
                parts.append(f"missing {_short(sorted(missing, key=str))}")
            if extra:
                parts.append(f"extra {_short(sorted(extra, key=str))}")
            return " · ".join(parts) or f"same items, different order or length ({len(mine)} vs {len(ref)})"
        except TypeError:
            pass
    return f"{_short(mine)} vs {_short(ref)}"


def use(fn, *args, reference, **kwargs):
    """Run `fn(*args, **kwargs)` next to `reference(*args, **kwargs)`; ALWAYS return the reference's result.

    - stub (body is `...`) or NotImplementedError → "using the reference solution";
    - otherwise ✅ when both agree, ❌ with a short diff when they do not.
    The notebook continues with the reference either way, so "Run all" never breaks on your code.
    """
    expected = reference(*args, **kwargs)
    name = getattr(fn, "__name__", "your function")
    if _is_stub(fn):
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    try:
        mine = fn(*args, **kwargs)
    except NotImplementedError:
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    except Exception as err:  # a learner's bug must not stop the notebook
        print(f"❌ {name} raised {type(err).__name__}: {err} · using the reference solution.")
        return expected
    if _same(mine, expected):
        print(f"✅ {name} matches the reference solution.")
    else:
        print(f"❌ {name} differs from the reference: {_diff(mine, expected)} · using the reference.")
    return expected


# --- answer_box(): a worked answer computed at runtime, closed until opened (N-024, SPEC L-005) ---
ANSWER_BG, ANSWER_INK = "#F6EFE4", "#5C4027"  # the build's `answer` callout colours


def answer_box(title, lines=(), fig=None, table=None):
    """Draw a collapsed 💡 toggle holding printed lines, a DataFrame and/or matplotlib figure(s)."""
    import base64
    import html
    import io

    from IPython.display import HTML, display

    parts = []
    if lines:
        text = html.escape("\n".join(lines))
        parts.append(f'<pre style="white-space:pre-wrap;margin:6px 0 0 0;font-size:0.92em;">{text}</pre>')
    if table is not None:
        parts.append(table.to_html(border=0))
    for f in fig if isinstance(fig, (list, tuple)) else [fig]:
        if f is None:
            continue
        buf = io.BytesIO()
        f.savefig(buf, format="png", bbox_inches="tight")
        plt.close(f)
        png = base64.b64encode(buf.getvalue()).decode()
        parts.append(f'<img src="data:image/png;base64,{png}" style="max-width:100%;margin-top:6px;">')
    box = (f"background:{ANSWER_BG};color:{ANSWER_INK};border-left:6px solid {ANSWER_INK};border-radius:6px;"
           "padding:12px 14px;line-height:1.5;font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;")
    display(HTML(f'<div style="{box}"><details style="margin:0;"><summary style="cursor:pointer;'
                 f'font-weight:bold;">💡 {html.escape(title)}</summary>{"".join(parts)}</details></div>'))


print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · seed: {SEED} · imagehash: ", end="")
try:
    import imagehash  # noqa: F401

    print("ok")
except ImportError:
    print("missing (run this cell again)")

In [ ]:
# @title Data helpers (download with cache, build the tables) { display-mode: "form" }
import json
import urllib.error
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

# Where the committed manifests live. In Colab they are fetched from the repository; locally they are read
# from ./data. ⬇️ DATA_BASE_URL: change this if the repository moves or you work on a fork.
DATA_BASE_URL = "https://raw.githubusercontent.com/FHNW-VISED/eda-teaching-notebook/main/data/"
DATA_DIR = Path("data")
IMAGE_CACHE = DATA_DIR / "cache" / "coco"
SMOKE_IMAGES_PER_CLASS = 6


def _github_headers() -> dict:
    """Authorization header from the Colab secret GITHUB_TOKEN, if the repository is private (SPEC §5)."""
    if not IN_COLAB:
        return {}
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return {"Authorization": f"token {token}"} if token else {}
    except Exception:
        return {}


def download_file(url: str, dest: Path, headers: dict | None = None, attempts: int = 3) -> Path | None:
    """Download `url` to `dest` once; a half-finished download is written to `.part` first (N-019)."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(attempts):
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as fh:
                fh.write(resp.read())
            part.rename(dest)
            return dest
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == attempts - 1:
                part.unlink(missing_ok=True)
                return None
    return None


def fetch_manifest(name: str) -> Path:
    """Return the local path of a committed manifest, downloading it from DATA_BASE_URL if needed."""
    path = DATA_DIR / name
    if not path.exists():
        if download_file(DATA_BASE_URL + name, path, headers=_github_headers()) is None:
            raise RuntimeError(
                f"Could not download {name} from {DATA_BASE_URL}. Check DATA_BASE_URL, or add a GITHUB_TOKEN "
                "secret in Colab if the repository is private."
            )
    return path


def download_images(rows: list[dict], workers: int = 16) -> dict[int, Path]:
    """Download every image in `rows` ('id', 'coco_url', 'file_name') into IMAGE_CACHE; one progress line."""
    done, paths = 0, {}

    def one(row):
        return row["id"], download_file(row["coco_url"], IMAGE_CACHE / row["file_name"])

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for image_id, path in pool.map(one, rows):
            done += 1
            if path is not None:
                paths[image_id] = path
            print(f"\r  images ready: {done}/{len(rows)}", end="", flush=True)
    print()
    return paths


def build_tables(coco: dict, paths: dict[int, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Tidy tables from a COCO-format dict: `images` (one row per photo), `annotations` (one per animal)."""
    cat_name = {c["id"]: c["name"] for c in coco["categories"]}
    lic_name = {lic["id"]: lic["name"] for lic in coco.get("licenses", [])}
    anns = pd.DataFrame(
        {
            "id": a["id"],
            "image_id": a["image_id"],
            "class": cat_name[a["category_id"]],
            "bbox": a["bbox"],
            "area": a["area"],
            "segmentation": a["segmentation"],
            "iscrowd": a["iscrowd"],
        }
        for a in coco["annotations"]
        if a["image_id"] in paths
    )
    per_image_class = anns.groupby("image_id")["class"].first()
    n_animals = anns.groupby("image_id").size()
    imgs = pd.DataFrame(
        {
            "id": im["id"],
            "file": im["file_name"],
            "path": str(paths[im["id"]]),
            "width": im["width"],
            "height": im["height"],
            "class": per_image_class.get(im["id"]),
            "n_animals": int(n_animals.get(im["id"], 0)),
            "license": lic_name.get(im["license"], str(im["license"])),
        }
        for im in coco["images"]
        if im["id"] in paths
    )
    imgs = imgs.sort_values("id").reset_index(drop=True)
    anns = anns.sort_values("id").reset_index(drop=True)
    return imgs, anns

In [ ]:
# @title Load the manifest, download the images, build the tables { display-mode: "form" }
COCO_RAW = json.loads(fetch_manifest("coco_animals_subset.json").read_text())
_cat_name = {c["id"]: c["name"] for c in COCO_RAW["categories"]}
_image_class = {a["image_id"]: _cat_name[a["category_id"]] for a in COCO_RAW["annotations"]}

_rows = COCO_RAW["images"]
if SMOKE:  # same code path, fewer images (N-014)
    _rng = random.Random(SEED)
    _rows = []
    for c in CLASSES:
        _of_class = [im for im in COCO_RAW["images"] if _image_class[im["id"]] == c]
        _rows += sorted(_rng.sample(_of_class, SMOKE_IMAGES_PER_CLASS), key=lambda im: im["id"])

print(f"{len(_rows)} images to fetch from images.cocodataset.org (cached in {IMAGE_CACHE}/)")
_paths = download_images(_rows)
if len(_paths) < len(_rows):
    print(f"  ⚠️ {len(_rows) - len(_paths)} images could not be downloaded and are left out; run again.")

images, annotations = build_tables(COCO_RAW, _paths)
RESULTS["n_images"] = int(len(images))

_counts = Counter(images["class"])
print(f"images: {len(images)} · animals (annotations): {len(annotations)}")
print("per class: " + " · ".join(f"{c} {_counts[c]}" for c in CLASSES))
images.drop(columns="license").head()  # the licence column stays in the table, not on screen (N-045)

### Optional Coding Exercises

Some parts of this notebook let you choose how to work:

- **Explore:** Run the provided code, experiment with it, and interpret the results.
- **Code it yourself:** Implement the function in the ✍️ cell and compare your result with the reference solution.

Both approaches let you continue through the notebook. If you choose to implement a function, run the reference cell first, then complete the ✍️ cell. An incomplete or incorrect implementation will not prevent you from continuing.


In [ ]:
# @title Reference solution — run this cell; open it only if you are stuck { display-mode: "form" }
def _ref_double(x):
    return 2 * x

In [ ]:
# ✍️ YOUR CODE
def double(x):
    """Return twice x.

    e.g. double(21) → 42
    """
    ...


result = use(double, 21, reference=_ref_double)
print("the notebook continues with:", result)

## Meet the Park: Explore Your Dataset

Wildwood Farm Park wants to monitor animals across several enclosures using fixed outdoor cameras. The park has sheep and horses in its paddocks, birds near the aviary pond, zebras in the savanna area, and cats and dogs around the visitor yard.

The park ranger wants to use computer vision for three different tasks:

In [ ]:

# @title Computer Vision Tasks
from IPython.display import HTML
HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Three Computer Vision Tasks</div>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Classification – Recognize:</strong> What animal is shown in the image? The park ranger wants to identify dogs entering the sheep paddock so they can intervene before the sheep are harmed.</li>
<li style="margin:2px 0;"><strong>Object Detection – Count:</strong> How many sheep, zebras, or birds are visible, and where are they located?</li>
<li style="margin:2px 0;"><strong>Segmentation – Monitor Space Usage:</strong> Which areas of the image are occupied by animals? The park ranger wants to understand how animals use the space in their enclosures.</li>
</ol>
</div>""")

The park does not have its own training data yet. Before collecting and annotating new images, the park ranger should ask: Does a suitable dataset already exist, and can we reuse it?

To investigate this question, we'll use the Wildwood animal dataset, a subset of the [COCO 2017 Dataset](https://cocodataset.org/) containing images of six animal classes: cats, dogs, sheep, horses, birds, and zebras.

Each image is assigned one animal class but may contain multiple animals or other objects.

We'll explore the images and annotations to understand the dataset and assess its suitability for the park's computer vision tasks.

The dataset is organized into two tables:

- `images`: One row per image, including its dimensions, animal class, and number of annotated animals (`n_animals`).
- `annotations`: One row per annotated animal, including its bounding box and segmentation mask.

In [ ]:

# @title 🧾 Karpathy's Recipe · Step 1 { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-recipe"
     style="background:#FFF8D6;color:#333333;border-left:6px solid #F5C400;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<div style="font-weight:bold;">🧾 Recipe · Step 1: Become one with the data</div>

<p style="margin:6px 0 0 0;"><em>
"The first step to training a neural net is to not touch any neural net code at all and instead begin by
thoroughly inspecting your data. This step is critical. I like to spend copious amount of time (measured in
units of hours) scanning through thousands of examples, understanding their distribution and looking for
patterns."
</em></p>

<p style="margin:6px 0 0 0;">
Andrej Karpathy,
<a href="https://karpathy.github.io/2019/04/25/recipe/"
   target="_blank" rel="noopener"
   style="color:inherit;text-decoration:underline;">
A Recipe for Training Neural Networks</a> (2019).
</p>

</div>
""")

We won't spend hours exploring the data, but let's spend a few minutes getting to know our dataset. We'll start with *qualitative inspection* (looking at images and annotations), followed by *quantitative inspection* (examining statistics, distributions, and outliers).

### Qualitative Inspection

Explore the images to identify patterns, unusual examples, and potential data quality issues. Begin with an overview across classes, then examine one class at a time.

Each image has an assigned animal class, indicated by its frame color. Pay attention to:

- Backgrounds and lighting
- Viewpoints, distances, and animal sizes
- Other objects and potential distractions
- Whether the assigned class reflects the main content of the image

We'll examine the annotations in more detail afterward.


#### 🟢 Images: What Do Images Look Like Across Classes?

Let's start with a quick visual overview of the Wildwood animal dataset. The next cell displays six example images from each animal class to give you a first impression of the dataset.

**Observe:**
- How do the images differ across classes?
- How much variation is there within each class (e.g., backgrounds, viewpoints, animal sizes)?
- Which images might be challenging for an image classification model, and why?

In [ ]:
# @title 🔎 Sample example photos per class, framed in the class colour { display-mode: "form" }
_per_class = 6
fig, axes = plt.subplots(_per_class, len(CLASSES), figsize=(13, 2.2 * _per_class))
for j, c in enumerate(CLASSES):
    _of_class = images[images["class"] == c]
    _picked = _of_class.sample(min(_per_class, len(_of_class)), random_state=SEED)
    for i in range(_per_class):
        ax = axes[i, j]
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(CLASS_COLORS[c])
            spine.set_linewidth(4)
        if i < len(_picked):
            _thumb = _Image.open(_picked.iloc[i]["path"]).convert("RGB")
            _thumb.thumbnail((256, 256))
            ax.imshow(_thumb)
        if i == 0:
            ax.set_title(c, color=INK, fontweight="bold")
fig.tight_layout()
plt.show()

Select an animal class and browse at least twelve images to explore variation within the class. Pay attention to backgrounds, lighting, viewpoints, animal sizes, and other objects in the images. Look for recurring patterns and unusual examples. If an animal is difficult to spot, enable the bounding boxes.

**Observe:**
- How much do images vary within each class?
- Which image characteristics might make classification challenging?
- Which classes contain images that are least representative of the park's fixed outdoor cameras?

In [ ]:

# @title 🔎 Browse Images Within a Class { display-mode: "form" }
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as _Image

# Shuffle images once, reproducibly
_gallery = {
    c: images[images["class"] == c]
       .sample(frac=1, random_state=SEED)
       .reset_index(drop=True)
    for c in CLASSES
}

def show_gallery(animal_class, page, show_boxes):
    rows = _gallery[animal_class]
    picked = rows.iloc[page * 6 : (page + 1) * 6]

    fig, axes = plt.subplots(2, 3, figsize=(9, 4.5))

    for ax, (_, row) in zip(axes.flat, picked.iterrows()):
        photo = _Image.open(row["path"]).convert("RGB")
        ax.imshow(photo)

        if show_boxes:
            anns = annotations[annotations["image_id"] == row["id"]]
            for _, ann in anns.iterrows():
                x, y, w, h = ann["bbox"]
                ax.add_patch(
                    patches.Rectangle(
                        (x, y), w, h,
                        fill=False,
                        edgecolor=CLASS_COLORS[ann["class"]],
                        linewidth=2
                    )
                )

        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_color(CLASS_COLORS[animal_class])
            spine.set_linewidth(3)

    for ax in list(axes.flat)[len(picked):]:
        ax.axis("off")

    fig.suptitle(animal_class, color=INK, fontweight="bold")
    fig.tight_layout(pad=0.5)
    plt.show()
    plt.close(fig)


class_dropdown = widgets.Dropdown(
    options=CLASSES,
    description="Class:"
)

page_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    description="Group:",
    continuous_update=False
)

box_checkbox = widgets.Checkbox(
    value=False,
    description="Show bounding boxes",
    indent=False
)

def update_pages(change):
    n = len(_gallery[class_dropdown.value])
    page_slider.max = max(0, (n - 1) // 6)
    page_slider.value = 0

class_dropdown.observe(update_pages, names="value")
update_pages(None)

out = widgets.interactive_output(
    show_gallery,
    {
        "animal_class": class_dropdown,
        "page": page_slider,
        "show_boxes": box_checkbox
    }
)

display(
    widgets.HBox([class_dropdown, page_slider, box_checkbox]),
    out
)

#### 🟢 Document Your Observations

Karpathy's first step also involves documenting what you discover. Write at least three specific observations about the images you explored in the next markdown cell. For each observation, briefly explain what the park should investigate further.

**Hint:** Be specific rather than vague.

- Vague: "Dog images are different."
- Specific: "Many dog images are taken indoors, often with people in the background. This may not represent the park's outdoor cameras."

Consider backgrounds, lighting, animal sizes, distances, multiple animals, people, and unusual examples.

**My three observations** (double-click to edit):

1. …
2. …
3. …

In [ ]:

# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>

<p style="margin:6px 0 0 0;">
Example observations from different animal classes. Your findings may differ.
</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:4px 0;"><strong>Setting:</strong> Many dogs appear indoors or on streets, often photographed up close. The park's cameras will capture animals outdoors, sometimes far away.</li>
<li style="margin:4px 0;"><strong>Background shortcuts:</strong> People, hands, or leashes appear in many dog images. A model might associate these features with dogs rather than recognizing the animals themselves.</li>
<li style="margin:4px 0;"><strong>Unusual examples:</strong> Some images contain multiple dogs, costumes, or screens displaying animals. These examples may require closer inspection.</li>
<li style="margin:4px 0;"> <strong>Class boundaries:</strong> One image in the sheep class shows several ibexes. Although related to sheep, ibexes are different animals. This raises questions about how the class is defined.</li>
<li style="margin:4px 0;"><strong>Individual animals vs. groups:</strong> Some sheep are annotated individually, while others appear to be annotated as an entire flock. This may affect counting individual animals.</li>
<li style="margin:4px 0;"><strong>Segmentation quality:</strong> Some bird masks extend beyond the actual animal. Such inaccuracies may affect segmentation model training and evaluation.</li>
<li style="margin:4px 0;"><strong>Label relevance:</strong> Some images are labeled as birds even though the bird is tiny and a large bear dominates the image. This may confuse an image classification model.</li>
<li style="margin:4px 0;"><strong>Non-real animals:</strong> Some bird images show stuffed toys rather than living animals. These may not represent the park's intended use case.</li>
<li style="margin:4px 0;"><strong>Image style and incomplete annotations:</strong> Some horse images have an old photographic appearance, and some legs are missing from the annotations. This raises questions about image representativeness and annotation completeness.</li>
</ol>
<p style="margin:6px 0 0 0;">
Keep your observations in mind when evaluating your own dataset for the mini-project.
</p>

</details>
</div>
""")


#### 🟢 Dataset Suitability: How Suitable Are These Images for the Park?

The park's cameras are fixed, outdoors, and often far from the animals.

**Question:**
Based on the images you explored, which animal classes seem least representative of the park's camera views, and why? Briefly justify your answer.


In [ ]:

# @title 💡 Sample Answer
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample answer</summary>
<p style="margin:6px 0 0 0;">
Cats and dogs often appear in indoor, close-up photos with people or furniture,
making them less representative of the park's fixed outdoor cameras.
Sheep, horses, and zebras are more often shown outdoors and at greater distances,
making their images somewhat closer to the park's camera views.
</p>
</details>
</div>
""")


#### 🟢 Annotations: What Does an Image Annotation Look Like?

An image can have different types of annotations, such as class labels, bounding boxes, and segmentation masks.

Let's visualize an example from the Wildwood animal dataset and examine how these annotations represent different information about the animals.


**Explore the annotations**

Run the next cell to visualize an image with its class label, bounding boxes, and segmentation masks.

**Observe:**
- What information does each type of annotation provide?
- How do bounding boxes and segmentation masks differ?
- Do the annotations correctly represent the animals in the image?

In [ ]:

# @title 🔎 Explore Images and Annotations { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from pycocotools import mask as mask_utils


def decode_mask(segmentation, height, width):
    """Decode a COCO polygon or RLE segmentation into a binary mask."""
    if isinstance(segmentation, list):
        rle = mask_utils.merge(
            mask_utils.frPyObjects(segmentation, height, width)
        )
    elif isinstance(segmentation["counts"], list):
        rle = mask_utils.frPyObjects(segmentation, height, width)
    else:
        rle = segmentation

    return mask_utils.decode(rle).astype(bool)


# Shuffle images reproducibly
explore_images = images.sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

# Navigation widgets
previous_btn = widgets.Button(description="← Previous")
next_btn = widgets.Button(description="Next →")
counter = widgets.HTML()

image_number_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=len(explore_images),
    description="Image #:",
    layout=widgets.Layout(width="190px")
)
go_btn = widgets.Button(description="Go")

# Annotation visibility controls
show_boxes = widgets.Checkbox(
    value=True,
    description="Show bounding boxes",
    indent=False,
    layout=widgets.Layout(width="190px")
)
show_masks = widgets.Checkbox(
    value=True,
    description="Show segmentation masks",
    indent=False,
    layout=widgets.Layout(width="210px")
)

output = widgets.Output()
state = {"index": 0}


def show_image(index):
    row = explore_images.iloc[index]
    photo = np.array(Image.open(row["path"]).convert("RGB"))
    shown = photo.copy()

    anns = annotations[annotations["image_id"] == row["id"]]
    overlays = []

    for _, ann in anns.iterrows():
        color = CLASS_COLORS[ann["class"]]

        # Decode and overlay masks only when enabled
        if show_masks.value:
            mask = decode_mask(
                ann["segmentation"],
                int(row["height"]),
                int(row["width"])
            )

            rgb = np.array([
                int(color[i:i+2], 16) for i in (1, 3, 5)
            ])

            shown[mask] = (
                0.45 * shown[mask] + 0.55 * rgb
            ).astype(np.uint8)

        overlays.append((ann, color))

    with output:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.imshow(shown)

        # Draw bounding boxes only when enabled
        if show_boxes.value:
            for ann, color in overlays:
                x, y, w, h = ann["bbox"]
                ax.add_patch(
                    patches.Rectangle(
                        (x, y), w, h,
                        fill=False,
                        edgecolor=color,
                        linewidth=2.5
                    )
                )

        ax.set_title(
            f"Class: {row['class']} | "
            f"Annotated animals: {len(anns)}"
        )
        ax.axis("off")
        plt.show()
        plt.close(fig)

        print(
            f"Image number: {index + 1} | "
            f"Size: {row['width']} × {row['height']} px"
        )

        for ann, _ in overlays:
            print(
                f"{ann['class']}: "
                f"box = {[round(v) for v in ann['bbox']]} px"
            )

    # Update navigation controls
    counter.value = (
        f"<b>Image {index + 1} / {len(explore_images)}</b>"
    )
    image_number_input.value = index + 1

    previous_btn.disabled = index == 0
    next_btn.disabled = index == len(explore_images) - 1


def navigate(change):
    state["index"] += change
    show_image(state["index"])


def go_to_image(_):
    state["index"] = image_number_input.value - 1
    show_image(state["index"])


previous_btn.on_click(lambda _: navigate(-1))
next_btn.on_click(lambda _: navigate(1))
go_btn.on_click(go_to_image)

# Refresh the image immediately when a checkbox changes
show_boxes.observe(
    lambda change: show_image(state["index"]),
    names="value"
)
show_masks.observe(
    lambda change: show_image(state["index"]),
    names="value"
)

# Display interface
display(
    widgets.HBox([previous_btn, counter, next_btn]),
    widgets.HBox([image_number_input, go_btn]),
    widgets.HBox([show_boxes, show_masks]),
    output
)

show_image(0)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;">

<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<p style="margin:6px 0 0 0;">
<strong>Class labels</strong> identify the animal category.
<strong>Bounding boxes</strong> indicate where animals are located using rectangles,
while <strong>segmentation masks</strong> outline their shapes more precisely.
Annotations are not always accurate: some masks extend beyond the animal,
some body parts are missing, and groups of animals may be annotated inconsistently.
</p>
</details>
</div>
""")

#### 🔴 Image Classification: One Label per Image

Image classification assigns one label to an entire image, even if it contains multiple animals or different classes. What information is lost, and when might this become a problem?

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;">

<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<p style="margin:6px 0 0 0;">
A single class label does not indicate how many animals are present,
where they are located, or whether other classes appear in the image.
The labeled animal may also be very small or in the background,
while a much larger object dominates the image.
This can make classification difficult because the label may not reflect
the most prominent image content. Image classification is also unsuitable
when the goal is to locate or count individual animals.
</p>
</details>
</div>
""")

### Quantitative Inspection

Now that we've explored individual images and annotations, let's examine the dataset as a whole.

We'll use statistics and visualizations to investigate class distributions, image properties, and outliers that may reveal potential data quality issues.

In [ ]:
# @title 🧾 Recipe · Step 1, continued { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-recipe"
     style="background:#FFF8D6;color:#333333;border-left:6px solid #F5C400;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<div style="font-weight:bold;">🧾 Recipe · Step 1, continued: Search, Filter, Sort</div>

<p style="margin:6px 0 0 0;"><em>
"Once you get a qualitative sense it is also a good idea to write some simple code to search/filter/sort
by whatever you can think of (e.g. type of label, size of annotations, number of annotations, etc.) and
visualize their distributions and the outliers along any axis. The outliers especially almost always uncover
some bugs in data quality or preprocessing."
</em></p>

<p style="margin:6px 0 0 0;">
Andrej Karpathy,
<a href="https://karpathy.github.io/2019/04/25/recipe/"
   target="_blank" rel="noopener"
   style="color:inherit;text-decoration:underline;">A Recipe for Training Neural Networks</a> (2019).
</p>

</div>
""")

In [ ]:
# @title 📘 Image Properties: What Can We Measure? { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-theory"
     style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<div style="font-weight:bold;">📘 Image Properties: What Can We Measure?</div>

<p style="margin:6px 0;">
We can measure image properties to identify patterns, unusual values, and potential data quality issues.
These measurements do not require class labels and can be applied to any image dataset.
</p>

<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:4px 0;"><strong>Dimensions:</strong> Width and height in pixels, and aspect ratio (width ÷ height).</li>
<li style="margin:4px 0;"><strong>Color mode:</strong> How colors are stored, such as RGB (three channels) or grayscale (one channel).</li>
<li style="margin:4px 0;"><strong>Brightness:</strong> Average pixel intensity, from 0 (black) to 255 (white).</li>
<li style="margin:4px 0;"><strong>Contrast:</strong> Variation in pixel intensity, indicating differences between dark and bright regions.</li>
<li style="margin:4px 0;"><strong>Colorfulness:</strong> How strongly image colors differ from gray.</li>
</ul>

</div>
""")

The next code cells calculate these properties for each image and store the results in a table called `image_stats`.

In [ ]:
# @title Measurement helpers { display-mode: "form" }
from matplotlib.patches import Rectangle
from PIL import ImageDraw

HIST_BINS = 32
CHANNEL_COLORS = {"R": "#C0392B", "G": "#27AE60", "B": "#2980B9"}
COLOUR_MODES = ["RGB", "L", "RGBA", "CMYK"]  # the modes we always report, even when the count is 0


def load_thumb(path, size=(256, 256)):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail(size)
        return im.copy()


def frame(ax, color):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(color)
        spine.set_linewidth(3)

def measure_image(path, thumb: int) -> tuple[dict, np.ndarray]:
    """Open one photo and return (stats, channel_hist).

    stats: width, height, mode (as stored in the file), aspect (width / height), long_side, megapixels,
    brightness (mean grey level 0–255 on a thumbnail), contrast (its standard deviation), colourfulness
    (Hasler & Süsstrunk 2003: 0 for a grayscale picture), mean_r / mean_g / mean_b.
    channel_hist: a (3, HIST_BINS) array, the density histogram of the R, G and B channels.
    """
    with Image.open(path) as im:
        mode, (w, h) = im.mode, im.size  # read these before decoding: they describe the file itself
        im.draft("RGB", (thumb, thumb))  # fast reduced-size JPEG decode (no effect on other formats)
        rgb = im.convert("RGB")
        rgb.thumbnail((thumb, thumb))
        arr = np.asarray(rgb, dtype=np.float32)
    grey = arr.mean(axis=2)
    channels = arr.reshape(-1, 3)
    rg = channels[:, 0] - channels[:, 1]  # red minus green
    yb = (channels[:, 0] + channels[:, 1]) / 2 - channels[:, 2]  # yellow (mean of R and G) minus blue
    hist = np.stack(
        [np.histogram(channels[:, k], bins=HIST_BINS, range=(0, 256), density=True)[0] for k in range(3)]
    )
    stats = {
        "width": w,
        "height": h,
        "mode": mode,
        "aspect": w / h,
        "long_side": max(w, h),
        "megapixels": w * h / 1e6,
        "brightness": float(grey.mean()),
        "contrast": float(grey.std()),
        "colourfulness": float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())),
        "mean_r": float(channels[:, 0].mean()),
        "mean_g": float(channels[:, 1].mean()),
        "mean_b": float(channels[:, 2].mean()),
    }
    return stats, hist


def show_rows(rows_df: pd.DataFrame, title: str, caption=None, cols: int = 6) -> None:
    """A small gallery of the photos in `rows_df` (rows of image_stats), each framed in its class colour.

    `caption(row) -> str` writes the title above each photo; by default the size and the colour mode.
    """
    if caption is None:

        def caption(row):
            return f"{row['class']} · {row['width']}×{row['height']} {row['mode']}"

    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.4 * cols, 2.7 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row), fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def _fmt(value) -> str:
    """A number for a caption: 640 stays 640, 1.33333 becomes 1.333."""
    return f"{value:.4g}"


def _closest_to_median(stat: str):
    """The row of `image_stats` whose value of `stat` is closest to the median (the 'typical' photo)."""
    col = image_stats[stat]
    return image_stats.iloc[int((col - col.median()).abs().to_numpy().argmin())]


def _photo(ax, row, caption: str) -> None:
    """Draw one photo with a caption, framed in its class colour."""
    ax.axis("on")
    ax.imshow(load_thumb(row["path"]))
    frame(ax, CLASS_COLORS[row["class"]])
    ax.set_title(caption, fontsize=8)


def show_extremes(stat: str, n: int = 4) -> None:
    """The `n` photos with the lowest value of `stat`, the median ('average') photo, and the `n` highest.

    `stat` is a numeric column of `image_stats` (e.g. "brightness", "aspect", "megapixels", "colourfulness").
    Three rows: lowest (the minimum first), the typical photo in the middle, highest (the maximum first).
    """
    numeric = [c for c in image_stats.columns if pd.api.types.is_numeric_dtype(image_stats[c]) and c != "id"]
    if stat not in numeric:
        raise SystemExit(f"stat must be one of {numeric}, not {stat!r}")
    ordered = image_stats.sort_values(stat)
    n = max(1, min(n, len(ordered) // 2))

    def caption(tag, row):
        size = f"{row['width']}×{row['height']} {row['mode']}"
        return f"{tag}: {stat} = {_fmt(row[stat])}\n{row['class']} · {size}"

    fig, axes = plt.subplots(3, n, figsize=(2.4 * n, 2.8 * 3), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes[0], ordered.head(n).iterrows(), strict=False):
        _photo(ax, row, caption("lowest", row))
    _photo(axes[1, n // 2], _closest_to_median(stat), caption("median", _closest_to_median(stat)))
    for ax, (_, row) in zip(axes[2], ordered.tail(n)[::-1].iterrows(), strict=False):
        _photo(ax, row, caption("highest", row))
    fig.suptitle(f"{stat}: the {n} lowest, the typical (median) photo, the {n} highest", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_min_median_max(stats: list[str]) -> None:
    """Show min, median, and max images for each property, with headings above."""

    spec = []
    for stat in stats:
        items = [
            ("Min", image_stats.loc[image_stats[stat].idxmin()]),
            ("Median", _closest_to_median(stat)),
            ("Max", image_stats.loc[image_stats[stat].idxmax()]),
        ]
        spec.append((stat, items))

    n_rows = -(-len(spec) // 2)
    fig = plt.figure(figsize=(13, 2.8 * n_rows))
    outer = fig.add_gridspec(n_rows, 2, hspace=0.65, wspace=0.25)

    for k, (stat, items) in enumerate(spec):
        r, c = divmod(k, 2)
        inner = outer[r, c].subgridspec(1, 3, wspace=0.15)

        for j, (tag, row) in enumerate(items):
            ax = fig.add_subplot(inner[0, j])
            _photo(ax, row, f"{tag}: {_fmt(row[stat])}\n{row['class']}")

        # Property heading centered above the three images
        x0 = outer[r, c].get_position(fig).x0
        x1 = outer[r, c].get_position(fig).x1
        y1 = outer[r, c].get_position(fig).y1

        fig.text((x0 + x1) / 2, y1 + 0.025,
                 stat.replace("_", " ").title(),
                 ha="center", va="bottom",
                 fontsize=12, fontweight="bold", color=INK)

    plt.show()
    plt.close(fig)



In [ ]:
# @title Measure Image Dimensions and Other Properties { display-mode: "form" }
# (a few seconds: each JPEG is decoded at thumbnail size)
THUMB = 64 if SMOKE else 128  # smaller thumbnails in smoke mode, same code (N-014)

_records, _hists = [], []
for _, _row in images.iterrows():
    _stats, _hist = measure_image(_row["path"], THUMB)
    _records.append({"id": _row["id"], "class": _row["class"], "path": _row["path"], **_stats})
    _hists.append(_hist)
image_stats = pd.DataFrame(_records)
CHANNEL_HISTS = np.stack(_hists)  # (n_images, 3, HIST_BINS)
RESULTS["image_stats_n"] = int(len(image_stats))

_same_size = image_stats[["width", "height"]].to_numpy() == images[["width", "height"]].to_numpy()
_agree = int(_same_size.all(axis=1).sum())
print(f"measured {len(image_stats)} photos · the manifest's width/height matches the file for {_agree}")
STAT_COLUMNS = ["width", "height", "aspect", "megapixels", "brightness", "contrast", "colourfulness"]
image_stats[STAT_COLUMNS].agg(["min", "median", "mean", "max"]).T.round(2)

#### 🟢 Image Properties: What Do the Statistics Reveal?

The next visualization shows images with the minimum, median, and maximum values for each property.

**Observe:**
- How do images with minimum and maximum values differ from those near the median?
- Which extreme values might indicate unusual images or potential data quality issues?

In [ ]:
# @title 🔎 Inspect Images with Minimum, Median, and Maximum Property Values { display-mode: "form" }
show_min_median_max(STAT_COLUMNS)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>

<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:4px 0;"><strong>Image dimensions:</strong> Most images are relatively small, with widths up to 640 pixels. Some have unusual aspect ratios, such as a very wide zebra image.</li>
<li style="margin:4px 0;"><strong>Brightness:</strong> Images range from very dark to very bright. Animals may be harder to recognize in extreme lighting conditions.</li>
<li style="margin:4px 0;"><strong>Potential issues:</strong> Extreme aspect ratios and lighting conditions deserve closer inspection. They are not necessarily errors, but may affect model performance.</li>
</ul>

</details>
</div>
""")

#### 🟢 Image Dimensions: What Do Their Distributions Reveal?

We've examined individual images and summary statistics. Now let's visualize how image dimensions and aspect ratios vary across the dataset.

The COCO images come from many different photographers, while the park will use a few fixed cameras.

**Observe:**
- What patterns do you see in the width-versus-height scatter plot?
- How do aspect ratios differ between animal classes?
- What might these differences mean when preparing images for a computer vision model?

In [ ]:
# @title 🔎 Understanding Width–Height Scatter Plot Patterns { display-mode: "form" }
# Three tiny example scatters of width (across) against height (up), to pick from.
_rng = np.random.default_rng(SEED)
_line_w = _rng.uniform(300, 640, 40)
_examples = {
    "one point\nall photos the same size": (np.full(40, 640.0), np.full(40, 480.0)),
    "a line\nsizes differ, same shape": (_line_w, _line_w * 0.75),
    "a cloud\nany size, any shape": (_rng.uniform(200, 640, 40), _rng.uniform(200, 640, 40)),
}
fig, axes = plt.subplots(1, 3, figsize=(8.4, 2.6))
for ax, (title, (xs, ys)) in zip(axes, _examples.items(), strict=True):
    ax.scatter(xs, ys, s=14, color=INK, alpha=0.7, edgecolor="none")
    ax.set_xlim(0, 700)
    ax.set_ylim(0, 700)
    ax.set_xlabel("width (px)", fontsize=8)
    ax.set_ylabel("height (px)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title(title, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 🔎 Explore Image Dimensions and Aspect Ratios by Class { display-mode: "form" }
# Left: every photo as a dot (width across, height up) in its class colour; the dashed line is a square
# photo. Right: the aspect ratio width/height, stacked by class.
fig, (ax_sc, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
for c in CLASSES:
    s = image_stats[image_stats["class"] == c]
    ax_sc.scatter(s["width"], s["height"], s=22, alpha=0.65, color=CLASS_COLORS[c], label=c, edgecolor="none")
_lim = float(max(image_stats["width"].max(), image_stats["height"].max())) * 1.08
ax_sc.plot([0, _lim], [0, _lim], "--", color=INK, lw=1)
ax_sc.set_xlim(0, _lim)
ax_sc.set_ylim(0, _lim)
ax_sc.set_xlabel("width (px)")
ax_sc.set_ylabel("height (px)")
ax_sc.set_title("Image sizes")
ax_sc.legend(frameon=False, fontsize=9, markerscale=1.6)

_bins = np.linspace(image_stats["aspect"].min() * 0.98, image_stats["aspect"].max() * 1.02, 28)
ax_hist.hist(
    [image_stats.loc[image_stats["class"] == c, "aspect"] for c in CLASSES],
    bins=_bins,
    stacked=True,
    color=[CLASS_COLORS[c] for c in CLASSES],
    label=CLASSES,
)
for ratio, name in [(3 / 4, "3:4"), (1.0, "1:1"), (4 / 3, "4:3")]:
    ax_hist.axvline(ratio, color=INK, lw=0.8, ls=":")
    ax_hist.text(ratio, ax_hist.get_ylim()[1] * 0.97, name, ha="center", va="top", fontsize=8, color=INK)
ax_hist.set_xlabel("aspect ratio = width / height  (left of 1: portrait, right: landscape)")
ax_hist.set_ylabel("photos")
ax_hist.set_title("Aspect ratios")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Image dimensions:</strong> The scatter plot shows horizontal and vertical bands around 500 and 640 pixels. Many images share the same width or height, suggesting standardized image sizes rather than completely random dimensions.</li>
<li><strong>Aspect ratios:</strong> Images include both landscape and portrait orientations, with common ratios around 4:3 and 3:4. The animal classes overlap considerably, although their distributions may differ.</li>
<li><strong>Implications:</strong> Images may require resizing or padding before training. At the park, fixed cameras could produce characteristic image sizes. If these correlate with animal classes, a model might learn camera-specific shortcuts instead of recognizing the animals.</li>
</ul>
</details>
</div>
""")

#### 🟢 Color Modes and Colors: What Do They Reveal?

Images can be stored in different color modes, such as RGB (three channels) or grayscale (one channel). However, an RGB image can also appear grayscale if its color channels contain similar values.

The next visualization shows the stored color modes, examples of images that appear grayscale, and RGB pixel-intensity distributions for each animal class.

**Observe:**
- Which color modes occur in the dataset? Do all images have the same number of channels?
- How many images appear grayscale, and are they necessarily stored in grayscale mode?
- What differences do you notice between the RGB distributions of animal classes?
- Why might these differences matter when preparing images or training a model?

**Go Deeper (optional):**
- What do the shaded bands (25th–75th percentile) reveal about variation within each class?
- Could class-specific color patterns become misleading shortcuts for a model?

In [ ]:
# @title 🔎 Explore Color Modes, Grayscale Images, and RGB Distributions { display-mode: "form" }
# Count color modes, show grayscale-looking images, and compare RGB distributions across classes.
_modes = COLOUR_MODES + sorted(set(image_stats["mode"]) - set(COLOUR_MODES))
mode_by_class = pd.crosstab(image_stats["mode"], image_stats["class"])
mode_by_class = mode_by_class.reindex(index=_modes, columns=CLASSES, fill_value=0)

image_stats["looks_gray"] = (image_stats["mode"] == "L") | (image_stats["colourfulness"] < 2.0)
print("color modes: " + " · ".join(f"{m} {int(mode_by_class.loc[m].sum())}" for m in _modes))
_n_gray = int(image_stats["looks_gray"].sum())
print(f"photos that look grayscale (mode L, or RGB with colorfulness below 2): {_n_gray}")

N_GRAY = 6 if SMOKE else 12
show_rows(image_stats[image_stats["looks_gray"]].head(N_GRAY), "photos that look grayscale")

fig = plt.figure(figsize=(13, 4.6))
_gs = fig.add_gridspec(
    2, 4, width_ratios=[1.4, 1, 1, 1],
    hspace=0.65, wspace=0.3,
    left=0.05, right=0.99, top=0.86, bottom=0.12
)

ax_modes = fig.add_subplot(_gs[:, 0])
_bottom = np.zeros(len(_modes))

for c in CLASSES:
    ax_modes.bar(
        _modes, mode_by_class[c].to_numpy(),
        bottom=_bottom, color=CLASS_COLORS[c], label=c
    )
    _bottom += mode_by_class[c].to_numpy()

ax_modes.set_ylabel("photos")
ax_modes.set_title("Color mode as stored in the file", fontsize=10)
ax_modes.legend(frameon=False, fontsize=8)

_edges = np.linspace(0, 256, HIST_BINS + 1)
_centres = (_edges[:-1] + _edges[1:]) / 2

for k, c in enumerate(CLASSES):
    ax = fig.add_subplot(_gs[k // 3, 1 + k % 3])

    # Per-image RGB histograms for this class
    _hists = CHANNEL_HISTS[(image_stats["class"] == c).to_numpy()]

    _mean_hist = _hists.mean(axis=0)
    _q25 = np.percentile(_hists, 25, axis=0)
    _q75 = np.percentile(_hists, 75, axis=0)

    for ch_idx, ch in enumerate("RGB"):
        ax.plot(
            _centres, _mean_hist[ch_idx],
            color=CHANNEL_COLORS[ch], lw=1.4, label=ch
        )
        ax.fill_between(
            _centres, _q25[ch_idx], _q75[ch_idx],
            color=CHANNEL_COLORS[ch], alpha=0.15,
            linewidth=0
        )

    ax.set_title(
        f"{c} ({len(_hists)} photos)",
        color=CLASS_COLORS[c], fontsize=10
    )
    ax.set_yticks([])
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=7)

    for spine in ax.spines.values():
        spine.set_edgecolor(CLASS_COLORS[c])
        spine.set_linewidth(1.6)

    if k == 0:
        ax.legend(frameon=False, fontsize=7, loc="upper left")

fig.text(
    0.67, 0.95,
    "Mean RGB histograms by class (shading: 25th–75th percentile)",
    ha="center", color=INK
)

display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Color modes:</strong> 637 images are RGB (three channels) and one is grayscale (L, one channel of brightness values from 0 to 255). Converting an L image to RGB duplicates its brightness values across three channels, ensuring consistent model inputs.</li>
<li><strong>Grayscale appearance:</strong> 17 images appear grayscale, but most are stored as RGB. Appearance and storage mode are not necessarily the same.</li>
<li><strong>RGB distributions:</strong> Histograms show how frequently different pixel intensities occur, but not where colors appear. The distributions differ between animal classes.</li>
<li><strong>Implications:</strong> Color differences may reflect backgrounds, lighting, or camera conditions rather than animal features. Consistent preprocessing and evaluation on new camera images are important.</li>
</ul>

<p style="margin:12px 0 4px;"><strong>🔎 Go Deeper (optional)</strong></p>
<ul style="margin:4px 0 0 0;padding-left:22px;">
<li><strong>Variability bands:</strong> The shaded areas show the 25th–75th percentile of pixel-intensity frequencies across images within each class. Wider bands indicate greater variation between images, while narrow bands indicate more consistent distributions.</li>
<li><strong>Potential shortcuts:</strong> A model might associate certain colors or backgrounds with particular animals instead of learning their visual features. Such shortcuts could fail when the animals are photographed in different environments, such as the park.</li>
</ul>
</details>
</div>
""")

#### 🟢 Image Properties: Are Extreme Values Problematic?

Extreme values can reveal unusual images or potential data quality issues.

**Explore:**
1. Select an image property from the dropdown.
2. Choose how many images to inspect and run the cell.
3. Try different properties and compare the results.

**Observe:**
- What makes these images unusual?
- Are they problematic or representative of realistic conditions?
- Which images deserve further investigation before training?
- Which images would you flag for further inspection or cleaning, and why?

In [ ]:
# @title 🔎 Explore Extreme Image Properties { display-mode: "form" }

STAT_TO_SHOW = "brightness" # @param ["brightness", "contrast", "colourfulness", "aspect", "megapixels", "long_side"]
N_IMAGES = 5 # @param {type:"slider", min:1, max:5, step:1}

show_extremes(STAT_TO_SHOW, N_IMAGES)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Extreme values:</strong> Very dark images may show animals in shadows or at dusk, while bright images may contain snow, sand, or overexposed regions. Unusual aspect ratios can indicate very wide or tall images.</li>
<li><strong>Data quality:</strong> Extreme values are not necessarily errors. Images showing realistic lighting or unusual viewpoints may be valuable for training.</li>
<li><strong>Further inspection:</strong> Uniform gray images, photos of screens, or images where the animal is barely recognizable deserve closer inspection. Depending on the use case, they may need cleaning, exclusion, or special handling.</li>
</ul>
</details>
</div>
""")

#### 🔴 Brightness: How Do Distributions Differ Between Classes? (Optional)

A violin plot shows the distribution of values within each class. Wider sections indicate where more images are concentrated. The horizontal dark line marks the median, while the vertical line and its endpoints show the minimum and maximum values.

**Observe:**
- Which animal classes have the highest and lowest median brightness?
- Which classes show the greatest variation in brightness?
- Do the distributions overlap, or are there clear differences between classes?

**Go Deeper:**
- Could differences in brightness become misleading shortcuts for a classification model?
- What can the violin plots reveal that the summary statistics alone cannot?

In [ ]:
# @title 🔎 Explore Brightness Distributions by Class { display-mode: "form" }
# Brightness per class as violin plots, in class colours. The dotted lines mark "very dark" and "very bright".
fig, ax = plt.subplots(figsize=(9, 3.8))
_data = [image_stats.loc[image_stats["class"] == c, "brightness"].to_numpy() for c in CLASSES]
_violins = ax.violinplot(_data, widths=0.8, showmedians=True)
for body, c in zip(_violins["bodies"], CLASSES, strict=True):
    body.set_facecolor(CLASS_COLORS[c])
    body.set_alpha(0.85)
_violins["cmedians"].set_color(INK)
for part in ("cmins", "cmaxes", "cbars"):
    _violins[part].set_color(INK)
ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES)
for level in (40, 215):
    ax.axhline(level, color=INK, lw=0.8, ls=":")
ax.set_ylim(0, 255)
ax.set_ylabel("brightness (mean grey level)")
ax.set_title("Brightness per class")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Median brightness:</strong> Some animal classes have higher median brightness than others, suggesting differences in typical lighting or backgrounds.</li>
<li><strong>Variation:</strong> Wider and taller violins indicate greater variation in brightness. Some classes contain both very dark and very bright images.</li>
<li><strong>Overlap:</strong> The distributions overlap substantially, so brightness alone cannot reliably distinguish animal classes.</li>
</ul>
<p style="margin:12px 0 4px;"><strong>🔎 Go Deeper (optional)</strong></p>
<ul style="margin:4px 0 0 0;padding-left:22px;">
<li><strong>Potential shortcuts:</strong> A model might associate certain brightness levels with particular animals instead of recognizing their visual features. This could reduce performance under different lighting conditions.</li>
<li><strong>Beyond summary statistics:</strong> Violin plots reveal where brightness values are concentrated, how widely they vary, and how much the classes overlap. Medians and ranges alone cannot show these patterns.</li>
</ul>
</details>
</div>
""")

#### 🟢 Class Distribution: Are All Animal Classes Equally Represented?

The number of images per class can influence how a classification model learns. Classes with fewer examples may be harder to recognize reliably.

The next visualization shows how many images belong to each animal class.

**Observe:**
- Which classes have the most and fewest images?
- How large are the differences between classes?
- Why might class imbalance matter when training and evaluating a classification model?

In [ ]:
# @title 🧾 Recipe · Step 1, continued: Imbalances and Biases { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div style="background:#FFF8D6;color:#333333;border-left:6px solid #F5C400;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;">
  <div style="font-weight:bold;margin-bottom:6px;">
    🧾 Recipe · Step 1, continued: Imbalances and Biases
  </div>
  <div style="font-style:italic;margin-bottom:6px;">
    “I look for data imbalances and biases.”
  </div>
  <div>
    Andrej Karpathy,
    <a href="https://karpathy.github.io/2019/04/25/recipe/"
       target="_blank" style="color:#333333;">
      A Recipe for Training Neural Networks
    </a> (2019).
  </div>
</div>
""")

In [ ]:
# @title 🔎 Explore Class Distribution { display-mode: "form" }
images_per_class = {c: int((images["class"] == c).sum()) for c in CLASSES}
RESULTS["images_per_class"] = images_per_class

_total = sum(images_per_class.values())
_largest = max(images_per_class, key=images_per_class.get)
_smallest = min(images_per_class, key=images_per_class.get)
_ratio = images_per_class[_largest] / max(1, images_per_class[_smallest])
print(f"{_total} photos · largest class {_largest} ({images_per_class[_largest]}) · smallest {_smallest} "
      f"({images_per_class[_smallest]}) · ratio {_ratio:.1f}×")

fig, ax = plt.subplots(figsize=(9, 3.8))
_order = sorted(CLASSES, key=images_per_class.get, reverse=True)
ax.bar(_order, [images_per_class[c] for c in _order], color=[CLASS_COLORS[c] for c in _order])
for i, c in enumerate(_order):
    ax.text(i, images_per_class[c], f"{100 * images_per_class[c] / _total:.0f} %", ha="center", va="bottom",
            fontsize=9)
ax.set_ylabel("photos")
ax.set_title("Images per class (one label per photo)")
ax.set_ylim(0, max(images_per_class.values()) * 1.15)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Class distribution:</strong> Cats and dogs have the most images (130 each), while sheep have the fewest (59). The largest classes contain about 2.2 times as many images as the smallest.</li>
<li><strong>Dataset bias:</strong> People may photograph pets more frequently than sheep. The distribution reflects the Flickr photo collection, not necessarily what the park's cameras will capture.</li>
<li><strong>Implications:</strong> A model may learn common classes better than underrepresented ones. Overall accuracy could hide poor performance on sheep, so performance should also be evaluated per class.</li>
<li><strong>Park use case:</strong> Dogs are common in the Play Dataset but may rarely enter the sheep paddock. Detecting these rare events reliably is important, even if dogs are well represented in the training data. Overall accuracy alone may not reflect this performance.</li>
</ul>
<p style="margin:8px 0 0;"><em>Note: The smaller smoke-test dataset may have a different, more balanced class distribution.</em></p>
</details>
</div>
""")

#### 🟢 Animals per Image: How Common Are Multiple Animals?

Each image in the Play Dataset has one class label, but it may contain several animals of that class. The dataset was selected from COCO to include only one of our six animal classes per image.

The next visualization shows the number of annotated animals per image and examples with particularly many animals.

**Observe:**
- What proportion of images contain more than one annotated animal?
- Which classes tend to have the most animals per image?
- What do you notice when inspecting images with many animals?

**Go Deeper (optional):**
- Why might assigning one label to an entire image be problematic, even when all annotated animals belong to the same class?

In [ ]:
# @title 🔎 Explore the Number of Animals per Image { display-mode: "form" }
_max_n = int(images["n_animals"].max())
_several = images[images["n_animals"] > 1]
print(f"{len(_several)} of {len(images)} photos ({100 * len(_several) / len(images):.0f} %) hold more than "
      f"one animal; the most crowded photo has {_max_n}.")
print("mean animals per photo: " + " · ".join(
    f"{c} {images.loc[images['class'] == c, 'n_animals'].mean():.1f}" for c in CLASSES))

fig, ax = plt.subplots(figsize=(9, 3.8))
_bins = np.arange(0.5, min(_max_n, 10) + 1.5, 1)
_per_class = [images.loc[images["class"] == c, "n_animals"].clip(upper=10) for c in CLASSES]
ax.hist(_per_class, bins=_bins, stacked=True, color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.set_xticks(range(1, min(_max_n, 10) + 1))
ax.set_xticklabels([str(i) if i < 10 else "10+" for i in range(1, min(_max_n, 10) + 1)])
ax.set_xlabel("animals in the photo")
ax.set_ylabel("photos")
ax.set_title("Animals per photo")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

N_CROWDED = 3 if SMOKE else 6  # how many crowded photos to show (N-014)
_crowded = images.sort_values(["n_animals", "id"], ascending=[False, True]).head(N_CROWDED)
show_rows(_crowded, "the most crowded photos", lambda r: f"{r['n_animals']} × {r['class']}", cols=N_CROWDED)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Multiple animals:</strong> On the full Play Dataset, 244 of 638 images (38%) contain more than one annotated animal. The most crowded image contains 26.</li>
<li><strong>Class differences:</strong> Sheep have the highest average (5.7 animals per image), followed by birds (3.4) and zebras (3.2). Cats and dogs usually appear individually.</li>
<li><strong>Crowded images:</strong> Images with many animals may contain overlapping or very small animals, making individual animals harder to distinguish.</li>
<li><strong>Classification limitation:</strong> A single label cannot describe how many animals are present or where they are located. An image with thirty sheep and one with a single lamb both receive the label "sheep". For counting or locating animals, object detection or segmentation would be more suitable.</li>
</ul>
<p style="margin:12px 0 4px;"><strong>🔎 Go Deeper (optional)</strong></p>
<ul style="margin:4px 0 0 0;padding-left:22px;">
<li><strong>Classification limitation:</strong> One image label does not describe how many animals are present or where they are located. This matters if the park needs to count animals or monitor their movements; object detection or segmentation would be more suitable.</li>
</ul>
</details>
</div>
""")

In [ ]:
# @title 📘 Labeling Strategies: One Label or Multiple Animals? { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;">
  <div style="font-weight:bold;margin-bottom:6px;">
    📘 Labeling Strategies: One Label or Multiple Animals?
  </div>
  <ul style="margin:6px 0;padding-left:22px;">
    <li><strong>Single-label classification:</strong> Assigns one class to an entire image, even if several animals are present.</li>
    <li><strong>Multi-label classification:</strong> Identifies all animal classes present, such as both sheep and dog.</li>
    <li><strong>Object detection:</strong> Identifies and locates individual animals using bounding boxes, allowing them to be counted.</li>
  </ul>
  <div>
    Our COCO subset excludes images containing multiple selected animal classes.
    This removes examples such as dogs among sheep—exactly the situations the park's dog alert needs to recognize.
  </div>
</div>
""")

#### 🔴 Class Imbalance: Can High Accuracy Be Misleading?

Imagine a camera monitoring a sheep paddock. Most images contain only sheep, while dogs appear rarely. A rule that always predicts the most common class might achieve high accuracy without detecting any dogs.

**Accuracy** is the proportion of correct predictions. **Dog recall** is the proportion of actual dog images correctly identified.

**Explore:**
1. Estimate accuracy and dog recall for two rules that ignore the image:
   - **Rule A — COCO subset:** Always predict the most common animal class.
   - **Rule B — Paddock camera:** Of 1,000 images, 20 contain a dog. Always predict "no dog".
2. Run the code and compare your estimates with the results.
3. Change `DOG_SHARE` using the slider and observe how the results change.

**Observe:**
- Which rule achieves higher accuracy?
- What is the dog recall for each rule?
- How does the proportion of dog images affect accuracy and dog recall?
- Why is accuracy alone insufficient for evaluating the park's dog alert?

In [ ]:
# @title 🔎 Explore Accuracy and Dog Recall Under Class Imbalance { display-mode: "form" }

DOG_SHARE = 0.1 # @param {type:"slider", min:0.01, max:0.2, step:0.01}

# Rule A: Always predict the most common class in the COCO subset
_counts = images["class"].value_counts()
_majority_class = _counts.idxmax()
_accuracy_a = _counts.max() / len(images)
_dog_recall_a = float(_majority_class == "dog")

# Rule B: Always predict "no dog" in the paddock
_accuracy_b = 1 - DOG_SHARE
_dog_recall_b = 0.0

print(f"Rule A — COCO subset (always '{_majority_class}')")
print(f"  Accuracy:   {_accuracy_a:.1%}")
print(f"  Dog recall: {_dog_recall_a:.1%}")

print(f"\nRule B — Paddock camera ({DOG_SHARE:.0%} dog images, always 'no dog')")
print(f"  Accuracy:   {_accuracy_b:.1%}")
print(f"  Dog recall: {_dog_recall_b:.1%}")

In [ ]:
# @title 🔎 See How the Rule Classifies Real Images { display-mode: "form" }

# The same day in miniature: 12 real photos, a share DOG_SHARE of them dogs, the rest sheep.
_N_STRIP = 12
_strip_dogs = min(int(round(DOG_SHARE * _N_STRIP)) or int(DOG_SHARE > 0), images_per_class["dog"])
_rng = random.Random(SEED)
_dog_idx = _rng.sample(sorted(images.index[images["class"] == "dog"]), _strip_dogs)
_sheep_idx = _rng.sample(sorted(images.index[images["class"] == "sheep"]),
                         min(_N_STRIP - _strip_dogs, images_per_class["sheep"]))
_strip_idx = _dog_idx + _sheep_idx
_rng.shuffle(_strip_idx)
_strip = images.loc[_strip_idx]

_cols = 6
_rows = -(-len(_strip) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(2.3 * _cols, 2.5 * _rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, (_, row) in zip(axes.ravel(), _strip.iterrows(), strict=False):
    ax.axis("on")
    ax.imshow(load_thumb(row["path"]))
    frame(ax, CLASS_COLORS[row["class"]])
    _right = row["class"] == "sheep"
    ax.text(0.04, 0.96, "sheep " + ("✔" if _right else "✘"), transform=ax.transAxes, ha="left", va="top",
            fontsize=10, fontweight="bold", color="white",
            bbox={"boxstyle": "round,pad=0.25", "facecolor": CLASS_COLORS["sheep"], "edgecolor": "none"})
    ax.set_title(f"true: {row['class']}", fontsize=8, color="#2E7D32" if _right else "#C62828")
_strip_right = int((_strip["class"] == "sheep").sum())
fig.suptitle(f"A lazy model stamps 'sheep' on {len(_strip)} paddock frames: {_strip_right} right, "
             f"{len(_strip) - _strip_right} wrong", color=INK)
fig.tight_layout()


plt.show()

print(
    f"This strip: accuracy {100 * _strip_right / len(_strip):.0f}%, "
    "recall for dog: 0%. "
    "The dog-in-the-paddock alert never fires."
)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Rule A:</strong> In the full COCO subset, the largest class contains 130 of 638 images, giving approximately 20% accuracy. Dog recall is 0% if the rule always predicts cat, or 100% if it always predicts dog.</li>
<li><strong>Rule B:</strong> Always predicting "no dog" gives 98% accuracy but 0% dog recall when 2% of images contain dogs. None of these dogs triggers an alert.</li>
<li><strong>Changing DOG_SHARE:</strong> As the proportion of dog images increases, accuracy decreases, while dog recall remains 0%.</li>
<li><strong>Implications:</strong> High accuracy can be misleading for imbalanced datasets. For the park's dog alert, detecting dogs is essential, so dog recall matters; false alarms must also be considered.</li>
</ul>
<p style="margin:8px 0 0;"><em>Results for Rule A may differ when using the smaller smoke-test subset.</em></p>
</details>
</div>
""")

## Common Dataset Errors & Cleaning

Exploring a dataset helps us understand whether it is suitable for a task. But even a suitable dataset may contain errors that affect model training and evaluation.

In this section, we investigate duplicate images, including flipped or rotated copies, and suspicious class labels. These problems can cause data leakage, misleading evaluation results, or incorrect learning.

Our goal is to identify potential issues and prepare a cleaner dataset for splitting into training and test sets.

In [ ]:
# @title 🧾 Recipe · Step 1, continued: Dataset Errors { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div style="background:#FFF8D6;color:#333333;border-left:6px solid #F5C400;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;">
  <div style="font-weight:bold;margin-bottom:6px;">
    🧾 Recipe · Step 1, continued: Dataset Errors
  </div>
  <div style="font-style:italic;margin-bottom:6px;">
    “One time I discovered that the data contained duplicate examples.
    Another time I found corrupted images / labels.”
  </div>
  <div>
    Andrej Karpathy,
    <a href="https://karpathy.github.io/2019/04/25/recipe/"
       target="_blank" style="color:#333333;">
      A Recipe for Training Neural Networks
    </a> (2019).
  </div>
</div>
""")

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · This is ordinary: two Kaggle stories</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Datasets built with care that still shipped duplicates</summary>
<p style="margin:6px 0 0 0;"><strong>PetFinder.my Pawpularity</strong> (2021) shipped near-identical photos with <em>different</em> scores, and <strong>Cassava
Leaf Disease</strong> (2020) identical leaf photos with two different labels. Both were built with care. Yours
will have duplicates too, unless you look.</p>
</details>
</div>""")

#### 🟢 Duplicate Images: What Can We Detect?

Duplicate images can lead to misleading evaluation results, especially when copies appear in both training and test sets.

- **Exact duplicates:** Identical image files with the same content.
- **Near-duplicates:** Images showing essentially the same content but differing in compression, resolution, cropping, or orientation.

We use MD5 to detect exact duplicates and perceptual hashing (pHash) to identify potential near-duplicates.

**Before running the code:**
- Do you expect a curated dataset such as COCO to contain exact duplicates?
- Can pHash also flag images that are visually similar but not duplicates?

**Explore:**
1. Run the duplicate detection and inspect the results.
2. Adjust the bit distance threshold to change which image pairs are flagged.
3. Browse the flagged pairs, ordered from most to least similar. Increase the threshold if no pairs are found.

**Observe:**
- How many exact duplicates are found?
- How does increasing the threshold affect the number of flagged pairs?
- Are all flagged pairs genuine duplicates?
- Why could duplicates across training and test sets lead to misleading evaluation results?

**Go Deeper (optional):**
- Why should flagged pairs be visually inspected before removing images?

In [ ]:
# @title 📘 Detecting Duplicates: MD5 vs. Perceptual Hashing { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;">
  <div style="font-weight:bold;margin-bottom:6px;">
    📘 Detecting Duplicates: MD5 vs. Perceptual Hashing
  </div>
  <ul style="margin:6px 0;padding-left:22px;">
    <li><strong>MD5:</strong> Computes a hash from file bytes. Identical files have identical hashes, but resizing or recompressing changes the hash.</li>
    <li><strong>Perceptual hashing (pHash):</strong> Represents coarse visual structure using a compact 64-bit hash. Visually similar images often produce similar hashes, even after resizing or recompression.</li>
    <li><strong>Hamming distance (bit distance):</strong> Counts how many bits differ between two hashes (0–64). Smaller distances indicate more similar hashes, but not necessarily more similar image content.</li>
    <li><strong>Bit distance threshold:</strong> Defines the maximum number of differing bits for flagging a pair as a potential near-duplicate.</li>
  </ul>
  <div style="margin-top:8px;">
    <strong>Important limitation:</strong> Different images may have similar hashes,
    while genuine near-duplicates may have larger bit distances.
    Flipped or rotated images may also require additional checks.
    The threshold identifies <strong>candidates for visual inspection</strong>,
    not confirmed duplicates.
  </div>
</div>
""")

In [ ]:
# @title 🔎 Helpers: Duplicate Detection and Image Quality { display-mode: "form" }
import hashlib
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
from PIL import Image

EXTREME_SIDE_PX = (64, 6000)


def file_md5(path) -> str:
    """Compute an MD5 hash from file bytes."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def exact_duplicate_groups(paths: list) -> list[list]:
    """Return groups of byte-identical files."""
    groups = {}
    for path in paths:
        groups.setdefault(file_md5(path), []).append(path)
    return [group for group in groups.values() if len(group) > 1]


def phash(image: Image.Image) -> imagehash.ImageHash:
    """Compute a 64-bit perceptual hash."""
    return imagehash.phash(image)


def hash_bits(hashes: list[imagehash.ImageHash]) -> np.ndarray:
    """Convert perceptual hashes to an (n, 64) bit array."""
    return np.array([h.hash.flatten() for h in hashes], dtype=bool)


def hash_distances(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute pairwise bit distances between hash arrays."""
    return (a[:, None, :] != b[None, :, :]).sum(axis=-1)


def is_readable(path) -> bool:
    """Check whether an image can be fully decoded."""
    try:
        with Image.open(path) as im:
            im.load()
        return True
    except (OSError, ValueError):
        return False


def flag_odd_files(stats: pd.DataFrame) -> pd.DataFrame:
    """Identify unreadable files, unusual color modes, and extreme dimensions."""
    rows = []

    for _, r in stats.iterrows():
        flags = []

        if not is_readable(r["path"]):
            flags.append("unreadable")
        if r["mode"] != "RGB":
            flags.append(f"mode {r['mode']}")
        if not EXTREME_SIDE_PX[0] <= r["long_side"] <= EXTREME_SIDE_PX[1]:
            flags.append(f"extreme size {r['width']}×{r['height']}")

        if flags:
            rows.append({
                "id": r["id"],
                "class": r["class"],
                "flags": ", ".join(flags)
            })

    return pd.DataFrame(rows, columns=["id", "class", "flags"])

In [ ]:
# @title 🔎 Inspect Near-Duplicates Using pHash { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output

import imagehash
from PIL import Image

hashes = []
for path in images["path"]:
    with Image.open(path) as img:
        hashes.append(imagehash.phash(img.convert("RGB")))

PAIRS = [
    (i, j, hashes[i] - hashes[j])
    for i in range(len(hashes))
    for j in range(i + 1, len(hashes))
    if hashes[i] - hashes[j] <= 20
]

PAIRS.sort(key=lambda pair: pair[2])

print(f"Found {len(PAIRS)} candidate pairs within 20 bits.")

threshold = widgets.IntSlider(
    value=8, min=0, max=20, step=1,
    description="Bit distance threshold:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

previous = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
summary = widgets.HTML()
counter = widgets.HTML()
output = widgets.Output()

state = {"index": 0}

def update_browser(*_):
    # Only show pairs within the selected threshold
    flagged_pairs = [pair for pair in PAIRS if pair[2] <= threshold.value]

    state["index"] = min(state["index"], max(0, len(flagged_pairs) - 1))

    summary.value = (
        f"<b>{len(images)}</b> images checked &nbsp; | &nbsp; "
        f"<b>{len(flagged_pairs)}</b> potential near-duplicate pairs"
    )


    previous.disabled = state["index"] == 0
    next_button.disabled = state["index"] >= len(flagged_pairs) - 1

    with output:
        clear_output(wait=True)

        if not flagged_pairs:
            counter.value = ""
            print("No image pairs found within this threshold. Try increasing it.")
            return

        i, j, distance = flagged_pairs[state["index"]]

        counter.value = (
            f"<b>Pair {state['index'] + 1} of {len(flagged_pairs)}</b>"
        )

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        for ax, idx in zip(axes, (i, j)):
            row = images.iloc[idx]
            ax.imshow(load_thumb(row["path"]))
            ax.set_title(f"Class: {row['class']}", fontsize=11)
            ax.axis("off")

        fig.suptitle(f"Bit distance: {distance} (out of 64 bits)", fontsize=12)

        fig.tight_layout()
        plt.show()
        plt.close(fig)

def navigate(step):
    state["index"] += step
    update_browser()

def change_threshold(change):
    state["index"] = 0
    update_browser()

previous.on_click(lambda _: navigate(-1))
next_button.on_click(lambda _: navigate(1))
threshold.observe(change_threshold, names="value")

display(summary)
display(threshold)
display(widgets.HBox([previous, counter, next_button]))
display(output)

update_browser()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Exact duplicates:</strong> No byte-identical files were found among the 638 images. MD5 detects identical files, but recompressing an image changes its hash. Even curated datasets should be checked, especially when new images are added or exported.</li>

<li><strong>Near-duplicates:</strong> At a threshold of 8 bits, one pair is flagged: two images of the same cat in front of a brick wall, with slightly different head positions. Most other images have larger bit distances, but visually similar scenes can still produce false positives.</li>
<li><strong>Threshold:</strong> Increasing the threshold flags more pairs, including unrelated images. A lower threshold may miss genuine near-duplicates.</li>
<li><strong>Data leakage:</strong> Similar images across training and test sets may lead to overly optimistic performance estimates because the model has already seen much of the visual content.</li>
<li><strong>Go Deeper:</strong> A small bit distance does not prove duplication. Flagged pairs should be inspected before deciding whether to remove or group them during dataset splitting.</li>
</ul>
<p style="margin:8px 0 0;"><em>Counts may differ for the smaller smoke-test subset or other thresholds.</em></p>
</details>
</div>
""")

### 🔴 Bit Distance Distribution (Optional)

The histogram shows the bit distance between each image and its closest match in the dataset. Smaller distances indicate greater visual similarity.

**Observe:**
- Where are most bit distances concentrated?
- How does changing the threshold affect the number of images considered potential duplicates?
- Why might choosing a suitable threshold be difficult?

In [ ]:
# @title 🔎 Explore Bit Distance Distribution { display-mode: "form" }

HASH_THRESHOLD = 5 # @param {type:"slider", min:0, max:20, step:1}

exact_groups = exact_duplicate_groups(images["path"].tolist())
print(f"{len(images)} files · {len(exact_groups)} groups of byte-identical files")
for _group in exact_groups[:5]:
    print("  ", " = ".join(Path(p).name for p in _group))

HASHES = np.stack([
    np.asarray(phash(load_thumb(path)).hash, dtype=np.uint8).ravel()
    for path in images["path"]
])

_dist = hash_distances(HASHES, HASHES)
np.fill_diagonal(_dist, 99)
nearest_distance = _dist.min(axis=1)  # each photo's distance to its closest other photo
_i, _j = np.where(np.triu(_dist <= HASH_THRESHOLD, k=1))
near_pairs = pd.DataFrame({"a": images["id"].to_numpy()[_i], "b": images["id"].to_numpy()[_j],
                           "distance": _dist[_i, _j]}).sort_values("distance").reset_index(drop=True)
print(f"pairs within {HASH_THRESHOLD} bits: {len(near_pairs)} · median nearest-neighbour distance: "
      f"{np.median(nearest_distance):.0f} bits")

fig, ax = plt.subplots(figsize=(9, 3.4))
_bins = np.arange(-0.5, 41, 2)
ax.hist([nearest_distance[(images["class"] == c).to_numpy()] for c in CLASSES], bins=_bins, stacked=True,
        color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.axvline(HASH_THRESHOLD + 0.5, color=INK, ls=":", lw=1)
ax.text(HASH_THRESHOLD + 1, ax.get_ylim()[1] * 0.9, f"HASH_THRESHOLD = {HASH_THRESHOLD}", fontsize=9)
ax.set_xlabel("Hamming distance to the closest other photo (bits)")
ax.set_ylabel("photos")
ax.set_title("How close is each photo's nearest neighbour?")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Distribution:</strong> Most images have a nearest-neighbor bit distance well above a strict duplicate threshold. Only a few pairs have very small distances.</li>
<li><strong>Threshold:</strong> Increasing the threshold flags more potential duplicates, while decreasing it flags fewer. The underlying distance distribution remains unchanged.</li>
<li><strong>Trade-off:</strong> A low threshold may miss genuine near-duplicates; a high threshold may flag unrelated images. There is no universally correct threshold, so visual inspection is important.</li>
</ul>
</details>
</div>
""")

### 🔴 How Does Perceptual Hashing Work? (Optional)

The following visualization shows how pHash generates a 64-bit representation of an image and why visually similar images often produce similar hashes.

In [ ]:
# @title 🔎 Explore Perceptual Hashing (pHash) { display-mode: "form" }

from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from scipy.fft import dctn
from io import BytesIO

highlight = "#FFDA00"
bit_colors = ListedColormap(["#F0F0F0", "#888888"])

c0 = images["class"].iloc[0]
pool = images[images["class"] != c0]

orig = load_thumb(images["path"].iloc[0])
unrelated = load_thumb((pool if len(pool) else images)["path"].iloc[-1])

buf = BytesIO()
orig.resize(
    (int(orig.width * 0.7), int(orig.height * 0.7))
).save(buf, "JPEG", quality=55)

buf.seek(0)
near = Image.open(buf).convert("RGB")

grey = orig.convert("L").resize((32, 32), Image.Resampling.LANCZOS)
dct = dctn(np.asarray(grey, dtype=float), type=2)

bits = {
    k: phash(im).hash
    for k, im in [("orig", orig), ("near", near), ("other", unrelated)]
}


def draw_bits(ax, values, other=None, title=""):
    """Display an 8×8 binary hash and highlight differing bits."""
    ax.imshow(values, cmap=bit_colors, vmin=0, vmax=1)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

    for (r, c), bit in np.ndenumerate(values):
        ax.text(
            c, r, str(int(bit)),
            ha="center", va="center", fontsize=7,
            color="white" if bit else INK
        )

        if other is not None and bit != other[r, c]:
            ax.add_patch(Rectangle(
                (c - 0.5, r - 0.5), 1, 1,
                fill=False, edgecolor=highlight, linewidth=2.5
            ))

    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(2, 4, figsize=(10, 5.2))

# How the hash is constructed
axes[0, 0].imshow(orig)
axes[0, 0].set_title("1. Original photo", fontsize=9)

axes[0, 1].imshow(grey, cmap="gray")
axes[0, 1].set_title("2. Resize to 32×32, grayscale", fontsize=9)

axes[0, 2].imshow(np.log1p(np.abs(dct)), cmap="Greys")
axes[0, 2].add_patch(Rectangle(
    (-0.5, -0.5), 8, 8,
    fill=False, edgecolor=highlight, linewidth=2.5
))
axes[0, 2].set_title("3. DCT: select top-left 8×8\n(coarse image structure)", fontsize=9)

draw_bits(
    axes[0, 3], bits["orig"],
    title="4. Above median? 1 / 0\n= 64 bits"
)

for ax in axes[0, :3]:
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

# Compare perceptual hashes
d_near = int((bits["orig"] != bits["near"]).sum())
d_other = int((bits["orig"] != bits["other"]).sum())

draw_bits(axes[1, 0], bits["orig"], title="Original")
draw_bits(
    axes[1, 1], bits["near"], bits["orig"],
    f"Re-compressed copy:\n{d_near} bits differ"
)
draw_bits(
    axes[1, 2], bits["other"], bits["orig"],
    f"Unrelated photo:\n{d_other} bits differ"
)

axes[1, 3].axis("off")
note = (
    "Yellow = differing bits\n\n"
    "Hamming distance =\n"
    "number of differing bits\n\n"
    f"Near copy: {d_near}\n"
    f"Unrelated: {d_other}"
)
axes[1, 3].text(0, 0.5, note, fontsize=9, va="center")

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML(r"""
<div class="callout callout-answer"
     style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<details style="margin:0;">
<summary style="cursor:pointer;font-weight:bold;">💡 Sample Answer</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Hash construction:</strong> The image is resized, converted to grayscale, and transformed using the DCT. Selected low-frequency coefficients are converted into a 64-bit hash.</li>
<li><strong>Near-duplicate:</strong> The recompressed image produces the same hash (0 differing bits), despite changes to its file representation.</li>
<li><strong>Unrelated image:</strong> The hashes differ in 38 of 64 bits, reflecting different visual structures.</li>
<li><strong>Interpretation:</strong> pHash captures coarse visual similarity rather than semantic content. Small bit distances suggest potential duplicates but do not guarantee them.</li>
</ul>
</details>
</div>
""")

### 🔴 Detecting Flipped and Rotated Duplicates (Optional)

Standard perceptual hashing can identify visually similar images, but may miss flipped or rotated copies.

In this experiment, we create a small dataset containing exact copies, recompressed images, and flipped or rotated versions of existing photos.

**Before running the code:**
- Predict which duplicates each method will detect: exact copies, recompressed images, flipped images, and rotated images. Compare MD5, standard pHash, and transformation-aware pHash.

**Explore:**
1. Generate the example dataset.
2. Compare the duplicates detected by the different methods.
3. Inspect which transformations each method detects or misses.

**Observe:**
- Which types of duplicates can MD5 detect?
- Does standard pHash detect all transformed copies?
- What additional duplicates are identified when flipped and rotated versions are compared?

**🔎 Go Deeper (optional):**
- What are the computational costs and potential drawbacks of checking multiple transformations?

In [ ]:
# @title 🔎 Generate a Dataset with Known Duplicates { display-mode: "form" }

from PIL import ImageOps
import shutil

HUNT_DIR = DATA_DIR / "cache" / "hunt5"
N_CARD_ORIGINALS = 12 if SMOKE else 40  # distinct photos on the card (N-014: fewer in smoke mode)
PLANT_KINDS = ["exact", "exact", "flipped", "rotated", "recompressed"]  # ≥ 4 planted copies, always present


def plant_hunt5(dest: Path, n_originals: int, seed: int) -> tuple[dict[str, Path], list[dict]]:
    """Write a 'memory card': n_originals distinct photos plus one planted copy per entry of PLANT_KINDS.

    Returns (files, key): files maps the card's neutral file names to paths; key lists the planted copies as
    {"copy": name, "source": name, "kind": kind}. Deterministic for a given seed.
    """
    rng = random.Random(seed)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    chosen = images.sample(n=min(n_originals, len(images)), random_state=seed).sort_values("id")
    sources = [Path(p) for p in chosen["path"]]
    # (bytes, kind or None for an original, index of the source photo)
    entries: list[tuple[bytes, str | None, int]] = [(s.read_bytes(), None, i) for i, s in enumerate(sources)]
    planted: list[tuple[str, int]] = []
    source_idx = rng.sample(range(len(sources)), len(PLANT_KINDS))
    for kind, i in zip(PLANT_KINDS, source_idx, strict=True):
        with Image.open(sources[i]) as im:
            rgb = im.convert("RGB")
        buf = BytesIO()
        if kind == "exact":
            data = sources[i].read_bytes()
        elif kind == "flipped":
            ImageOps.mirror(rgb).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        elif kind == "rotated":
            rgb.transpose(Image.Transpose.ROTATE_90).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        else:  # recompressed: smaller and at a lower JPEG quality
            small = rgb.resize((max(8, int(rgb.width * 0.7)), max(8, int(rgb.height * 0.7))))
            small.save(buf, "JPEG", quality=55)
            data = buf.getvalue()
        entries.append((data, kind, i))
        planted.append((kind, i))
    order = list(range(len(entries)))
    rng.shuffle(order)
    names = {idx: f"card_{k:03d}.jpg" for k, idx in enumerate(order)}
    files, key = {}, []
    for idx, (data, kind, i) in enumerate(entries):
        path = dest / names[idx]
        path.write_bytes(data)
        files[names[idx]] = path
        if kind is not None:
            key.append({"copy": names[idx], "source": names[i], "kind": kind})
    return dict(sorted(files.items())), key


hunt_files, HUNT5_KEY = plant_hunt5(HUNT_DIR, N_CARD_ORIGINALS, SEED)
hunt_images = {name: load_thumb(path) for name, path in hunt_files.items()}

print(f"Generated {len(hunt_files)} images, including {len(HUNT5_KEY)} intentionally added duplicates.")

In [ ]:
# @title Helpers: Compare Duplicate Detection Methods { display-mode: "form" }

KIND_OF_VARIANT = {"original": "recompressed", "flip_lr": "flipped", "flip_tb": "flipped",
                   "rot90": "rotated", "rot180": "rotated", "rot270": "rotated"}


def find_copies(files: dict[str, Path], threshold: int, use_variants: bool) -> pd.DataFrame:
    """Pairs of files on a card that are copies of each other: columns a, b, kind, distance.

    kind is "exact" (same bytes), "recompressed" (hashes within `threshold`), or, with use_variants=True,
    "flipped" / "rotated" (a flipped or rotated variant of a hashes within `threshold` of b).
    """
    names = list(files)
    md5s = {n: file_md5(p) for n, p in files.items()}
    thumbs = {n: load_thumb(p) for n, p in files.items()}
    plain = hash_bits([phash(thumbs[n]) for n in names])
    variant_names = VARIANT_NAMES if use_variants else ["original"]
    # distances[v][i, j]: hash of variant v of file i vs the plain hash of file j
    distances = {v: hash_distances(hash_bits([phash(variants(thumbs[n])[v]) for n in names]), plain)
                 for v in variant_names}
    rows = []
    for i, a in enumerate(names):
        for j in range(i + 1, len(names)):
            b = names[j]
            if md5s[a] == md5s[b]:
                rows.append({"a": a, "b": b, "kind": "exact", "distance": 0})
                continue
            best_v = min(variant_names, key=lambda v: distances[v][i, j])
            best_d = int(distances[best_v][i, j])
            if best_d <= threshold:
                rows.append({"a": a, "b": b, "kind": KIND_OF_VARIANT[best_v], "distance": best_d})
    return pd.DataFrame(rows, columns=["a", "b", "kind", "distance"])


def score_hunt(found: pd.DataFrame, key: list[dict]) -> int:
    """How many planted copies in `key` appear as a pair (either order) in `found`."""
    pairs = {frozenset((r["a"], r["b"])) for _, r in found.iterrows()}
    return sum(frozenset((k["copy"], k["source"])) in pairs for k in key)

In [ ]:
# @title 🔎 Compare Duplicate Detection Methods { display-mode: "form" }
from PIL import ImageOps

def variants(img):
    """Generate image transformations for duplicate detection."""
    return {
        "original": img,
        "flip": ImageOps.mirror(img),
        "rotate90": img.rotate(90, expand=True),
        "rotate180": img.rotate(180, expand=True),
        "rotate270": img.rotate(270, expand=True),
    }

VARIANT_NAMES = list(variants(Image.new("RGB", (4, 4))))

KIND_OF_VARIANT = {
    "original": "original",
    "flip": "flipped",
    "rotate90": "rotated",
    "rotate180": "rotated",
    "rotate270": "rotated",
}
plain = find_copies(hunt_files, threshold=8, use_variants=False)
variant = find_copies(hunt_files, threshold=8, use_variants=True)

print(f"Ground truth of intentionally added duplicates:\t{len(HUNT5_KEY)}")
print(f"MD5 + standard pHash detected:\t\t\t{score_hunt(plain, HUNT5_KEY)}")
print(f"MD5 + transformation-aware pHash detected:\t{score_hunt(variant, HUNT5_KEY)}")

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML("""
<details style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
                border-radius:6px;padding:12px 14px;line-height:1.5;">
  <summary><strong>💡 Sample Answer</strong></summary>
  <ul>
    <li><strong>MD5:</strong> Detects exact copies because their file bytes are identical.</li>
    <li><strong>Standard pHash:</strong> Can also detect recompressed images, but may miss flipped or rotated copies.</li>
    <li><strong>Transformation-aware pHash:</strong> Detects all five intentionally added duplicates by additionally comparing flipped and rotated versions.</li>
    <li><strong>Trade-off:</strong> Checking more transformations increases computation and may also produce more false positives.</li>
  </ul>
  <p><strong>Interpretation:</strong> In this experiment, standard hashing detects 3 of 5 known duplicates, compared with 5 of 5 for transformation-aware hashing. This does not mean the latter is always more accurate: unrelated images may also be flagged.</p>
</details>
""")

Why, on one photo and five kinds of copy: the pHash distance to the original, plain vs variant.

In [ ]:
# @title 🔎 How Do Image Transformations Affect pHash? { display-mode: "form" }
_src = load_thumb(images["path"].iloc[0])

def _reencode(im, quality):
    buf = BytesIO()
    im.convert("RGB").save(buf, "JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

_copies = {
    "original": _src,
    "exact copy": _src,
    "resized": _src.resize((int(_src.width * 0.6), int(_src.height * 0.6))),
    "flipped": ImageOps.mirror(_src),
    "recompressed": _reencode(_src, 25),
    "rotated": _src.transpose(Image.Transpose.ROTATE_90),
}

_h0 = phash(_src)
FOUND_COLOR, MISSED_COLOR = "#1E7B6E", "#B03A2E"

fig, axes = plt.subplots(
    1, len(_copies),
    figsize=(2.6 * len(_copies), 3.8)
)

for ax, (name, im) in zip(axes, _copies.items(), strict=True):
    ax.imshow(im)
    frame(ax, INK)
    ax.set_title(name, fontsize=10)

    if name == "original":
        ax.text(
            0.5, -0.12, "Reference image",
            transform=ax.transAxes, ha="center", fontsize=9
        )
        continue

    d_plain = phash(im) - _h0
    d_variant = min(phash(v) - _h0 for v in variants(im).values())

    for k, d in enumerate((d_plain, d_variant)):
        detected = d <= HASH_THRESHOLD
        verdict = "✓" if detected else "✗"
        color = FOUND_COLOR if detected else MISSED_COLOR

        ax.text(
            0.5, -0.10 - 0.13 * k,
            f"{d} bits  {verdict}",
            transform=ax.transAxes,
            ha="center", va="top",
            fontsize=10, fontweight="bold", color=color
        )

fig.suptitle(
    f"Which copies are detected? (Threshold: {HASH_THRESHOLD} bits)",
    fontsize=13, color=INK
)

fig.text(0.5, 0.12, "First row: Standard pHash    |    Second row: Transformation-aware pHash",
         ha="center", fontsize=10, color=INK)

fig.subplots_adjust(bottom=0.30, top=0.82, wspace=0.12)

display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML("""
<details style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
                border-radius:6px;padding:12px 14px;line-height:1.5;">
  <summary><strong>💡 Sample Answer</strong></summary>
  <ul>
    <li><strong>Standard pHash:</strong> Detects exact, resized, and recompressed copies, but may miss flipped or rotated images because their bit distances are larger.</li>
    <li><strong>Transformation-aware pHash:</strong> Also detects flipped and rotated copies by comparing multiple transformations. Reversing a flip or rotation can reduce the bit distance to zero.</li>
    <li><strong>Trade-off:</strong> Computing additional hashes increases computational cost and may produce more false positives.</li>
  </ul>
  <p><strong>Important:</strong> A bit distance of zero means the perceptual hashes match, not necessarily that the image files are identical.</p>
</details>
""")

### 🔴 Identifying Unusual Images (Optional)

Images with unusual characteristics may reveal interesting variations or potential data quality issues.

We use color histograms to identify images whose color distributions differ most from others in the same class.

**Explore:**
1. Run the code and inspect the histogram distances across classes.
2. Browse the two most unusual images per class.

**Observe:**
- What makes these images unusual (e.g., background, lighting, framing)?
- Are the images and their labels valid?
- Why does a high histogram distance not necessarily indicate a data quality issue?

**Go Deeper (optional):**
- What are the limitations of color histograms for identifying unusual images?

In [ ]:
# @title 🔎 Find Unusual Images Using Color Histograms { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output

N_ATYPICAL = 1 if SMOKE else 2  # Images shown per class
_features = CHANNEL_HISTS.reshape(len(image_stats), -1)
_class_mean = {c: _features[(image_stats["class"] == c).to_numpy()].mean(axis=0) for c in CLASSES}
image_stats["atypicality"] = [float(np.abs(f - _class_mean[c]).sum())
                              for f, c in zip(_features, image_stats["class"], strict=True)]

fig, ax = plt.subplots(figsize=(9, 3.4))
for k, c in enumerate(CLASSES):
    _vals = image_stats.loc[image_stats["class"] == c, "atypicality"].to_numpy()
    _jitter = np.random.RandomState(SEED + k).uniform(-0.25, 0.25, len(_vals))
    ax.scatter(k + _jitter, _vals, s=14, color=CLASS_COLORS[c], alpha=0.8)

ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES)
ax.set_ylabel("Color histogram distance to class average")
ax.set_title("Which images have unusual color distributions?")
fig.tight_layout()
display(fig)
plt.close(fig)

print("Higher distances indicate unusual colors, not necessarily incorrect labels.")

least_typical = (image_stats.sort_values("atypicality", ascending=False)
                 .groupby("class", sort=False).head(N_ATYPICAL))

# Browse the most unusual images, two per class
_class_index = 0
_prev = widgets.Button(description="← Previous")
_next = widgets.Button(description="Next →")
_out = widgets.Output()

def _show_class():
    c = CLASSES[_class_index]
    subset = least_typical[least_typical["class"] == c]
    with _out:
        clear_output(wait=True)
        show_rows(subset, f"{c.capitalize()} — images flagged for manual review",
                  lambda r: f"Distance: {r['atypicality']:.2f}", cols=2)
    _prev.disabled = _class_index == 0
    _next.disabled = _class_index == len(CLASSES) - 1

def _navigate(step):
    global _class_index
    _class_index = max(0, min(len(CLASSES) - 1, _class_index + step))
    _show_class()

_prev.on_click(lambda _: _navigate(-1))
_next.on_click(lambda _: _navigate(1))

display(widgets.HBox([_prev, _next]), _out)
_show_class()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML

HTML("""
<details style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;
                border-radius:6px;padding:12px 14px;line-height:1.5;">
  <summary><strong>💡 Sample Answer</strong></summary>
  <ul>
    <li><strong>Unusual images:</strong> High histogram distances may result from different backgrounds, lighting, colors, or framing.</li>
    <li><strong>Data quality:</strong> Unusual images are not necessarily problematic. They may represent valid variations within a class. Images and labels should be checked before making changes.</li>
    <li><strong>Limitations:</strong> Color histograms ignore spatial structure and object identity. They cannot reliably distinguish unusual backgrounds from incorrect labels or other data quality issues.</li>
  </ul>
</details>
""")

### 🔴 Exploring Image Embeddings (Optional)

Images can be represented by numerical features. While **color histograms** describe color distributions, **pretrained visual embeddings** capture more complex visual patterns. We compare both representations to explore image similarity, visualize class distributions using UMAP, and identify unusual images.

**Before running the code:**
- Do you expect color histograms and pretrained visual embeddings to identify the same similar images and outliers?

**Explore:**
1. Compare the UMAP visualizations of color histograms and pretrained visual embeddings.
2. Explore the most similar image pairs and unusual images within each class.
3. Compare the median within-class similarities across animal classes.

**Observe and interpret:**
- How do the class distributions differ between the two representations?
- Do the two representations identify the same similar image pairs and unusual images?
- Which classes have the highest and lowest within-class similarity, and what might explain these differences?
- How useful are these approaches for identifying potential data quality issues?

**Go Deeper (optional):**
- Why might two images have similar visual content but very different color distributions?

In [ ]:
# @title 📘 Image Embeddings, Similarity, and UMAP { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1;
            padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 Image Embeddings, Similarity, and UMAP</strong>

<p>Different image representations capture different information:</p>

<ul>
  <li><strong>Color histograms</strong> describe color distributions but ignore spatial arrangement and object identity.</li>
  <li><strong>Pretrained visual embeddings</strong> capture learned visual features, such as shapes, textures, and broader visual patterns.</li>
  <li><strong>Cosine similarity</strong> measures how closely two feature vectors align. Higher values indicate greater similarity in the chosen representation.</li>
  <li><strong>UMAP</strong> projects high-dimensional features into two dimensions for visualization. It approximates local relationships, so distances in the plot should not be interpreted as exact similarities.</li>
</ul>

<p><strong>Important:</strong> Similar images are not necessarily duplicates, and unusual images are not necessarily mislabeled. Results depend on the representation and should always be checked against the original images.</p>

</div>
"""))

In [ ]:
# @title Helpers: Image Embeddings { display-mode: "form" }

EMBEDDINGS_NAME = "coco_animals_embeddings.npz"  # npz keys: ids (n,), embeddings (n, d); matched by image id


def load_embeddings() -> tuple[np.ndarray, str]:
    """(vectors aligned with `images`, source). Source is "precomputed" or "colour histograms (stand-in)".

    Reads data/coco_animals_embeddings.npz, fetching it from DATA_BASE_URL outside smoke mode. Never runs a
    model. Falls back to the colour histograms when the file is missing or does not cover every image.
    """
    path = DATA_DIR / EMBEDDINGS_NAME
    if not path.exists() and not SMOKE:
        download_file(DATA_BASE_URL + EMBEDDINGS_NAME, path, headers=_github_headers())
    if path.exists():
        try:
            z = np.load(path)
            by_id = dict(zip(z["ids"].tolist(), z["embeddings"], strict=True))
            if all(i in by_id for i in images["id"]):
                return np.stack([by_id[i] for i in images["id"]]).astype(np.float32), "precomputed"
        except Exception:
            pass
    return CHANNEL_HISTS.reshape(len(images), -1).astype(np.float32), "colour histograms (stand-in)"


def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    """Cosine similarity between every pair of rows (after centring the columns)."""
    centred = vectors - vectors.mean(axis=0)
    norm = np.linalg.norm(centred, axis=1, keepdims=True) + 1e-9
    unit = centred / norm
    return unit @ unit.T

In [ ]:
# @title Helpers: Pretrained Visual Embeddings { display-mode: "form" }
%pip -q install transformers umap-learn bokeh

import torch
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from tqdm.auto import tqdm

MODEL_NAME = "facebook/dinov2-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

def dinov2_embeddings(paths, batch_size=16):
    vectors = []

    for start in tqdm(range(0, len(paths), batch_size), desc="Extracting visual features"):
        batch = []

        for path in paths[start:start + batch_size]:
            with Image.open(path) as im:
                batch.append(im.convert("RGB"))

        inputs = processor(images=batch, return_tensors="pt").to(DEVICE)

        with torch.inference_mode():
            features = model(**inputs).last_hidden_state[:, 0]
            features = torch.nn.functional.normalize(features, dim=1)

        vectors.append(features.cpu().numpy())

    return np.concatenate(vectors).astype("float32")

In [ ]:
# @title Precompute Image Representations and UMAP { display-mode: "form" }
import hashlib
import warnings
import numpy as np
import umap

CACHE_DIR = DATA_DIR / "cache" / "embeddings"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

paths = image_stats["path"].tolist()

def cached_features(name, compute):
    key = hashlib.sha256(
        (name + MODEL_NAME + repr(paths)).encode()
    ).hexdigest()[:16]
    cache_path = CACHE_DIR / f"{name}_{key}.npy"

    if cache_path.exists():
        print(f"Loaded {name} from cache.")
        return np.load(cache_path)

    features = compute()
    np.save(cache_path, features)
    print(f"Computed and cached {name}.")
    return features

representations = {
    "Color histograms": CHANNEL_HISTS.reshape(
        len(image_stats), -1
    ).astype("float32"),

    "Pretrained visual embeddings": cached_features(
        "global_dinov2",
        lambda: dinov2_embeddings(paths)
    ),
}

# Compute or load one UMAP projection per representation
projections = {}

for name, features in representations.items():
    settings = dict(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=SEED
    )

    key = hashlib.sha256(
        np.ascontiguousarray(features).tobytes()
        + repr(settings).encode()
    ).hexdigest()[:16]

    cache_path = CACHE_DIR / f"umap_{key}.npy"

    if cache_path.exists():
        projections[name] = np.load(cache_path)
        print(f"Loaded UMAP: {name}")
    else:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore", message="n_jobs value 1 overridden"
            )
            projections[name] = umap.UMAP(
                **settings
            ).fit_transform(features)

        np.save(cache_path, projections[name])
        print(f"Computed and cached UMAP: {name}")

print("Ready:", ", ".join(representations))

In [ ]:
# @title 🔎 Explore Image Representations with UMAP { display-mode: "form" }
import base64
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap

# Choose representation before running the plot
UMAP_FEATURES = "Pretrained visual embeddings" # @param ["Color histograms", "Pretrained visual embeddings"]

def thumbnail_uri(path):
    im = load_thumb(path)
    im.thumbnail((120, 120))
    buf = BytesIO()
    im.convert("RGB").save(buf, "JPEG", quality=75)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

projection = projections[UMAP_FEATURES]

source = ColumnDataSource(dict(
    x=projection[:, 0],
    y=projection[:, 1],
    cls=image_stats["class"].tolist(),
    image=[thumbnail_uri(p) for p in image_stats["path"]],
))

output_notebook()

p = figure(
    title=f"UMAP of Image Representations — {UMAP_FEATURES}",
    width=850,
    height=500,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom"
)

p.scatter(
    "x", "y",
    source=source,
    size=8,
    alpha=0.8,
    color=factor_cmap(
        "cls",
        palette=[CLASS_COLORS[c] for c in CLASSES],
        factors=list(CLASSES)
    ),
    legend_field="cls"
)

p.add_tools(HoverTool(tooltips="""
    <div style="width:150px;">
        <img src="@image" width="130"
             style="max-height:130px;object-fit:contain;">
        <div><b>@cls</b></div>
    </div>
"""))

p.xaxis.axis_label = "UMAP 1"
p.yaxis.axis_label = "UMAP 2"
p.legend.title = "Class"
p.legend.location = "top_right"
p.legend.click_policy = "hide"

show(p)

In [ ]:
# @title 🔎 Explore Similar and Unusual Images { display-mode: "form" }
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

SIMILARITY_FEATURES = "Pretrained visual embeddings" # @param ["Color histograms", "Pretrained visual embeddings"]

features = representations[SIMILARITY_FEATURES]
classes = image_stats["class"].to_numpy()
indices = np.arange(len(image_stats))

sim = cosine_similarity_matrix(features)
np.fill_diagonal(sim, -np.inf)

# Rank all image pairs by cosine similarity
i, j = np.triu_indices(len(sim), k=1)
order = np.argsort(sim[i, j])[::-1]
top_pairs = [(int(i[k]), int(j[k])) for k in order]

# Rank images by average similarity to others in their class
class_similarity = np.array([
    sim[k, (classes == c) & (indices != k)].mean()
    for k, c in enumerate(classes)
])

outlier_indices = {
    c: np.where(classes == c)[0][
        np.argsort(class_similarity[classes == c])
    ].tolist()
    for c in CLASSES
}

# Exploration controls
mode = widgets.ToggleButtons(
    options=["Most similar pairs", "Unusual images"],
    description="Explore:"
)
class_picker = widgets.Dropdown(
    options=list(CLASSES),
    description="Class:"
)
n_pairs = widgets.BoundedIntText(
    value=10, min=1, max=min(100, len(top_pairs)),
    description="Pairs:",
    layout=widgets.Layout(width="160px")
)
n_unusual = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description="Images:",
    continuous_update=False,
    style={"description_width": "initial"}
)
prev_btn = widgets.Button(description="← Previous")
next_btn = widgets.Button(description="Next →")
counter = widgets.HTML()
preview = widgets.Output()
position = 0

def update_view(change=None):
    global position

    if mode.value == "Most similar pairs":
        total = min(n_pairs.value, len(top_pairs))
        position = max(0, min(position, total - 1))

        a, b = top_pairs[position]
        rows = image_stats.iloc[[a, b]]

        title = (
            f"Pair {position + 1}/{total} · "
            f"cosine similarity: {sim[a, b]:.2f}"
        )
        labels = lambda r: r["class"]
        cols = 2

        prev_btn.disabled = position == 0
        next_btn.disabled = position == total - 1
        counter.value = f"{position + 1}/{total}"

    else:
        c = class_picker.value
        selected = outlier_indices[c][:n_unusual.value]
        rows = image_stats.iloc[selected].copy()
        rows["class_similarity"] = class_similarity[selected]

        title = f"{c.capitalize()} · lowest average similarity within class"
        labels = lambda r: f"Avg. similarity: {r['class_similarity']:.2f}"
        cols = min(4, len(selected))

    with preview:
        clear_output(wait=True)
        show_rows(rows, title, labels, cols=cols)

    is_pairs = mode.value == "Most similar pairs"
    class_picker.layout.display = "none" if is_pairs else ""
    n_pairs.layout.display = "" if is_pairs else "none"
    n_unusual.layout.display = "none" if is_pairs else ""
    prev_btn.layout.display = "" if is_pairs else "none"
    next_btn.layout.display = "" if is_pairs else "none"
    counter.layout.display = "" if is_pairs else "none"

def navigate(step):
    global position
    position = max(0, min(n_pairs.value - 1, position + step))
    update_view()

def reset_view(change=None):
    global position
    position = 0
    update_view()

prev_btn.on_click(lambda _: navigate(-1))
next_btn.on_click(lambda _: navigate(1))

mode.observe(reset_view, names="value")
class_picker.observe(reset_view, names="value")
n_pairs.observe(reset_view, names="value")
n_unusual.observe(reset_view, names="value")

display(widgets.HBox([mode, class_picker, n_pairs, n_unusual]))
display(widgets.HBox([prev_btn, next_btn, counter]))
display(preview)

update_view()

In [ ]:
# @title 🔎 Compare Within-Class Image Similarity { display-mode: "form" }
for c in CLASSES:
    idx = np.where(classes == c)[0]
    values = sim[np.ix_(idx, idx)]
    values = values[np.triu_indices(len(idx), k=1)]
    print(f"{c:8s} median similarity: {np.median(values):.3f}")

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary style="cursor:pointer;"><strong>💡 Sample Answer</strong></summary>

<ul>
  <li><strong>UMAP:</strong> Pretrained embeddings may group visually similar
  animals more clearly than color histograms. However, overlapping clusters
  and 2D distances should be interpreted cautiously.</li>

  <li><strong>Similar pairs:</strong> Color histograms emphasize similar color
  distributions, while pretrained embeddings capture similarities in
  appearance, shape, and scene content. The closest pairs may therefore differ.</li>

  <li><strong>Unusual images:</strong> The two representations may identify
  different outliers. Unusual colors do not necessarily imply unusual
  visual content, or vice versa.</li>

  <li><strong>Within-class similarity:</strong> Higher median similarity suggests
  more consistent image representations within a class. Differences may reflect
  animal appearance, backgrounds, viewpoints, or dataset composition,
  rather than classification accuracy.</li>

  <li><strong>Data quality:</strong> Both approaches can highlight images worth
  inspecting, but neither establishes labeling errors without further examination.</li>

  <li><strong>Go Deeper:</strong> Two images may depict the same type of animal
  but have different colors due to lighting, backgrounds, or appearance.
  Pretrained embeddings may still recognize similarities in their shapes
  and visual structures.</li>
</ul>

<p><strong>Key takeaway:</strong> Image similarity depends on the chosen
representation. Comparing representations reveals different aspects
of visual diversity and potential data quality issues.</p>

</details>
"""))

## From Data Exploration to Cleaning

Before training a model, potential data quality issues should be reviewed to decide which images to keep, correct, or remove. Unreadable images and confirmed duplicates are candidates for removal, while near-duplicates and unusual images require closer inspection.


### 🔴 Which potential data quality issues should be addressed before training a model?

**Reflect and justify:**
- Which images would you remove from this dataset, and why?
- Why might automatically removing unusual images be problematic?
- Why should duplicates be reviewed before splitting the dataset?

> ⚠️ **Pitfall: Data Leakage**
>
> If duplicate or near-duplicate images appear in both training and test sets, performance estimates may be overly optimistic. Review duplicates **before splitting** and ensure related images do not end up in different splits.

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary style="cursor:pointer;"><strong>💡 Sample Answer</strong></summary>

<ul>
  <li><strong>Cleaning decisions:</strong> Remove unreadable images and
  confirmed duplicates, keeping one image per duplicate group.
  Review potential near-duplicates and suspicious labels before deciding.
  Unusual images are not necessarily incorrect.</li>

  <li><strong>Removing unusual images:</strong> Automatically excluding
  unusual images can reduce dataset diversity and introduce selection bias.
  These images may represent valid but less common conditions that the
  model should learn to handle.</li>

  <li><strong>Data leakage:</strong> Duplicate or near-duplicate images
  appearing in both training and test sets can produce overly optimistic
  performance estimates. Duplicate groups should therefore be reviewed
  before splitting and kept within the same split if retained.</li>
</ul>

<p><strong>Key takeaway:</strong> Dataset cleaning requires justified
decisions, not automatic removal of every unusual or similar image.</p>

</details>
"""))

In [ ]:
# @title 📘 Image Dataset Quality Checklist { display-mode: "form" }
from IPython.display import HTML, display

display(HTML(r"""
<div class="callout callout-theory"
     style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<div style="font-weight:bold;">📘 Image Dataset Quality Checklist</div>

<p style="margin:6px 0;">Before using an image dataset for computer vision, check:</p>

<div style="font-weight:bold;margin:10px 0 4px 0;">Qualitative Inspection</div>
<ul style="list-style:none;margin:0;padding-left:4px;">
<li style="margin:5px 0;">☐ <strong>Use-case fit:</strong> Do the images represent the conditions in which the model will be used?</li>
<li style="margin:5px 0;">☐ <strong>Visual variation:</strong> Do the images cover relevant differences in backgrounds, viewpoints, lighting, and object sizes?</li>
<li style="margin:5px 0;">☐ <strong>Recognizability:</strong> Are the relevant objects visible and distinguishable?</li>
<li style="margin:5px 0;">☐ <strong>Annotation availability:</strong> Are the labels, bounding boxes, or masks needed for the task available?</li>
<li style="margin:5px 0;">☐ <strong>Annotation quality:</strong> Are annotations accurate, complete, and consistent across images and objects?</li>
<li style="margin:5px 0;">☐ <strong>Label relevance:</strong> Are class definitions clear, and do assigned labels reflect the relevant image content?</li>
</ul>

<div style="font-weight:bold;margin:12px 0 4px 0;">Quantitative Inspection</div>
<ul style="list-style:none;margin:0;padding-left:4px;">
<li style="margin:5px 0;">☐ <strong>Dataset size:</strong> Are there enough images and annotations for the intended task?</li>
<li style="margin:5px 0;">☐ <strong>Class distribution:</strong> Are all classes sufficiently represented, or are some underrepresented?</li>
<li style="margin:5px 0;">☐ <strong>Image properties:</strong> Are image dimensions, resolutions, and aspect ratios suitable for the task?</li>
<li style="margin:5px 0;">☐ <strong>Image integrity:</strong> Can all image files be opened and processed correctly?</li>
<li style="margin:5px 0;">☐ <strong>Outliers:</strong> Are unusual images identified using different visual features and inspected before deciding whether to keep or remove them?</li>
<li style="margin:5px 0;">☐ <strong>Duplicates and data leakage:</strong> Are exact and potential near-duplicates reviewed, and are related images kept within the same data split?</li>
</ul>

</div>
"""))

## Dataset Splitting & Data Leakage

How can we split an image dataset so that model performance on unseen images provides a reliable estimate of generalization?

Before training a model, we need to decide which images belong in the training, validation, and test sets. Different splitting strategies affect class distributions and the reliability of model evaluation.

In [ ]:
# @title 📘 Data Leakage and Generalization { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1;
            padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 Data Leakage and Generalization</strong>

<p>A test set should provide an independent estimate of how well a model
performs on unseen data.</p>

<p><strong>Data leakage</strong> occurs when information that should be
unavailable during training or model selection influences the model or
its evaluation. For example:</p>

<ul>
  <li><strong>Duplicate images:</strong> Exact or near-duplicates appear
  in both training and test sets.</li>
  <li><strong>Related images:</strong> Images from the same animal, camera,
  or recording session appear in different splits, potentially making
  the test set less independent.</li>
  <li><strong>Preprocessing:</strong> Data-dependent transformations,
  such as normalization using dataset statistics, are fitted using
  test data.</li>
</ul>

<p><strong>Key principle:</strong> Choose a splitting strategy that reflects
the intended application. If the model must generalize to new animals,
cameras, or locations, these groups should be kept separate across splits.</p>

</div>
"""))

### 🟢 Random vs. Stratified Splitting: How Do Class Distributions Differ?

A *random split* assigns images to training and test sets without explicitly preserving class proportions. A *stratified split* aims to maintain similar class distributions across both sets.

**Before running the code:**
- Which classes do you expect to show the largest differences between their overall proportion and their proportion in a random test set?

**Explore:**
1. Compare the class distributions of the random and stratified 80/20 splits.

**Observe and interpret:**
- Which class shows the largest relative deviation in the random test set?
- Does stratification preserve class proportions exactly? Why or why not?
- Why might stratification be particularly useful for smaller classes?

In [ ]:
# @title 🔎 Compare Random and Stratified Splits { display-mode: "form" }

from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    LeaveOneGroupOut,
    LeaveOneOut,
    StratifiedKFold,
    train_test_split,
)

TEST_SIZE = 0.2
_ids = images["id"].to_numpy()
_y = images["class"].to_numpy()
_tr_rand, _te_rand = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED)
_tr_strat, _te_strat = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED, stratify=_y)


def class_shares(ids) -> pd.Series:
    """Class proportions for the given image IDs, in CLASSES order."""
    counts = images.set_index("id").loc[list(ids), "class"].value_counts()
    return counts.reindex(CLASSES).fillna(0) / max(len(ids), 1)


shares = pd.DataFrame(
    {
        "whole table": class_shares(_ids),
        "random · test": class_shares(_te_rand),
        "stratified · test": class_shares(_te_strat),
    }
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col in zip(axes, ["random · test", "stratified · test"], strict=True):
    x = np.arange(len(CLASSES))
    ax.bar(x - 0.2, shares["whole table"], width=0.4, color="#DDE5E7", label="whole table")
    ax.bar(x + 0.2, shares[col], width=0.4, color=[CLASS_COLORS[c] for c in CLASSES], label=col)
    ax.set_xticks(x, CLASSES)
    ax.set_title(f"{col} ({len(_te_rand)} photos)")
    ax.set_ylabel("share of photos")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()
_drift = (shares.sub(shares["whole table"], axis=0).abs() * 100).round(1)
print("largest class-share drift from the whole table, in percentage points:")
print(f"  random split: {_drift['random · test'].max():.1f} ({_drift['random · test'].idxmax()})   "
      f"stratified split: {_drift['stratified · test'].max():.1f} ({_drift['stratified · test'].idxmax()})")
_whole = shares["whole table"].replace(0, np.nan)
_rel = shares.sub(shares["whole table"], axis=0).abs().div(_whole, axis=0) * 100
print("largest drift relative to the class's own share, in %:")
print(f"  random split: {_rel['random · test'].max():.0f} ({_rel['random · test'].idxmax()})   "
      f"stratified split: {_rel['stratified · test'].max():.0f} ({_rel['stratified · test'].idxmax()})")
RESULTS["split_drift_random_pp"] = float(_drift["random · test"].max())
RESULTS["split_drift_stratified_pp"] = float(_drift["stratified · test"].max())

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary style="cursor:pointer;"><strong>💡 Sample Answer</strong></summary>
<ul>
<li><strong>Before running:</strong> Smaller classes may show larger relative deviations because a few images can substantially change their proportions.</li>
<li><strong>Random split:</strong> Class proportions can differ from the full dataset. The class with the largest relative deviation depends on the random split and should be identified from the results.</li>
<li><strong>Stratified split:</strong> Class proportions are preserved more closely, but not necessarily exactly because image counts must be whole numbers.</li>
<li><strong>Smaller classes:</strong> Stratification helps ensure that underrepresented classes are represented proportionally in both sets, although very small classes may still have too few test images for reliable evaluation.</li>
</ul>
<p><strong>Key takeaway:</strong> Stratification improves class balance across splits, but does not prevent data leakage from duplicate or related images.</p>
</details>
"""))

### 🟢 Cross-Validation: How Do We Choose Validation Sets?

A single train/test split provides only one estimate of model performance, which depends on the selected images. *Cross-validation* evaluates a model across multiple splits, providing an estimate of its average performance and variability.

We compare four strategies-k-fold, stratified k-fold, leave-one-out (LOO), and leave-one-group-out (LOGO)-using 12 example images from three cameras.

**Before running the code:**
- How might different splitting strategies affect class balance and the independence of validation images?

**Explore:**
1. Compare which images are used for training and validation across the four strategies.

**Observe and interpret:**
- How do k-fold and stratified k-fold differ in their class distributions?
- What distinguishes leave-one-out from k-fold?
- Which strategy evaluates generalization to an unseen camera, and why?

**Go Deeper (optional):**
- What are the computational disadvantages of leave-one-out cross-validation?

In [ ]:
# @title 📘 Cross-Validation Strategies { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1;
            padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 Cross-Validation Strategies</strong>

<p>Cross-validation evaluates a model across multiple training and validation
splits to estimate its performance and variability.</p>

<ul>
<li><strong>K-fold:</strong> Divide the dataset into k folds. Each fold is
used once for validation, while the remaining k−1 folds are used for training.</li>

<li><strong>Stratified k-fold:</strong> Similar to k-fold, but aims to preserve
class proportions in each fold, particularly useful for imbalanced datasets.</li>

<li><strong>Leave-one-out (LOO):</strong> Each validation fold contains a single
image. This maximizes the training data per fold but requires one training run
per image, making it computationally expensive.</li>

<li><strong>Leave-one-group-out (LOGO):</strong> Each fold holds out an entire
group, such as all images from one camera, to evaluate generalization to unseen
groups. It requires meaningful group identifiers.</li>
</ul>

<p><strong>Key principle:</strong> Choose a cross-validation strategy that
reflects the intended application. Stratification helps preserve class balance,
while group-based splitting prevents related images from appearing in both
training and validation sets.</p>

</div>
"""))

The diagram shows four strategies using 12 example images from three cameras (D = dog, the rare class). Magenta indicates validation images; gray-blue indicates training images.

In [ ]:
# @title 🔎 Visualize Cross-Validation Strategies { display-mode: "form" }

split_colors = {"train": "#B0B0B0", "test": "#FFDA00"}
dogs = {2, 3, 6}  # Rare class in a toy dataset of 12 photos

cv_folds = {
    "K-fold (a fold may miss the dog)": [{2, 3, 7, 10}, {0, 5, 8, 11}, {1, 4, 6, 9}],
    "Stratified K-fold (each fold has one dog)": [{0, 2, 4, 9}, {1, 3, 7, 10}, {5, 6, 8, 11}],
    "LOO (12 folds, first three shown)": [{0}, {1}, {2}],
    "LOGO (one camera per fold)": [{0, 1, 2, 3}, {4, 5, 6, 7}, {8, 9, 10, 11}],
}

fig, axes = plt.subplots(len(cv_folds), 1, figsize=(8.5, 7.2))

for ax, (name, folds) in zip(axes, cv_folds.items(), strict=True):
    for i, test in enumerate(folds):
        for p in range(12):
            x = p + 0.35 * (p // 4)  # Gap between cameras

            ax.add_patch(mpl.patches.FancyBboxPatch(
                (x + 0.06, i + 0.08), 0.88, 0.84,
                boxstyle="round,pad=0.02",
                facecolor=split_colors["test" if p in test else "train"],
                edgecolor="none"
            ))

            if p in dogs:
                ax.text(
                    x + 0.5, i + 0.5, "D",
                    ha="center", va="center",
                    fontsize=8, color=INK, fontweight="bold"
                )

    ticks = [0.5, 1.5, 2.5]
    labels = ["fold 1", "fold 2", "fold 3"]

    if name.startswith("LOO"):
        for p in range(12):
            ax.text(
                p + 0.35 * (p // 4) + 0.5, 3.3, "⋮",
                ha="center", va="center", fontsize=11, color=INK
            )
        ticks.append(3.3)
        labels.append("folds 4–12\nnot shown")

    ax.set_xlim(0, 12.7)
    ax.set_ylim(3.6, 0)
    ax.set_xticks([])
    ax.set_yticks(ticks, labels, fontsize=8)
    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(name, loc="left", fontsize=9, color=INK)

axes[-1].set_xticks(
    [2, 6.35, 10.7],
    ["camera A", "camera B", "camera C"],
    fontsize=8
)
axes[-1].tick_params(length=0)

fig.suptitle(
    "Cross-validation: grey = training, yellow = test, D = dog",
    color=INK
)

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary style="cursor:pointer;"><strong>💡 Sample Answer</strong></summary>
<ul>
<li><strong>Before running:</strong> Different strategies may produce different class distributions and levels of independence between training and validation images.</li>
<li><strong>K-fold vs. stratified k-fold:</strong> Standard k-fold may produce folds without examples of a rare class. Stratification aims to preserve class proportions across folds.</li>
<li><strong>Leave-one-out:</strong> Each validation fold contains a single image, unlike k-fold, which validates on a subset of images.</li>
<li><strong>Leave-one-group-out:</strong> Holding out an entire camera evaluates generalization to a camera not seen during training.</li>
<li><strong>Go Deeper:</strong> Leave-one-out requires one training run per image, making it computationally expensive.</li>
</ul>
<p><strong>Key takeaway:</strong> Cross-validation strategies differ in class balance, computational cost, and independence between training and validation data.</p>
</details>
"""))

### 🔴 Cross-Validation in Practice (Optional)

How many training runs do different cross-validation strategies require?

**Explore:**
1. Compare the number of folds and validation images per fold for k-fold, stratified k-fold, and leave-one-out.

**Observe and interpret:**
- Which strategy requires the most training runs?
- Why is leave-one-out evaluated on only a small subset?
- What are the limitations of evaluating a single image per fold?

In [ ]:
# @title 🔎 Compare Cross-Validation Fold Counts { display-mode: "form" }
K_FOLDS = 5
LOO_N = 12 if SMOKE else 40

def summarise_folds(name: str, splitter, X, y, groups=None) -> dict:
    """Summarize the number and size of validation folds."""
    sizes = [
        len(test_idx)
        for _, test_idx in splitter.split(X, y, groups)
    ]
    return {
        "Strategy": name,
        "Images": len(X),
        "Folds / training runs": len(sizes),
        "Validation images per fold": f"{min(sizes)}–{max(sizes)}",
    }

_loo_ids = _ids[:LOO_N]

cv_rows = [
    summarise_folds(
        "K-fold",
        KFold(K_FOLDS, shuffle=True, random_state=SEED),
        _ids, _y
    ),
    summarise_folds(
        "Stratified k-fold",
        StratifiedKFold(K_FOLDS, shuffle=True, random_state=SEED),
        _ids, _y
    ),
    summarise_folds(
        "Leave-one-out",
        LeaveOneOut(),
        _loo_ids, _y[:LOO_N]
    ),
]

display(pd.DataFrame(cv_rows))

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary style="cursor:pointer;"><strong>💡 Sample Answer</strong></summary>
<ul>
<li><strong>Training runs:</strong> K-fold and stratified k-fold each require 5 training runs. Leave-one-out requires one training run per image (40 in this example, or 12 in smoke mode).</li>
<li><strong>Small subset:</strong> Leave-one-out is demonstrated on fewer images to limit computational cost. On the full dataset, it would require one training run for every image.</li>
<li><strong>Single-image validation:</strong> Each LOO fold produces a prediction for only one image. Individual fold accuracy is therefore either 0 or 1, and per-class performance cannot be meaningfully estimated from a single fold.</li>
</ul>
<p><strong>Key takeaway:</strong> More folds increase computational cost. Leave-one-out uses nearly all available images for training in each fold, but requires substantially more training runs than k-fold.</p>
</details>
"""))

### Group-Based Splitting: The Trail-Camera Dataset

The farm park has installed motion-triggered cameras in its woodland reserve. Each camera can capture several images of the same animal within a short period, producing bursts of closely related images.

We use a subset of the [Caltech Camera Traps (CCT20) dataset](https://lila.science/datasets/caltech-camera-traps) containing images from eight cameras and five animal classes: opossum, raccoon, coyote, bobcat, and rabbit.

The following cells load the dataset and display example image bursts. The first run may take a few minutes to download and resize the images.

The farm park has installed motion-triggered cameras in its woodland reserve. Each camera can capture several images of the same animal within a short period. Such a sequence of closely related images is called a burst.

In [ ]:
# @title Group-Based Splitting and Data Leakage { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1; padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 Group-Based Splitting and Data Leakage</strong>

<p>
Images captured in the same burst are often very similar. If some appear in training and others in testing, the model may achieve overly optimistic test results. This is a form of <strong>data leakage</strong>.
</p>

<p>
Different splitting strategies address different goals:
</p>

<ul>
  <li><strong>Random splitting:</strong> Assigns images randomly to training and test sets.</li>
  <li><strong>Stratified splitting:</strong> Preserves approximately the same class proportions in both sets, but does not prevent related images from being split.</li>
  <li><strong>Group by burst:</strong> Keeps all images from the same burst together.</li>
  <li><strong>Group by camera:</strong> Keeps all images from the same camera together, allowing evaluation on unseen camera locations.</li>
</ul>

<p>
The <strong>sibling rate</strong> measures the percentage of test images that have at least one image from the same burst in the training set. A higher rate indicates greater potential for leakage.
</p>

</div>
"""))

In [ ]:
# @title Prepare Trail-Camera Helpers { display-mode: "form" }
import re

CCT_DIR = DATA_DIR / "cache" / "cct20"
CCT_SIDE = 512
CCT_CLASSES = ["opossum", "raccoon", "coyote", "bobcat", "rabbit"]
CCT_COLORS = dict(zip(CCT_CLASSES, [CLASS_COLORS[c] for c in CLASSES], strict=False))
SMOKE_CCT_LOCATIONS = 3
SMOKE_CCT_IMAGES = 30


def select_cct_rows(table: pd.DataFrame) -> pd.DataFrame:
    """The slice to use: everything, or in smoke mode whole bursts from 3 locations up to 30 photos."""
    if not SMOKE:
        return table
    keep = []
    locations = sorted(table["location"].unique())[:SMOKE_CCT_LOCATIONS]
    per_location = SMOKE_CCT_IMAGES // SMOKE_CCT_LOCATIONS
    for loc in locations:
        n = 0
        at_loc = table[table["location"] == loc].sort_values(["seq_id", "frame_num"])
        for _, burst in at_loc.groupby("seq_id", sort=True):
            if n + len(burst) > per_location:
                continue
            keep.append(burst)
            n += len(burst)
    return pd.concat(keep).reset_index(drop=True)


def cct_original_size(path: Path) -> tuple[int, int] | None:
    """The camera's (width, height), stored in the 512 px copy's JPEG comment; None for an older copy."""
    with Image.open(path) as im:
        found = re.match(rb"original (\d+)x(\d+)", im.info.get("comment", b""))
    return (int(found[1]), int(found[2])) if found else None


def fetch_cct_image(row: dict) -> tuple[str, Path | None]:
    """Download one trail-camera photo and store a 512 px copy; returns (file_name, path or None)."""
    dest = CCT_DIR / row["file_name"]
    if dest.exists() and dest.stat().st_size > 0 and cct_original_size(dest):
        return row["file_name"], dest
    raw = download_file(row["url"], CCT_DIR / "raw" / row["file_name"])
    if raw is None:
        return row["file_name"], None
    with Image.open(raw) as im:
        width, height = im.size  # the boxes in the manifest are in these pixels
        im.draft("RGB", (CCT_SIDE, CCT_SIDE))
        small = im.convert("RGB")
        small.thumbnail((CCT_SIDE, CCT_SIDE))
        small.save(dest, "JPEG", quality=90, comment=f"original {width}x{height}".encode())
    raw.unlink(missing_ok=True)
    return row["file_name"], dest


def download_cct(table: pd.DataFrame, workers: int = 16) -> pd.DataFrame:
    """Fetch every photo of `table` (parallel, cached), return the rows whose photo is on disk with a path."""
    CCT_DIR.mkdir(parents=True, exist_ok=True)
    paths, done = {}, 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for name, path in pool.map(fetch_cct_image, table.to_dict("records")):
            done += 1
            if path is not None:
                paths[name] = str(path)
            print(f"\r  trail-camera photos ready: {done}/{len(table)}", end="", flush=True)
    print()
    out = table[table["file_name"].isin(paths)].copy()
    out["path"] = out["file_name"].map(paths)
    return out.sort_values(["location", "seq_id", "frame_num"]).reset_index(drop=True)


def draw_cct_boxes(ax, row, thumb) -> None:
    """Draw the animal's box(es) from the manifest on a thumbnail, in the class colour (none when empty)."""
    original = cct_original_size(Path(row["path"]))
    if not isinstance(row["bbox"], str) or original is None:
        return
    scale = thumb.width / original[0]
    for x, y, w, h in json.loads(row["bbox"]):
        ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False, lw=2,
                                           edgecolor=CCT_COLORS[row["label"]]))


def show_bursts(table: pd.DataFrame, n_bursts: int, size: int = 256,
                show_boxes: bool = True) -> None:
    """One row per burst: its frames side by side, framed in the class colour, titled with the location.

    Picks bursts of different animals at different cameras, with a box on every frame, where the table
    allows it.
    """
    candidates = [b for _, b in table.groupby("seq_id", sort=True) if len(b) > 1]
    candidates.sort(key=lambda b: not b["bbox"].notna().all())  # fully boxed bursts first (stable sort)
    bursts, seen_labels, seen_locations = [], set(), set()
    for b in candidates:
        label, location = b["label"].iloc[0], b["location"].iloc[0]
        if label not in seen_labels and location not in seen_locations:
            bursts.append(b)
            seen_labels.add(label)
            seen_locations.add(location)
    bursts += [b for b in candidates if not any(b is k for k in bursts)][: n_bursts - len(bursts)]
    bursts = bursts[:n_bursts]
    cols = max(len(b) for b in bursts)
    fig, axes = plt.subplots(len(bursts), cols, figsize=(2.6 * cols, 2.1 * len(bursts)), squeeze=False)
    for r, burst in enumerate(bursts):
        for c in range(cols):
            ax = axes[r, c]
            if c < len(burst):
                row = burst.iloc[c]
                thumb = load_thumb(row["path"], (size, size))
                ax.imshow(thumb)
                if show_boxes:
                    draw_cct_boxes(ax, row, thumb)
                frame(ax, CCT_COLORS[row["label"]])
                ax.set_title(f"camera {row['location']} · {row['label']} · frame {row['frame_num']}",
                             fontsize=8)
            else:
                ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# @title 🔎 Explore Trail-Camera Image Bursts { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output

cct_all = pd.read_csv(fetch_manifest("cct20_slice.csv"))
cct = download_cct(select_cct_rows(cct_all))

if len(cct) < len(select_cct_rows(cct_all)):
    print(f"  ⚠️ {len(select_cct_rows(cct_all)) - len(cct)} photos could not be downloaded; run again.")

CCT_LOCATIONS = sorted(cct["location"].unique().tolist())
print(f"{len(cct)} photos · {cct['seq_id'].nunique()} bursts · cameras {CCT_LOCATIONS}")
print("photos per class: " + " · ".join(f"{c} {int((cct['label'] == c).sum())}" for c in CCT_CLASSES))

# Prepare bursts for browsing
bursts = [b for _, b in cct.groupby("seq_id", sort=True) if len(b) > 1]
bursts.sort(key=lambda b: not b["bbox"].notna().all())

bursts_per_page = 2
page = 0

show_bounding_boxes = widgets.Checkbox(value=True, description="Show bounding boxes")
previous_button = widgets.Button(description="Previous", icon="chevron-left")
next_button = widgets.Button(description="Next", icon="chevron-right")
counter = widgets.HTML()
output = widgets.Output()


def update_browser(change=None):
    start = page * bursts_per_page
    selected = bursts[start:start + bursts_per_page]

    previous_button.disabled = page == 0
    next_button.disabled = start + bursts_per_page >= len(bursts)
    counter.value = f"Bursts {start + 1}–{start + len(selected)} of {len(bursts)}"

    with output:
        clear_output(wait=True)
        if selected:
            show_bursts(
                pd.concat(selected),
                n_bursts=len(selected),
                show_boxes=show_bounding_boxes.value,
            )


def previous_page(button):
    global page
    page = max(0, page - 1)
    update_browser()


def next_page(button):
    global page
    page = min((len(bursts) - 1) // bursts_per_page, page + 1)
    update_browser()


previous_button.on_click(previous_page)
next_button.on_click(next_page)
show_bounding_boxes.observe(update_browser, names="value")

controls = widgets.HBox([previous_button, next_button, counter])
display(widgets.VBox([show_bounding_boxes, controls, output]))
update_browser()

### 🟢 Splitting Strategies: What Does Grouping Change?

Different splitting strategies determine which images can appear together in training and test sets. The appropriate strategy depends on what the model should generalize to.

**Before running the code:**
- Which splitting strategies do you expect to prevent images from the same burst appearing in both training and test sets?

**Explore:**
- Switch between the four splitting strategies. Compare the train/test assignments, camera locations, and reported metrics.
- Change the random seed. Observe which results change and which remain consistent.

**Observe and interpret:**
- Which strategies keep images from the same burst together?
- Which strategies ensure that training and test sets contain different cameras?
- Why does stratified splitting not necessarily prevent leakage?
- Why might group-based splitting produce a test share different from the requested 20%?

**Go Deeper (optional):**
- Which splitting strategy would you choose for a model deployed at existing cameras versus new camera locations? Justify your choice.

In [ ]:
# @title 🔎 Explore Leakage Across Splitting Strategies { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output

strategy = widgets.Dropdown(
    options=["Random", "Stratified", "Group by burst", "Group by camera"],
    description="Strategy:"
)
seed = widgets.IntSlider(value=SEED, min=0, max=100, description="Seed:")
show_boxes = widgets.Checkbox(value=True, description="Bounding boxes")
output = widgets.Output()


def update(change=None):
    with output:
        clear_output(wait=True)

        method = strategy.value
        indices = np.arange(len(cct))

        if method in ["Random", "Stratified"]:
            train, test = train_test_split(
                indices, test_size=0.2, random_state=seed.value,
                stratify=cct["label"] if method == "Stratified" else None
            )
        else:
            groups = cct["seq_id"] if method == "Group by burst" else cct["location"]
            train, test = next(
                GroupShuffleSplit(test_size=0.2, random_state=seed.value)
                .split(indices, groups=groups)
            )

        assignment = np.full(len(cct), "Train", dtype=object)
        assignment[test] = "Test"

        # Select illustrative bursts from both splits
        bursts = [b for _, b in cct.groupby("seq_id") if len(b) > 1]
        mixed = [b for b in bursts if len(set(assignment[b.index])) > 1]
        intact_train = [b for b in bursts if set(assignment[b.index]) == {"Train"}]
        intact_test = [b for b in bursts if set(assignment[b.index]) == {"Test"}]

        examples = (mixed[:2] + intact_train[:1] + intact_test[:1])[:2]
        if not mixed:
            examples = (intact_train[:1] + intact_test[:1])[:2]

        # Metrics for the entire split
        train_data, test_data = cct.iloc[train], cct.iloc[test]
        sibling_rate = 100 * test_data["seq_id"].isin(train_data["seq_id"]).mean()
        shared_cameras = len(
            set(train_data["location"]) & set(test_data["location"])
        )

        print(
            f"Test images: {len(test)} ({100 * len(test) / len(cct):.1f}%)  ·  "
            f"Sibling rate: {sibling_rate:.1f}%  ·  "
            f"Shared cameras: {shared_cameras}"
        )

        # Display real images with train/test borders
        if examples:
            cols = max(len(b) for b in examples)
            fig, axes = plt.subplots(
                len(examples), cols,
                figsize=(3 * cols, 2.7 * len(examples)),
                squeeze=False
            )

            for r, burst in enumerate(examples):
                for c in range(cols):
                    ax = axes[r, c]
                    if c >= len(burst):
                        ax.axis("off")
                        continue

                    row = burst.iloc[c]
                    part = assignment[burst.index[c]]
                    thumb = load_thumb(row["path"], (256, 256))

                    ax.imshow(thumb)
                    if show_boxes.value:
                        draw_cct_boxes(ax, row, thumb)

                    frame(ax, "#4C78A8" if part == "Train" else "#E76F51")
                    ax.set_title(
                        f"Camera {row['location']} · Frame {row['frame_num']} · {part}",
                        fontsize=9
                    )

            fig.suptitle(f"Train/test assignments — {method}")
            plt.tight_layout()
            plt.show()


strategy.observe(update, names="value")
seed.observe(update, names="value")
show_boxes.observe(update, names="value")

display(widgets.HBox([strategy, seed, show_boxes]), output)
update()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>💡 Sample Answer</strong></summary>

<ul>
  <li><strong>Random splitting:</strong> Images from the same burst can appear in both training and test sets, potentially leading to overly optimistic results.</li>
  <li><strong>Stratified splitting:</strong> Preserves class proportions but does not prevent burst-level leakage.</li>
  <li><strong>Group by burst:</strong> Keeps related images together, resulting in a sibling rate of 0%.</li>
  <li><strong>Group by camera:</strong> Separates camera locations and also prevents burst-level leakage. The number of shared cameras is 0.</li>
</ul>

<p>
Group-based splitting may produce a test share different from 20% because entire groups, rather than individual images, are assigned together.
</p>

<p>
<strong>Choosing a strategy:</strong> Grouping by burst is appropriate for evaluating new observations at existing cameras. Grouping by camera provides a more realistic evaluation when deploying a model at new camera locations.
</p>

</details>
"""))

In [ ]:
# @title 🔴 Splitting for Object Detection and Segmentation { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display

question = widgets.HTML(
    "<b>Which statements about splitting datasets for object detection "
    "and semantic segmentation are correct? Select all that apply.</b>"
)

options = [
    "A. Bounding boxes from the same image can be assigned to different splits.",
    "B. Stratification is more challenging in object detection because images may contain multiple classes.",
    "C. Individual pixels can be randomly assigned to training and test sets in semantic segmentation.",
    "D. Related images should be kept together in both tasks to avoid data leakage."
]

answers = [
    widgets.Checkbox(
        description=option,
        indent=False,
        layout=widgets.Layout(width="100%")
    )
    for option in options
]

check = widgets.Button(description="Check Answer")
solution = widgets.Button(description="Show Solution")
feedback = widgets.HTML()


def check_answer(button):
    selected = {i for i, answer in enumerate(answers) if answer.value}

    if selected == {1, 3}:
        feedback.value = (
            "<span style='color:green'>✅ Correct!</span> "
            "B and D are the correct answers."
        )
    else:
        feedback.value = (
            "<span style='color:#C2410C'>Not quite. Try again!</span><br>"
            "Think about which data units must stay together during splitting."
        )


def show_solution(button):
    feedback.value = (
        "<b>Correct answers: B and D</b><br><br>"
        "<b>A — False:</b> Bounding boxes from the same image must stay together.<br>"
        "<b>B — True:</b> Images may contain multiple classes, making stratification more complex.<br>"
        "<b>C — False:</b> Pixels from the same image must not be split independently.<br>"
        "<b>D — True:</b> Related images should stay together to prevent data leakage."
    )


check.on_click(check_answer)
solution.on_click(show_solution)

display(
    question,
    *answers,
    widgets.HBox([check, solution]),
    feedback
)

## Object Detection

In image classification, a model predicts what is present in an image. But what if the park wants to count individual animals and determine where they are?

Object detection extends classification by predicting a class label and a bounding box for each object. This allows us to investigate not only which animals appear, but also how many there are, where they are located, and how large they appear.

In this section, we explore how object size, position, crowding, and class imbalance affect an object detection dataset. We also examine the additional annotation effort required compared with image classification.


### From Labels to Bounding Boxes

Object detection datasets contain one annotation per object rather than one label per image. Each annotation specifies the object's class and bounding box.

In our dataset, bounding boxes use the COCO format `[x, y, w, h]`, where `x` and `y` indicate the top-left corner and `w` and `h` specify the width and height in pixels.

Unlike image classification, an image can contain multiple annotated objects, potentially belonging to different classes.

We explore four characteristics of the object detection dataset:

- **Class imbalance:** How are object instances distributed across classes?
- **Object size:** How large or small are the annotated objects?
- **Object position:** Where are objects located within images?
- **Crowding:** How many objects appear together, and how much do they overlap?

### 🟢 Exploring Bounding Boxes: What Do You Notice?

**Before running the code:**
- How might the number and size of annotated objects differ between images?

**Explore:**
- Examine the four example images and their bounding boxes.

**Observe and interpret:**
- How many objects are annotated in each image?
- How do the object sizes vary, and why might this matter for object detection?

In [ ]:
# @title Box helpers (draw boxes on thumbnails) { display-mode: "form" }
def box_area_bucket(w: float, h: float) -> str:
    """COCO's size buckets from a box's width and height in pixels: small < 32², medium < 96², else large."""
    area = w * h
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"


def show_boxes(rows_df: pd.DataFrame, boxes_df: pd.DataFrame, title: str, caption=None, cols: int = 4,
               size: int = 384) -> None:
    """Gallery of the photos in `rows_df` with their boxes from `boxes_df` drawn in the class colour.

    `caption(row) -> str` writes the title above each photo; by default the class and the number of boxes.
    """
    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(3.4 * cols, 3.2 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        thumb = load_thumb(row["path"], (size, size))
        scale = thumb.width / row["width"]
        ax.axis("on")
        ax.imshow(thumb)
        own = boxes_df[boxes_df["image_id"] == row["id"]]
        for _, b in own.iterrows():
            x, y, w, h = b["bbox"]
            ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False,
                                               lw=2, edgecolor=CLASS_COLORS[b["class"]]))
        frame(ax, CLASS_COLORS[row["class"]])
        text = caption(row) if caption else f"{row['class']} · {len(own)} boxes"
        ax.set_title(text, fontsize=9)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
# @title 🔎 Explore Bounding Boxes { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output

ann = annotations[annotations["image_id"].isin(images["id"])]
imgs = images[images["id"].isin(ann["image_id"])].reset_index(drop=True)

page = 0
page_size = 4
output = widgets.Output()

previous = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
counter = widgets.HTML()


def show_page():
    with output:
        clear_output(wait=True)

        start = page * page_size
        selected = imgs.iloc[start:start + page_size]

        show_boxes(
            selected,
            ann,
            "Bounding boxes: one annotation per animal",
            caption=lambda r: r["class"]
        )

        total_pages = (len(imgs) + page_size - 1) // page_size
        counter.value = f"Page {page + 1} of {total_pages}"
        previous.disabled = page == 0
        next_button.disabled = page >= total_pages - 1


def change_page(step):
    global page
    page += step
    show_page()


previous.on_click(lambda b: change_page(-1))
next_button.on_click(lambda b: change_page(1))

display(widgets.HBox([previous, next_button, counter]), output)
show_page()

### 🟢 Boxes per Class vs. Photos per Class

In image classification, class distributions are typically counted by images. In object detection, we can also count individual objects. For example, one image containing 12 sheep contributes one image but 12 bounding boxes.

**Before running the code:**
- Do you expect the class ranking to change when counting bounding boxes instead of images? Which class might increase the most?

**Coding Exercise:**

In this and subsequent coding exercises, you will implement small functions to explore the dataset. Complete the provided code cells and check your results. If you get stuck, you can reveal the reference solution. The notebook continues to work even if your implementation is incomplete.

- Implement `count_boxes_per_class(annotations)` to return the number of bounding boxes per class.
- Run the validation and compare the resulting distribution with the image-level distribution from Section 4.

**Observe and interpret:**
- Which classes change most in their relative frequency?
- Why might image-level and object-level class imbalance lead to different conclusions?

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">count_boxes_per_class</code></div>
<p style="margin:6px 0 0 0;">Write the function in the next cell: given an annotations table (one row per animal, with a <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">class</code>
column), return a dict <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">{class: number of boxes}</code> with <strong>every</strong> class in <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">CLASSES</code> as a key, counts as
plain <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">int</code>s. <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">use()</code> checks it against the reference and continues with the reference either way.</p>
</div>""")

In [ ]:
# @title Reference solution — peek only if you are stuck { display-mode: "form" }
def _ref_count_boxes_per_class(annotations):
    counts = annotations["class"].value_counts()
    return {c: int(counts.get(c, 0)) for c in CLASSES}

In [ ]:
# ✍️ YOUR CODE
def count_boxes_per_class(annotations):
    """Number of boxes (rows) per class.

    Inputs: `annotations`, a DataFrame with one row per animal and a `class` column.
    Output: a dict {class: int} with every class of CLASSES as a key (0 when a class has no box).

    e.g. {"cat": 12, "dog": 9, "bird": 30, "horse": 7, "sheep": 25, "zebra": 14}
    """
    ...


boxes = annotations[annotations["image_id"].isin(images["id"])].reset_index(drop=True)
boxes_per_class = use(count_boxes_per_class, boxes, reference=_ref_count_boxes_per_class)

In [ ]:
# @title 🔎 Compare Class Distributions: Photos vs. Boxes { display-mode: "form" }

photos_per_class = {c: int((images["class"] == c).sum()) for c in CLASSES}
results = {}
results["boxes_per_class"] = boxes_per_class
results["n_boxes"] = int(len(boxes))

by_photos = sorted(CLASSES, key=lambda c: -photos_per_class[c])
by_boxes = sorted(CLASSES, key=lambda c: -boxes_per_class[c])

print(f"{len(images)} photos · {len(boxes)} boxes")
print("Ranked by photos: " + " > ".join(f"{c} {photos_per_class[c]}" for c in by_photos))
print("Ranked by boxes:  " + " > ".join(f"{c} {boxes_per_class[c]}" for c in by_boxes))

for c in CLASSES:
    move = by_photos.index(c) - by_boxes.index(c)
    if move:
        print(
            f"  {c}: rank {by_photos.index(c) + 1} by photos → "
            f"rank {by_boxes.index(c) + 1} by boxes "
            f"({'↑' if move > 0 else '↓'}{abs(move)})"
        )

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

for ax, counts, order, what in zip(
    axes,
    [photos_per_class, boxes_per_class],
    [by_photos, by_boxes],
    ["Photos", "Boxes"],
    strict=True
):
    ax.bar(order, [counts[c] for c in order], color=[CLASS_COLORS[c] for c in order])
    ax.set_title(f"{what} per class (ranked)")
    ax.set_ylabel(what)

    for label in ax.get_xticklabels():
        label.set_color(CLASS_COLORS[label.get_text()])

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer
from IPython.display import HTML; HTML(r"""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>

<p>
The class ranking changes because some images contain many animals. Bird and sheep move up when counting bounding boxes, while cat and dog move down because they typically appear individually.
</p>

<p>
Image-level counts describe how frequently classes appear in photos, whereas box-level counts describe how many object instances are annotated. Therefore, class imbalance can look very different depending on which unit is counted.
</p>

</details>
""")

### 🟢 Object Sizes: How Small Are the Animals?

Bounding-box area is calculated as width × height in pixels. COCO distinguishes three object-size categories:

- **Small:** area < 32² pixels
- **Medium:** 32² ≤ area < 96² pixels
- **Large:** area ≥ 96² pixels

To compare differently shaped boxes, we use the square root of their area, `√(w × h)`, representing the side length of a square with the same area.

**Before running the code:**
- Which animal class do you expect to have the smallest bounding boxes?

**Explore:**
- Compare the box-size distributions across classes.
- Use Previous / Next to explore bounding boxes of different sizes and their enlarged crops. The selected box has a solid outline; other annotated objects have dotted outlines.

**Observe and interpret:**
- Which classes contain the smallest objects?
- How much do object sizes vary within the same class?
- Why might small objects be particularly challenging for object detection?
- What do the enlarged crops reveal about the visual information available for detecting small objects?

In [ ]:
# @title 🔎 Explore Object Size Distributions { display-mode: "form" }

boxes["w"] = boxes["bbox"].map(lambda b: b[2])
boxes["h"] = boxes["bbox"].map(lambda b: b[3])
boxes["box_area"] = boxes["w"] * boxes["h"]
boxes["side"] = np.sqrt(boxes["box_area"])
boxes["bucket"] = [
    box_area_bucket(w, h)
    for w, h in zip(boxes["w"], boxes["h"], strict=True)
]

bucket_share = (
    boxes.groupby("class")["bucket"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(index=CLASSES, columns=["small", "medium", "large"], fill_value=0)
)

results["small_box_share"] = {
    c: float(bucket_share.loc[c, "small"]) for c in CLASSES
}

print("Share of each class's boxes that are small / medium / large (%):")
display((bucket_share * 100).round(0).astype(int).T)

fig, ax = plt.subplots(figsize=(8, 3.8))

sides = [
    boxes.loc[boxes["class"] == c, "side"].clip(lower=4).to_numpy()
    for c in CLASSES
]

# Use log10 to display a wide range of object sizes
parts = ax.violinplot(
    [np.log10(v) for v in sides],
    showmedians=True, showextrema=False, widths=0.8
)

for body, c in zip(parts["bodies"], CLASSES, strict=True):
    body.set_facecolor(CLASS_COLORS[c])
    body.set_alpha(0.75)

parts["cmedians"].set_color(INK)
parts["cmedians"].set_linewidth(2.5)

tick_names = [
    f"{c}\nmedian {np.median(v):.0f} px"
    for c, v in zip(CLASSES, sides, strict=True)
]

ax.set_xticks(range(1, len(CLASSES) + 1), tick_names, fontsize=8)

for label, c in zip(ax.get_xticklabels(), CLASSES, strict=True):
    label.set_color(CLASS_COLORS[c])

ax.set_yticks(
    np.log10([4, 10, 32, 100, 300, 1000]),
    ["4", "10", "32", "100", "300", "1000"]
)

for threshold, name in [(32, "small ↓"), (96, "large ↑")]:
    ax.axhline(np.log10(threshold), color=INK, linestyle="--", linewidth=1)
    ax.text(
        0.5, np.log10(threshold),
        f" {threshold} px · {name}",
        va="bottom", fontsize=8, color=INK
    )

ax.set_ylabel("Box side √(w·h), px (log)")
ax.set_title("Box size per class (bar = median)")

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 🔎 Explore Bounding Box Sizes in Images { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output

by_area = boxes.sort_values(["box_area", "id"]).reset_index(drop=True)
photo_of = images.set_index("id")

# Start with the smallest, median, and largest boxes
first_page = [
    by_area.iloc[0],
    by_area.iloc[len(by_area) // 2],
    by_area.iloc[-1]
]

# Additional examples from each COCO size category
examples = []
for bucket in ["small", "medium", "large"]:
    subset = by_area[by_area["bucket"] == bucket]
    if not subset.empty:
        positions = np.linspace(0, len(subset) - 1, min(12, len(subset)), dtype=int)
        examples.extend(subset.iloc[positions].to_dict("records"))

# Remove duplicates while preserving the initial examples
seen = set()
selected_boxes = []
for b in [*first_page, *examples]:
    if b["id"] not in seen:
        selected_boxes.append(b)
        seen.add(b["id"])

pages = [
    selected_boxes[i:i + 3]
    for i in range(0, len(selected_boxes), 3)
]

page = 0
previous = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
counter = widgets.HTML()
output = widgets.Output()


def show_page():
    with output:
        clear_output(wait=True)

        selected = pages[page]
        fig, axes = plt.subplots(2, len(selected), figsize=(3.7 * len(selected), 7), squeeze=False)

        for col, b in enumerate(selected):
            photo = photo_of.loc[b["image_id"]]
            thumb = load_thumb(photo["path"], (384, 384))

            scale_x = thumb.width / photo["width"]
            scale_y = thumb.height / photo["height"]

            ax = axes[0, col]
            ax.imshow(thumb)

            for _, obj in boxes[boxes["image_id"] == b["image_id"]].iterrows():
                x, y, w, h = obj["bbox"]
                ax.add_patch(mpl.patches.Rectangle(
                    (x * scale_x, y * scale_y),
                    w * scale_x, h * scale_y,
                    fill=False,
                    lw=3 if obj["id"] == b["id"] else 1,
                    edgecolor=CLASS_COLORS[obj["class"]],
                    ls="-" if obj["id"] == b["id"] else ":"
                ))

            frame(ax, CLASS_COLORS[b["class"]])
            ax.set_title(
                f"{b['class']}: {b['w']:.0f}×{b['h']:.0f} px\n"
                f"{b['box_area']:,.0f} px² ({b['bucket']})",
                fontsize=9
            )

            # Enlarged crop around the selected box
            x, y, w, h = b["bbox"]
            pad = 0.25 * max(w, h) + 4
            x0, y0 = max(0, x - pad), max(0, y - pad)

            with Image.open(photo["path"]) as img:
                crop = img.convert("RGB").crop((
                    int(x0), int(y0),
                    int(min(img.width, x + w + pad)),
                    int(min(img.height, y + h + pad))
                ))

            ax = axes[1, col]
            ax.imshow(crop, interpolation="nearest")
            ax.add_patch(mpl.patches.Rectangle(
                (x - int(x0), y - int(y0)),
                w, h, fill=False, lw=2,
                edgecolor=CLASS_COLORS[b["class"]]
            ))
            frame(ax, CLASS_COLORS[b["class"]])
            ax.set_title("Bounding-box crop (enlarged)", fontsize=9)

        fig.tight_layout()
        plt.show()

        counter.value = f"Page {page + 1} of {len(pages)}"
        previous.disabled = page == 0
        next_button.disabled = page == len(pages) - 1


def change_page(step):
    global page
    page += step
    show_page()


previous.on_click(lambda b: change_page(-1))
next_button.on_click(lambda b: change_page(1))

display(widgets.HBox([previous, next_button, counter]), output)
show_page()

In [ ]:
# @title 💡 Sample Answer: Object Sizes { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>

<p>
Birds have the smallest bounding boxes by a wide margin, while cats, dogs, and horses are mostly represented by large boxes. Object sizes can vary considerably within a class, depending on the animal's distance from the camera.
</p>

<p>
Small objects are particularly challenging because they contain fewer pixels and less visual detail. This may make distant birds difficult to detect reliably.
</p>

</details>
"""))

### 🟢 Object Positions: Where Are the Animals?

A bounding box's centre is `(x + w/2, y + h/2)`. Dividing these coordinates by the image width and height gives normalized positions between 0 and 1, allowing us to compare images of different sizes.

A **heatmap** shows where bounding-box centres occur most frequently. Darker regions indicate more objects.

**Before running the code:**
- Do you expect animals to appear equally often across all image positions?

**Explore:**
- Compare the box-centre heatmaps across classes.
- Look for differences in how concentrated or spread out the positions are.

**Observe and interpret:**
- Where are most animals located within the images?
- Which classes have more widely distributed positions?
- Why might a detector trained on these images struggle with animals near the image edges?

In [ ]:
# @title 🔎 Explore Object Positions Across Classes { display-mode: "form" }
# Box centres on the unit square, one heatmap per class in its own colour (white = no boxes there).
_size = images.set_index("id")[["width", "height"]]
_img_w = boxes["image_id"].map(_size["width"])
_img_h = boxes["image_id"].map(_size["height"])
boxes["cx"] = (boxes["bbox"].map(lambda b: b[0] + b[2] / 2) / _img_w).clip(0, 1)
boxes["cy"] = (boxes["bbox"].map(lambda b: b[1] + b[3] / 2) / _img_h).clip(0, 1)
HEAT_BINS = 6 if SMOKE else 12  # coarser cells with few boxes (N-014)
_edge_share = float(((boxes["cx"] < 0.2) | (boxes["cx"] > 0.8) | (boxes["cy"] < 0.2) | (boxes["cy"] > 0.8))
                    .mean())
RESULTS["edge_box_share"] = _edge_share

fig, axes = plt.subplots(1, len(CLASSES), figsize=(2.3 * len(CLASSES), 2.8))
for ax, c in zip(axes, CLASSES, strict=True):
    own = boxes[boxes["class"] == c]
    cmap = mpl.colors.LinearSegmentedColormap.from_list(c, ["white", CLASS_COLORS[c]])
    heat, _, _ = np.histogram2d(own["cy"], own["cx"], bins=HEAT_BINS, range=[[0, 1], [0, 1]])
    ax.imshow(heat, cmap=cmap, extent=[0, 1, 1, 0], vmin=0)
    ax.set_title(f"{c} · {len(own)} boxes", fontsize=9, color=CLASS_COLORS[c])
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.grid(False)
axes[0].set_ylabel("box centre y (0 = top)")
fig.suptitle(f"Where the box centres are · {_edge_share:.0%} of all boxes are in the outer 20 % of the frame",
             color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer: Object Positions { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<p>
Bounding-box centres are concentrated near the middle of the images, suggesting
that animals are often photographed centrally. Birds and sheep show a wider
spatial distribution, possibly because multiple animals appear together.
</p>
<p>
A detector trained mainly on centrally positioned animals may perform less
reliably near image edges, where fewer training examples are available.
However, the heatmaps alone do not establish how well the detector performs.
</p>
</details>
"""))

### 🟢 Crowding: How Many Animals Appear Together?

Some images contain a single animal, while others show several animals close together. Crowded scenes can be challenging for object detection, particularly when animals partially hide one another.

**Before running the code:**
- Which animal classes do you expect to appear most often in images with many animals?

**Explore:**
- Compare the distributions of object counts across classes.
- Use the image browser to compare images with one, a few, and many annotated objects.

**Observe and interpret:**
- Which classes tend to have more objects per image?
- How do images with few objects differ from those with many objects?
- Does a higher object count always mean a more crowded scene? Why or why not?
- What challenges might these differences pose for object detection?

In [ ]:
# @title 🔎 Explore Object Counts Across Classes { display-mode: "form" }

# Count annotated objects per image
object_counts = (
    boxes.groupby("image_id")
    .size()
    .rename("n_objects")
    .reset_index()
)

# Include images without annotations
crowding = images[["id", "class"]].merge(
    object_counts, left_on="id", right_on="image_id", how="left"
)
crowding["n_objects"] = crowding["n_objects"].fillna(0).astype(int)

results["max_objects_per_image"] = int(crowding["n_objects"].max())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Distribution of object counts
count_distribution = crowding["n_objects"].value_counts().sort_index()

axes[0].bar(
    count_distribution.index,
    count_distribution.values,
    color="#B0B0B0"
)
axes[0].set_xlabel("Number of objects per image")
axes[0].set_ylabel("Number of images")
axes[0].set_title("Objects per image")

# Object counts by class
data = [
    crowding.loc[crowding["class"] == c, "n_objects"].to_numpy()
    for c in CLASSES
]

bp = axes[1].boxplot(
    data,
    tick_labels=CLASSES,
    showfliers=False,
    patch_artist=True
)

for box, c in zip(bp["boxes"], CLASSES, strict=True):
    box.set_facecolor(CLASS_COLORS[c])
    box.set_alpha(0.75)

for median in bp["medians"]:
    median.set_color(INK)
    median.set_linewidth(2)

for label, c in zip(axes[1].get_xticklabels(), CLASSES, strict=True):
    label.set_color(CLASS_COLORS[c])

axes[1].set_ylabel("Number of objects per image")
axes[1].set_title("Object counts by class")

fig.tight_layout()
plt.show()

In [ ]:
# @title 🔎 Explore Crowded Images { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output

# Sort images from most to fewest annotated objects
crowded_images = crowding.sort_values(
    ["n_objects", "id"], ascending=[False, True]
).reset_index(drop=True)

# Select object count (group counts of 10 or more)
available_counts = sorted(
    int(n) for n in crowding["n_objects"].unique() if n < 10
)

count_selector = widgets.Dropdown(
    options=(
        [("All", None)]
        + [(str(n), n) for n in available_counts]
        + ([("10 or more", "10+")] if (crowding["n_objects"] >= 10).any() else [])
    ),
    value=None,
    description="Objects:",
    style={"description_width": "initial"}
)

page = 0
page_size = 4

first = widgets.Button(description="⇤ First")
previous = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
last = widgets.Button(description="Last ⇥")
counter = widgets.HTML()
output = widgets.Output()


def get_filtered_images():
    if count_selector.value is None:
        return crowded_images
    if count_selector.value == "10+":
        return crowded_images[crowded_images["n_objects"] >= 10]
    return crowded_images[
        crowded_images["n_objects"] == count_selector.value
    ]


def show_crowded_page():
    filtered = get_filtered_images()
    total_pages = max(1, (len(filtered) + page_size - 1) // page_size)

    with output:
        clear_output(wait=True)

        selected = filtered.iloc[
            page * page_size:(page + 1) * page_size
        ]

        # Preserve the selected order
        selected_images = selected[["id", "n_objects"]].merge(
            images, on="id", how="left", sort=False
        )

        show_boxes(
            selected_images,
            boxes,
            "Images ordered by number of annotated objects",
            caption=lambda r: f"{r['class']} · {r['n_objects']} objects"
        )

    count_label = (
        "All counts" if count_selector.value is None
        else "10+ objects" if count_selector.value == "10+"
        else f"{count_selector.value} objects"
    )

    counter.value = (
        f"{count_label} · {len(filtered)} images · "
        f"Page {page + 1} of {total_pages}"
    )

    first.disabled = previous.disabled = page == 0
    next_button.disabled = last.disabled = page >= total_pages - 1


def go_to_page(new_page):
    global page
    total_pages = max(
        1, (len(get_filtered_images()) + page_size - 1) // page_size
    )
    page = max(0, min(new_page, total_pages - 1))
    show_crowded_page()


def change_count(change):
    if change["name"] == "value":
        go_to_page(0)


count_selector.observe(change_count, names="value")

first.on_click(lambda b: go_to_page(0))
previous.on_click(lambda b: go_to_page(page - 1))
next_button.on_click(lambda b: go_to_page(page + 1))
last.on_click(lambda b: go_to_page(10**9))

display(
    widgets.VBox([
        count_selector,
        widgets.HBox([first, previous, next_button, last, counter])
    ]),
    output
)

show_crowded_page()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<p>Images of herd animals, such as sheep or zebras, may contain more annotated objects than images of cats or dogs. Images with many objects often show animals close together, but some animals may still be clearly separated.</p>
<p>A higher object count does not necessarily mean more crowding. Object size, spacing, and occlusion also matter. Closely packed animals can be difficult to detect individually, increasing the risk of missed or duplicate detections.</p>
</details>
"""))

### 🔴 Bounding Box Overlap: Understanding IOU

Counting objects tells us how many animals are annotated, but not how closely they appear together. We can measure bounding-box overlap using **Intersection over Union (IOU)**:

$$
\mathrm{IOU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}
$$

IOU ranges from 0 (no overlap) to 1 (identical boxes).

**Before running the code:**
- What happens to IOU when two boxes move closer together?

**Explore:**
- Move the X and Y sliders to change how two bounding boxes overlap.
- Observe how IOU changes and when it exceeds the threshold of 0.1.

**Observe and interpret:**
- How does IOU change as the boxes move apart?
- Why is bounding-box overlap only an approximation of how crowded a scene is?

In [ ]:
# @title IOU Helper Function { display-mode: "form" }
def box_iou_dice(a: tuple, b: tuple) -> tuple[float, float]:
    """IoU and Dice of two boxes given as (x, y, w, h)."""
    ix = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = ix * iy
    area_a, area_b = a[2] * a[3], b[2] * b[3]
    return inter / (area_a + area_b - inter), 2 * inter / (area_a + area_b)

In [ ]:
# @title 🔎 Explore Bounding Box Overlap (IOU) { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display

overlap_iou = 0.1

box_grey = "#B0B0B0"
box_edge = "#666666"
fhnw_yellow = "#FFDA00"

a = (20, 20, 60, 40)


def show_overlap(x_shift, y_shift):
    b = (20 + x_shift, 20 + y_shift, 60, 40)

    iou = box_iou_dice(a, b)[0]

    ix0 = max(a[0], b[0])
    iy0 = max(a[1], b[1])
    iw = max(0, min(a[0] + a[2], b[0] + b[2]) - ix0)
    ih = max(0, min(a[1] + a[3], b[1] + b[3]) - iy0)

    intersection = iw * ih
    union = a[2] * a[3] + b[2] * b[3] - intersection

    fig, ax = plt.subplots(figsize=(7, 3.5))

    for box in [a, b]:
        ax.add_patch(mpl.patches.Rectangle(
            box[:2], *box[2:],
            facecolor=box_grey,
            edgecolor=box_edge,
            alpha=0.45,
            linewidth=1.5
        ))

    if intersection > 0:
        ax.add_patch(mpl.patches.Rectangle(
            (ix0, iy0), iw, ih,
            facecolor=fhnw_yellow,
            edgecolor=INK,
            linewidth=1.5
        ))

    ax.set_xlim(10, 160)
    ax.set_ylim(120, 10)
    ax.set_aspect("equal")
    ax.axis("off")

    ax.set_title(
        f"IOU = {intersection} / {union} = {iou:.2f} "
        f"({'overlapping' if iou > overlap_iou else 'not counted'})",
        fontsize=11
    )

    plt.show()
    plt.close(fig)


x_slider = widgets.IntSlider(
    value=25, min=0, max=90, step=5,
    description="X shift:",
    continuous_update=False
)

y_slider = widgets.IntSlider(
    value=10, min=0, max=70, step=5,
    description="Y shift:",
    continuous_update=False
)

output = widgets.interactive_output(
    show_overlap,
    {"x_shift": x_slider, "y_shift": y_slider}
)

display(widgets.VBox([x_slider, y_slider]), output)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<p>IOU increases as the boxes overlap more strongly, reaching 1 when they are identical. When the boxes move apart, IOU decreases and becomes 0 when they no longer intersect.</p>
<p>Bounding-box overlap is only an approximation of crowding. Boxes can overlap even when the animals themselves do not, and animals can partially occlude one another without their boxes having a high IOU.</p>
</details>
"""))

### 🔴 Crowding: Bounding Box Overlap Across Classes

Images with many animals do not necessarily contain overlapping bounding boxes. Let's investigate how often bounding boxes overlap in our dataset.

We consider two bounding boxes overlapping if their IOU exceeds 0.1.

**Before running the code:**
- Which animal classes do you expect to have the most overlapping bounding boxes?

**Explore:**
- Compare the number of overlapping box pairs across classes.
- Compare the proportion of overlapping pairs relative to all possible pairs.

**Observe and interpret:**
- Which classes have the most overlapping box pairs?
- Does this ranking change when considering the proportion of overlapping pairs?
- Why might bounding-box overlap be challenging for object detection?

In [ ]:
# @title 🔎 Explore Bounding Box Overlap Across Classes { display-mode: "form" }

from itertools import combinations

overlap_iou = 0.1

# Count overlapping box pairs in each image
rows = []

for image_id, group in boxes.groupby("image_id"):
    bounding_boxes = group["bbox"].tolist()
    n_boxes = len(bounding_boxes)
    n_pairs = n_boxes * (n_boxes - 1) // 2

    n_overlapping = sum(
        box_iou_dice(a, b)[0] > overlap_iou
        for a, b in combinations(bounding_boxes, 2)
    )

    rows.append({
        "image_id": image_id,
        "n_boxes": n_boxes,
        "n_pairs": n_pairs,
        "n_overlapping": n_overlapping
    })

overlap_counts = pd.DataFrame(rows)

# Associate each image with its class
overlap_counts = overlap_counts.merge(
    images[["id", "class"]],
    left_on="image_id",
    right_on="id",
    how="left"
)

# Aggregate by class
overlap_by_class = (
    overlap_counts.groupby("class")
    .agg(
        boxes_per_image=("n_boxes", "mean"),
        overlapping_pairs=("n_overlapping", "sum"),
        possible_pairs=("n_pairs", "sum")
    )
    .reindex(CLASSES)
)

overlap_by_class["overlap_share"] = (
    100
    * overlap_by_class["overlapping_pairs"]
    / overlap_by_class["possible_pairs"].replace(0, float("nan"))
)

# Visualize overlap across classes
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

colors = [CLASS_COLORS[c] for c in CLASSES]

axes[0].bar(
    CLASSES,
    overlap_by_class["overlapping_pairs"],
    color=colors
)
axes[0].set_title("Overlapping box pairs")
axes[0].set_ylabel("Number of pairs")

axes[1].bar(
    CLASSES,
    overlap_by_class["overlap_share"],
    color=colors
)
axes[1].set_title("Share of overlapping pairs")
axes[1].set_ylabel("Overlapping pairs (%)")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    for label, c in zip(ax.get_xticklabels(), CLASSES):
        label.set_color(CLASS_COLORS[c])

fig.tight_layout()
plt.show()

display(
    overlap_by_class[
        ["boxes_per_image", "overlapping_pairs", "overlap_share"]
    ].round(1)
)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<p>Classes with many animals per image may have more overlapping box pairs. However, the ranking can change when comparing the proportion of overlapping pairs, because images with more boxes also have more possible pairs.</p>
<p>Overlapping boxes make it harder to distinguish individual animals and assign accurate bounding boxes. This can lead to missed objects, inaccurate localization, or duplicate detections. The overlap statistics describe bounding boxes, not necessarily actual occlusion between animals.</p>
</details>
"""))

### 🟢 Bounding Box Quality: How Well Do the Boxes Fit?

Bounding boxes should enclose the annotated animal as closely as possible. However, annotations may be too large, too small, or incorrectly positioned.

**Before running the code:**
- What makes a bounding box a good annotation?

**Explore:**
- Browse images from different animal classes and inspect their bounding boxes.
- Check whether each box tightly encloses the animal, covers its visible parts, and corresponds to the correct animal.
- Look for examples where accurate annotation is difficult.

**Observe and interpret:**
- How well do the bounding boxes fit the visible animals?
- Which images or animal characteristics make accurate annotation difficult?
- How might inaccurate bounding boxes affect object detection?

In [ ]:
# @title 🔎 Inspect Bounding Box Quality { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output

# Browse annotated images by class
class_selector = widgets.Dropdown(
    options=CLASSES,
    value=CLASSES[0],
    description="Class:",
    style={"description_width": "initial"}
)

page_size = 4
quality_page = 0

previous = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
shuffle = widgets.Button(description="Shuffle")
counter = widgets.HTML()
output = widgets.Output()

rng = np.random.default_rng(SEED)
quality_images = images[images["class"] == class_selector.value].copy()


def show_quality_page():
    with output:
        clear_output(wait=True)

        selected = quality_images.iloc[
            quality_page * page_size:(quality_page + 1) * page_size
        ]

        show_boxes(
            selected,
            boxes,
            "Inspect bounding box fit",
            caption=lambda r: f"{r['class']} · Image {r['id']}"
        )

    total_pages = max(1, (len(quality_images) + page_size - 1) // page_size)
    counter.value = f"Page {quality_page + 1} of {total_pages}"
    previous.disabled = quality_page == 0
    next_button.disabled = quality_page >= total_pages - 1


def change_quality_page(step):
    global quality_page
    quality_page += step
    show_quality_page()


def update_quality_images(change=None):
    global quality_page, quality_images
    quality_page = 0

    quality_images = images[
        images["class"] == class_selector.value
    ].sample(frac=1, random_state=SEED).reset_index(drop=True)

    show_quality_page()


def shuffle_quality_images(button):
    global quality_page, quality_images
    quality_page = 0

    quality_images = quality_images.sample(
        frac=1,
        random_state=int(rng.integers(0, 2**31 - 1))
    ).reset_index(drop=True)

    show_quality_page()


class_selector.observe(update_quality_images, names="value")
previous.on_click(lambda b: change_quality_page(-1))
next_button.on_click(lambda b: change_quality_page(1))
shuffle.on_click(shuffle_quality_images)

display(
    widgets.VBox([
        class_selector,
        widgets.HBox([previous, next_button, shuffle, counter])
    ]),
    output
)

update_quality_images()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<p>Bounding boxes do not always fit tightly around animals. For example, cats with long tails may require large boxes even though much of the box contains background. In another image, a cat wearing a hat has only its face annotated rather than its entire body.</p>
<p>Animals are not always fully visible; sometimes only parts of their bodies appear in the image. A reflection of a cat on a car is not annotated, although it looks like a real cat. These examples raise questions about annotation guidelines and which visible objects or parts should be labeled.</p>
<p>For very small animals, it is difficult to judge box accuracy. Partial visibility, occlusion, unusual poses, and unclear boundaries make annotation challenging. Inconsistent annotations can affect how accurately an object detector learns to localize animals.</p>
</details>
"""))

### 🟢 Missing Annotations: Find the Missing Box

Bounding-box annotations are not always complete. An annotator may overlook an animal, especially in crowded scenes. During training, an unannotated animal may then be treated as background.

To simulate such an annotation error, we create a modified copy of our annotations (`park_boxes`) and deliberately remove **one bounding box**. The original annotations remain unchanged.

Your task is to identify the affected image and locate the missing box.

**Before running the code:**
- How could you detect a missing bounding box without manually inspecting every image?

**Explore:**
- Compare the number of boxes in `park_boxes` with the recorded number of animals per image.
- Identify images where these counts disagree.
- Inspect the suspicious image and locate the missing bounding box.

**Observe and interpret:**
- Which animal is missing a bounding box?
- Why might this annotation error be difficult to notice?
- How could missing annotations affect the training and evaluation of an object detector?

In [ ]:
# @title 🔎 Simulate and Detect a Missing Annotation { display-mode: "form" }

import random

# Simulate an annotation error by removing exactly one bounding box
# from a copy of the original annotations.
# The original boxes dataframe remains unchanged.

def simulate_missing_box(boxes_df, images_df, seed):
    rng = random.Random(seed)

    counts = boxes_df.groupby("image_id").size()
    image_classes = images_df.set_index("id")["class"]

    # Prefer a crowded sheep image
    candidates = counts[counts >= 3].index.tolist()
    sheep_candidates = [
        image_id for image_id in candidates
        if image_classes.get(image_id) == "sheep"
    ]

    if sheep_candidates:
        candidates = sheep_candidates
    if not candidates:
        candidates = [counts.idxmax()]

    image_id = rng.choice(sorted(candidates))
    image_boxes = boxes_df[boxes_df["image_id"] == image_id]
    missing_box_id = rng.choice(sorted(image_boxes["id"]))

    simulated_boxes = boxes_df[
        boxes_df["id"] != missing_box_id
    ].copy()

    answer_key = {
        "image_id": image_id,
        "annotation_id": missing_box_id
    }

    return simulated_boxes, answer_key


# Create the simulated dataset (keep the answer key hidden)
park_boxes, hunt_answer_key = simulate_missing_box(
    boxes, images, SEED
)

# Compare recorded animal counts with the simulated annotations
box_counts = park_boxes.groupby("image_id").size()

annotation_check = images[["id", "class", "n_animals"]].copy()
annotation_check["n_boxes"] = (
    annotation_check["id"].map(box_counts).fillna(0).astype(int)
)
annotation_check["difference"] = (
    annotation_check["n_animals"] - annotation_check["n_boxes"]
)

suspicious_images = annotation_check[
    annotation_check["difference"] != 0
].copy()

print("Simulation: One bounding box was deliberately removed.")
print(f"Images checked: {len(annotation_check)}")
print(f"Images with mismatched counts: {len(suspicious_images)}")

display(
    suspicious_images[
        ["id", "class", "n_animals", "n_boxes", "difference"]
    ].rename(columns={
        "id": "Image ID",
        "class": "Class",
        "n_animals": "Animals",
        "n_boxes": "Boxes",
        "difference": "Missing boxes"
    })
)

In [ ]:
# @title 🔎 Inspect Missing Annotations { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display

image_selector = widgets.Dropdown(
    options=[
        (f"Image {image_id}", image_id)
        for image_id in suspicious_images["id"]
    ],
    description="Image ID:",
    style={"description_width": "initial"}
)

output = widgets.Output()


def inspect_annotations(change=None):
    with output:
        output.clear_output(wait=True)

        selected_images = images[
            images["id"] == image_selector.value
        ]

        show_boxes(
            selected_images,
            park_boxes,
            "Inspect the available bounding boxes",
            caption=lambda r: (
                f"{r['class']} · "
                f"{r['n_animals']} animals expected"
            )
        )


image_selector.observe(inspect_annotations, names="value")

display(image_selector, output)
inspect_annotations()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF; border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<p><strong>Finding the error:</strong> Comparing the recorded number of animals with the remaining bounding boxes reveals the affected image. Inspecting that image shows an animal without a bounding box.</p>
<p><strong>Why it matters:</strong> Missing annotations can be difficult to notice in crowded scenes, particularly when animals overlap or are partially hidden. During training, an unannotated animal may be treated as background, potentially confusing the detector. During evaluation, a correct detection of that animal may incorrectly count as a false positive.</p>
</details>
"""))

#### Visualizing a Missing Annotation

The illustration below shows a simplified example: three animals are present, but only two have bounding boxes.

The solid grey boxes represent existing annotations. The dashed yellow box indicates the missing annotation.

Consider how this mismatch could affect an object detector trained on these annotations.

In [ ]:
# @title 🔎 Visualize a Missing Bounding Box { display-mode: "form" }

box_grey = "#B0B0B0"
fhnw_yellow = "#FFDA00"

fig, ax = plt.subplots(figsize=(7, 2.5))

for k, x in enumerate([2, 5, 8]):
    # Draw an animal
    ax.add_patch(mpl.patches.Ellipse(
        (x, 1), 2.0, 1.2,
        facecolor=box_grey, alpha=0.45,
        edgecolor="none"
    ))

    # Draw existing annotations or highlight the missing one
    missing = k == 2

    ax.add_patch(mpl.patches.Rectangle(
        (x - 1.3, 0.1), 2.6, 1.8,
        fill=False,
        linewidth=2,
        linestyle="--" if missing else "-",
        edgecolor=fhnw_yellow if missing else box_grey
    ))

    if missing:
        ax.text(
            x, 1, "Missing box",
            ha="center", va="center",
            fontsize=10, color=INK,
            fontweight="bold"
        )

ax.set_xlim(0, 10)
ax.set_ylim(-0.1, 2.2)
ax.set_aspect("equal")
ax.axis("off")

ax.set_title(
    "3 animals present · 2 boxes annotated · 1 missing",
    fontsize=11, color=INK
)

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 📘 What Matters for Object Detection? { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1;
            padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 What Matters for Object Detection?</strong>
<p>Beyond image classification, object detection requires us to understand
the characteristics and quality of individual object annotations.</p>

<ul style="margin-bottom:0;">
<li><strong>Class distribution:</strong> Counting images and bounding boxes can reveal different class imbalances.</li>
<li><strong>Object size:</strong> Small objects contain fewer pixels and may be harder to detect.</li>
<li><strong>Object position:</strong> Position biases may affect generalization to less common locations.</li>
<li><strong>Crowding:</strong> Closely spaced or overlapping objects can be difficult to distinguish.</li>
<li><strong>Annotation quality:</strong> Missing or inaccurate boxes can affect training and evaluation.</li>
</ul>

<p style="margin-bottom:0; margin-top:12px;">
<strong>Key principle:</strong> Always consider what is being counted and how
dataset characteristics relate to the intended computer vision task.
</p>

</div>
"""))


## Semantic Segmentation: Which Pixels Belong to the Animals?

So far, we have explored two ways of describing animals in images:

- **Image classification:** Which animal class is in the image?
- **Object detection:** Where are the individual animals, represented by bounding boxes?

*Semantic segmentation* goes one step further: it assigns a class to every pixel, distinguishing animal areas from the background.

Our dataset contains segmentation annotations in addition to class labels and bounding boxes. These annotations allow us to explore the actual areas occupied by animals, rather than just the rectangles surrounding them.



In [ ]:
# @title Semantic Segmentation: Masks and Pixels { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1;
            padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 Semantic Segmentation: Masks and Pixels</strong>

<ul>
<li><strong>Binary mask:</strong> A two-dimensional array indicating which pixels
belong to an object (1) and which do not (0).</li>

<li><strong>Semantic segmentation:</strong> Assigns a class label to every pixel.
Pixels that do not belong to any animal are labeled as <em>background</em>.</li>

<li><strong>Instance segmentation:</strong> Identifies individual objects using
separate masks, even when they belong to the same class.</li>
</ul>

<p style="margin-bottom:0;">
Our COCO dataset provides instance-level segmentation annotations.
We can combine these masks to obtain pixel-level class labels for semantic segmentation.
</p>

</div>
"""))

### 🟢 Exploring Segmentation Masks

Segmentation masks show which pixels belong to animals, rather than just the rectangular areas defined by bounding boxes.

**Before running the code:**
- How much of an image do you expect to be covered by animals compared with the background?

**Explore:**
- Browse images with their segmentation masks overlaid.
- Compare the masks across different animals and images.

**Observe and interpret:**
- How much of each image is covered by animals, and how much is background?
- How do masks represent animal shapes differently from bounding boxes?
- Which animal characteristics might make accurate segmentation difficult?

In [ ]:
# @title Mask helpers and data preparation { display-mode: "form" }
import cv2
from pycocotools import mask as mask_utils

MIN_JUDGED_AREA = 32**2  # masks under COCO's "small" limit (32×32 px) are too few pixels to judge their shape
THIN_PX = 7  # parts narrower than this many photo pixels count as thin (about 2 px at 224×224)
BLOB_SHARE = 0.005  # pieces and holes under 0.5 % of the mask (or 16 px) are rasterisation noise, not shape


def to_rle(segmentation, height: int, width: int) -> dict:
    """A COCO segmentation (polygon list, uncompressed or compressed RLE) as one compressed RLE."""
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation["counts"], list):
        return mask_utils.frPyObjects(segmentation, height, width)
    return segmentation


def decode(rle: dict) -> np.ndarray:
    """RLE → 0/1 uint8 array of the photo's size (C-ordered, so OpenCV accepts it)."""
    return np.ascontiguousarray(mask_utils.decode(rle))


def encode(mask: np.ndarray) -> dict:
    """0/1 array → compressed RLE, the way COCO stores masks."""
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))


def thin_part(mask: np.ndarray) -> np.ndarray:
    """The pixels of `mask` in parts narrower than THIN_PX: what an opening with a THIN_PX disc removes."""
    disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (THIN_PX, THIN_PX))
    return mask & (1 - cv2.morphologyEx(mask, cv2.MORPH_OPEN, disc))


def mask_shape(mask: np.ndarray, bbox) -> dict:
    """Shape numbers of one mask (0/1 array) against its annotated box (x, y, w, h).

    box_iou: IoU of the mask's own tight box with the annotated box · fill: mask area / box area ·
    pieces, holes: separate parts and enclosed gaps · solidity: mask area / area of its convex hull (a rubber
    band around it; 1.0 = no dents at all) · thin: share of the mask's pixels in thin parts.
    """
    area = int(mask.sum())
    if area == 0:
        return {"box_iou": 0.0, "fill": 0.0, "pieces": 0, "holes": 0, "solidity": 0.0, "thin": 0.0}
    mx, my, mw, mh = cv2.boundingRect(mask)
    crop = np.pad(mask[my:my + mh, mx:mx + mw], 1)
    min_blob = max(16, BLOB_SHARE * area)
    _, _, parts, _ = cv2.connectedComponentsWithStats(crop, connectivity=8)
    pieces = int((parts[1:, cv2.CC_STAT_AREA] >= min_blob).sum())
    _, labels, gaps, _ = cv2.connectedComponentsWithStats(1 - crop, connectivity=4)
    outside = labels[0, 0]  # the background that touches the border is not a hole
    holes = sum(int(gaps[i, cv2.CC_STAT_AREA] >= min_blob) for i in range(1, len(gaps)) if i != outside)
    contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = np.zeros_like(crop)
    cv2.fillPoly(hull, [cv2.convexHull(np.vstack(contours))], 1)
    w, h = bbox[2], bbox[3]
    return {"box_iou": box_iou_dice((mx, my, mw, mh), tuple(bbox))[0],
            "fill": area / max(w * h, 1),
            "pieces": pieces,
            "holes": holes,
            "solidity": area / max(int(hull.sum()), 1),
            "thin": int(thin_part(crop).sum()) / area}



def show_masks(rows_df: pd.DataFrame, masks_df: pd.DataFrame, title: str, caption=None, cols: int = 4,
               size: int = 384, mark_thin: bool = False, highlight: dict | None = None,
               show_mask: bool = True, show_box: bool = True) -> None:
    """Gallery of photos with optional segmentation masks and bounding boxes."""
    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return

    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(3.4 * cols, 3.2 * n_rows), squeeze=False)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        thumb = load_thumb(row["path"], (size, size))
        scale = thumb.width / row["width"]
        own = masks_df[masks_df["image_id"] == row["id"]]

        flagged = (highlight or {}).get(row["id"])
        if flagged is not None:
            own = own.assign(_top=own["id"] == flagged).sort_values("_top")

        ax.axis("on")
        ax.imshow(thumb)

        if show_mask:
            overlay = np.zeros((thumb.height, thumb.width, 4))

            for _, m in own.iterrows():
                full = decode(m["rle"])
                small = cv2.resize(full, thumb.size, interpolation=cv2.INTER_NEAREST) > 0
                alpha = 0.6 if flagged is None else (0.8 if m["id"] == flagged else 0.15)
                overlay[small] = (*mpl.colors.to_rgb(CLASS_COLORS[m["class"]]), alpha)

                if mark_thin:
                    thin = cv2.resize(thin_part(full), thumb.size, interpolation=cv2.INTER_NEAREST) > 0
                    overlay[thin] = (*mpl.colors.to_rgb(INK), 0.95)

            ax.imshow(overlay)

        if show_box:
            for _, m in own.iterrows():
                x, y, w, h = m["bbox"]
                if flagged is not None and m["id"] != flagged:
                    continue

                ax.add_patch(mpl.patches.Rectangle(
                    (x * scale, y * scale), w * scale, h * scale,
                    fill=False,
                    lw=2.5 if flagged is not None else 1.5,
                    ls="-" if flagged is not None else "--",
                    edgecolor=CLASS_COLORS[m["class"]]
                ))

        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row) if caption else f"{row['class']} · {len(own)} masks", fontsize=9)

    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


# Prepare masks for subsequent inspection activities.
image_sizes = images.set_index("id")[["width", "height"]]

masks = boxes[
    ["id", "image_id", "class", "bbox", "iscrowd", "segmentation"]
].copy()

masks["rle"] = [
    to_rle(seg, int(image_sizes.at[i, "height"]), int(image_sizes.at[i, "width"]))
    for seg, i in zip(masks["segmentation"], masks["image_id"], strict=True)
]

masks["mask_area"] = [int(mask_utils.area(r)) for r in masks["rle"]]
masks = masks.drop(columns="segmentation")

In [ ]:

# @title 🔎 Explore Segmentation Masks { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

mask_images = images[images["id"].isin(masks["image_id"].unique())].copy()
mask_images = mask_images.sample(frac=1, random_state=SEED).reset_index(drop=True)

page = 0
page_size = 4

prev_btn = widgets.Button(description="Previous", icon="chevron-left")
next_btn = widgets.Button(description="Next", icon="chevron-right")
shuffle_btn = widgets.Button(description="Shuffle", icon="shuffle")

mask_toggle = widgets.Checkbox(value=True, description="Show masks", indent=False)
box_toggle = widgets.Checkbox(value=True, description="Show boxes", indent=False)

page_label = widgets.HTML()
output = widgets.Output()

def show_page():
    total_pages = (len(mask_images) + page_size - 1) // page_size
    start = page * page_size
    selected = mask_images.iloc[start:start + page_size]

    page_label.value = f"Page {page + 1} of {total_pages}"
    prev_btn.disabled = (page == 0)
    next_btn.disabled = (page >= total_pages - 1)

    with output:
        output.clear_output(wait=True)
        show_masks(
            selected,
            masks,
            "Segmentation masks",
            caption=lambda r: r["class"],
            show_mask=mask_toggle.value,
            show_box=box_toggle.value
        )

def navigate(change):
    global page, mask_images

    total_pages = (len(mask_images) + page_size - 1) // page_size

    if change == "previous":
        page = max(0, page - 1)
    elif change == "next":
        page = min(total_pages - 1, page + 1)
    elif change == "shuffle":
        mask_images = mask_images.sample(frac=1).reset_index(drop=True)
        page = 0

    show_page()

prev_btn.on_click(lambda b: navigate("previous"))
next_btn.on_click(lambda b: navigate("next"))
shuffle_btn.on_click(lambda b: navigate("shuffle"))

mask_toggle.observe(lambda change: show_page(), names="value")
box_toggle.observe(lambda change: show_page(), names="value")

display(
    widgets.HBox([prev_btn, next_btn, shuffle_btn, page_label]),
    widgets.HBox([mask_toggle, box_toggle]),
    output
)

show_page()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>

<ul>
<li><strong>Animal vs. background:</strong> The proportion of animal pixels varies
considerably. Some animals occupy large parts of an image, while others cover
only a small area.</li>

<li><strong>Masks vs. bounding boxes:</strong> Masks follow animal shapes more
closely, excluding background pixels that fall inside bounding boxes.</li>

<li><strong>Segmentation challenges:</strong> Thin structures such as legs, tails,
and beaks, as well as small or partially hidden animals, can be difficult
to segment accurately.</li>
</ul>

</details>
"""))

### 🟢 Pixel-Level Class Imbalance

In semantic segmentation, class distribution is measured by pixels rather than images or objects.

**Before running the code:**
- Do you expect background pixels to make up a quarter, half, or most of the dataset?
- Do you expect birds to be frequent or rare when measured by pixels?

**Explore:**
- Compare class distributions measured by images, bounding boxes, and pixels, including background.

**Observe and interpret:**
- How dominant is the background?
- Which classes change most in relative frequency when counting pixels instead of images or objects?
- How might these differences affect the training and evaluation of a segmentation model?

In [ ]:
# @title Pixel-Level Class Imbalance { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<div style="background:#E3F2FD; border-left:4px solid #0D47A1;
            padding:14px 18px; border-radius:6px; color:#0D47A1;">

<strong>📘 Pixel-Level Class Imbalance</strong>

<p>In semantic segmentation, each pixel is assigned a class label,
including background. This changes how we measure class imbalance.</p>

<ul style="margin-bottom:0;">
<li><strong>Background dominance:</strong> Large parts of an image may contain
no animals, resulting in many background pixels.</li>
<li><strong>Object size:</strong> Large animals contribute more labeled pixels
than small animals, even when both count as one object.</li>
<li><strong>Training implications:</strong> With an unweighted pixel-wise loss,
larger regions generally contribute more loss terms. Class weighting or
alternative loss functions can help address this imbalance.</li>
</ul>

</div>
"""))

In [ ]:

# @title 🔎 Explore Pixel-Level Class Imbalance { display-mode: "form" }

# Calculate pixel shares
photo_rle = {}  # One merged mask per photo, reused by the heatmaps below
animal_px = dict.fromkeys(CLASSES, 0)

for image_id, own in masks.groupby("image_id"):
    photo_rle[image_id] = mask_utils.merge(list(own["rle"]), intersect=False)
    animal_px[own["class"].iloc[0]] += int(mask_utils.area(photo_rle[image_id]))

total_px = int((images["width"] * images["height"]).sum())

pixels_per_class = {c: animal_px[c] / total_px for c in CLASSES}
pixels_per_class["background"] = (
    total_px - sum(animal_px.values())
) / total_px

RESULTS["pixels_per_class"] = pixels_per_class

BACKGROUND_COLOR = "#D9DEDF"

# Compare class distributions across counting units
units = {
    "photos": {
        c: photos_per_class[c] / sum(photos_per_class.values())
        for c in CLASSES
    },
    "boxes": {
        c: boxes_per_class[c] / sum(boxes_per_class.values())
        for c in CLASSES
    },
    "pixels": {
        c: animal_px[c] / max(sum(animal_px.values()), 1)
        for c in CLASSES
    },
}

fig, axes = plt.subplots(
    1, 2, figsize=(12, 3.8),
    gridspec_kw={"width_ratios": [2.2, 1]}
)

# Left: class distribution among animals only
ax = axes[0]
x = np.arange(len(CLASSES))

for k, (unit, alpha) in enumerate([
    ("photos", 0.3),
    ("boxes", 0.6),
    ("pixels", 1.0)
]):
    values = [units[unit][c] for c in CLASSES]

    bars = ax.bar(
        x + (k - 1) * 0.27,
        values,
        width=0.27,
        alpha=alpha,
        color=[CLASS_COLORS[c] for c in CLASSES],
        edgecolor="white"
    )

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.004,
            f"{value:.0%}",
            ha="center",
            va="bottom",
            fontsize=7
        )

ax.set_xticks(x, CLASSES)

for label in ax.get_xticklabels():
    label.set_color(CLASS_COLORS[label.get_text()])

ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_ylim(0, max(max(v.values()) for v in units.values()) * 1.18)
ax.set_title("Class distribution among animals: photos vs. boxes vs. pixels")

legend = [
    mpl.patches.Patch(color=INK, alpha=alpha, label=unit)
    for unit, alpha in [
        ("photos", 0.3),
        ("boxes", 0.6),
        ("pixels", 1.0)
    ]
]
ax.legend(handles=legend, frameon=False, fontsize=8)

# Right: pixel distribution including background
ax = axes[1]
left = 0.0

for name in ["background", *CLASSES]:
    share = pixels_per_class[name]

    ax.barh(
        0,
        share,
        left=left,
        color=BACKGROUND_COLOR if name == "background" else CLASS_COLORS[name],
        edgecolor="white"
    )

    if name == "background":
        ax.text(
            left + share / 2,
            0,
            f"Background\n{share:.1%}",
            ha="center",
            va="center",
            fontsize=10,
            color=INK
        )

    left += share

ax.set_xlim(0, 1)
ax.set_yticks([])
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("All pixels, background included")

fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>

<ul>
<li><strong>Background dominance:</strong> About 85% of all pixels belong to the background.
Animals occupy only about 15%.</li>

<li><strong>Class differences:</strong> Birds account for about 1% of all pixels,
despite representing 25% of bounding boxes. They are often small or distant.
Cats and dogs occupy relatively more pixels, partly because they frequently
appear in close-up images.</li>

<li><strong>Implications:</strong> Class frequencies depend on whether we count
images, objects, or pixels. Background dominance and small animal regions
can make segmentation challenging, particularly when using an unweighted
pixel-wise loss.</li>
</ul>

</details>
"""))

### 🟢 Mask Size, Shape, and Fine Structures

A segmentation mask describes the pixels occupied by an animal. Its **fill ratio** measures how much of its bounding box is covered by the mask:

$$
\text{Fill ratio} = \frac{\text{Mask area}}{\text{Bounding box area}}
$$

Masks also reveal fine structures, such as legs, tails, and beaks, which may be difficult to preserve when images are resized.

**Before running the code:**
- What proportion of its bounding box do you expect a typical animal to fill: about 25%, 50%, or 80%?
- Which animal class do you expect to have the lowest fill ratio?
- Which classes might have the most masks containing thin structures?

**Explore:**
- Compare the distributions of mask sizes and fill ratios across animal classes.
- Inspect examples with low, typical, and high fill ratios.
- Compare the proportion of masks with thin regions across classes, including and excluding small masks.
- Inspect examples with particularly thin regions.

**Observe and interpret:**
- How do mask sizes and fill ratios differ between classes?
- What characteristics of an animal might explain a low fill ratio?
- Why can two animals with similar bounding box sizes have very different mask areas?
- How does excluding small masks change the frequency of thin regions, and why?
- What challenges might thin structures create when resizing masks?

In [ ]:
# @title 🔎 Explore Mask Sizes and Fill Ratios { display-mode: "form" }
animals = masks[masks["iscrowd"] == 0].reset_index(drop=True)
_shape = pd.DataFrame([mask_shape(decode(r), b)
                       for r, b in zip(animals["rle"], animals["bbox"], strict=True)])
animals = pd.concat([animals, _shape], axis=1)
_photo_px = animals["image_id"].map(_size["width"] * _size["height"])
animals["photo_share"] = animals["mask_area"] / _photo_px
_fill = animals.groupby("class")["fill"].median().reindex(CLASSES)
RESULTS["median_fill"] = {c: float(_fill[c]) for c in CLASSES}
print(f"{len(animals)} animal masks measured · median fill ratio " + " · ".join(
    f"{c} {_fill[c]:.0%}" for c in CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, label, log in [(axes[0], "photo_share", "mask area, share of the photo (log)", True),
                            (axes[1], "fill", "fill ratio = mask area / box area", False)]:
    data = [animals.loc[animals["class"] == c, col].clip(lower=1e-5).to_numpy() for c in CLASSES]
    if log:  # violins are drawn on log10(value) so the KDE is smooth on a log axis; ticks are relabelled
        data = [np.log10(v) for v in data]
    parts = ax.violinplot(data, showmedians=True, showextrema=False, widths=0.8)
    for body, c in zip(parts["bodies"], CLASSES, strict=True):
        body.set_facecolor(CLASS_COLORS[c])
        body.set_alpha(0.75)
    parts["cmedians"].set_color(INK)
    ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES)
    if log:
        _ticks = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]
        ax.set_yticks(np.log10(_ticks), [f"{100 * t:g}%" for t in _ticks])
    else:
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_ylabel(label)
    for lab in ax.get_xticklabels():
        lab.set_color(CLASS_COLORS[lab.get_text()])
axes[0].set_title("How big is a mask?")
axes[1].set_title("How much of its box does it fill?")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:

# @title 🔎 Inspect Mask Fill Ratios { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

# Select examples, excluding very small masks
candidates = (
    animals[animals["mask_area"] >= MIN_JUDGED_AREA]
    .sort_values(["fill", "id"])
    .drop_duplicates("image_id")
    .reset_index(drop=True)
)

n_examples = min(20, len(candidates) // 3)

examples = {
    "Low": candidates.head(n_examples).reset_index(drop=True),
    "Typical": candidates.iloc[
        (len(candidates) - n_examples) // 2:
        (len(candidates) + n_examples) // 2
    ].reset_index(drop=True),
    "High": candidates.tail(n_examples).reset_index(drop=True)
}

category = widgets.ToggleButtons(
    options=["Low", "Typical", "High"],
    description="Fill ratio:"
)
prev_btn = widgets.Button(description="Previous", icon="chevron-left")
next_btn = widgets.Button(description="Next", icon="chevron-right")
mask_toggle = widgets.Checkbox(value=True, description="Show masks", indent=False)
box_toggle = widgets.Checkbox(value=True, description="Show boxes", indent=False)
page_label = widgets.HTML()
output = widgets.Output()

page = 0
page_size = 4
image_lookup = images.set_index("id")

def show_page():
    selected = examples[category.value]
    total_pages = max(1, (len(selected) + page_size - 1) // page_size)
    start = page * page_size
    current = selected.iloc[start:start + page_size]

    page_label.value = f"Page {page + 1} of {total_pages}"
    prev_btn.disabled = page == 0
    next_btn.disabled = page >= total_pages - 1

    image_rows = image_lookup.loc[current["image_id"]].reset_index()
    fill_values = dict(zip(current["image_id"], current["fill"]))

    with output:
        output.clear_output(wait=True)
        show_masks(
            image_rows,
            masks,
            f"{category.value} fill ratios",
            caption=lambda r: f"{r['class']} · fill {fill_values[r['id']]:.0%}",
            show_mask=mask_toggle.value,
            show_box=box_toggle.value,
            highlight=dict(zip(current["image_id"], current["id"]))
        )

def navigate(direction):
    global page
    total_pages = max(1, (len(examples[category.value]) + page_size - 1) // page_size)
    page = max(0, min(page + direction, total_pages - 1))
    show_page()

def change_category(change):
    global page
    if change["name"] == "value":
        page = 0
        show_page()

prev_btn.on_click(lambda b: navigate(-1))
next_btn.on_click(lambda b: navigate(1))
category.observe(change_category, names="value")
mask_toggle.observe(lambda change: show_page(), names="value")
box_toggle.observe(lambda change: show_page(), names="value")

display(
    widgets.HBox([category, prev_btn, next_btn, page_label]),
    widgets.HBox([mask_toggle, box_toggle]),
    output
)
show_page()

In [ ]:
# @title 🔎 Explore Thin Structures in Segmentation Masks { display-mode: "form" }

THIN_SHARE = 0.05  # a mask "has thin parts" when at least this share of its pixels is thin
_big = animals["mask_area"] >= MIN_JUDGED_AREA
_has_thin = animals["thin"] >= THIN_SHARE
thin_by_class = pd.DataFrame({
    "all masks": _has_thin.groupby(animals["class"]).mean(),
    f"masks ≥ {int(MIN_JUDGED_AREA**0.5)}² px": _has_thin[_big].groupby(animals.loc[_big, "class"]).mean(),
}).reindex(CLASSES).fillna(0)
RESULTS["thin_mask_share"] = {c: float(thin_by_class.loc[c, "all masks"]) for c in CLASSES}
display((thin_by_class * 100).round(0).astype(int).astype(str).add(" %").T)

fig, ax = plt.subplots(figsize=(8, 3.2))
_x = np.arange(len(CLASSES))
for k, (col, alpha) in enumerate([(thin_by_class.columns[0], 1.0), (thin_by_class.columns[1], 0.45)]):
    ax.bar(_x + (k - 0.5) * 0.38, thin_by_class[col], width=0.38, alpha=alpha,
           color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="white", label=col)
ax.set_xticks(_x, CLASSES)
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title(f"Masks with ≥ {THIN_SHARE:.0%} of their pixels thinner than {THIN_PX} px: all (solid), "
             "large enough to judge (pale)", fontsize=10)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

N_THIN_SHOW = 2 if SMOKE else 4  # photos with the most thin pixels to show (N-014)
animals["thin_px"] = animals["thin"] * animals["mask_area"]
_most_thin = animals.sort_values(["thin_px", "id"], ascending=[False, True]).drop_duplicates("image_id").head(
    N_THIN_SHOW)
show_masks(images.set_index("id").loc[list(_most_thin["image_id"])].reset_index(), masks,
           f"the photos with the most thin pixels (dark = narrower than {THIN_PX} px)", mark_thin=True,
           cols=N_THIN_SHOW)

In [ ]:

# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>

<ul>
<li><strong>Mask size:</strong> Cats and dogs often occupy larger portions
of an image, while birds tend to have smaller masks.</li>

<li><strong>Fill ratio:</strong> A typical animal fills roughly half its
bounding box. Median fill ratios range from about 44% (birds) to
57% (cats). Shape, pose, and occlusion help explain these differences.
Animals with similar bounding boxes can therefore have very different
mask areas.</li>

<li><strong>Low-fill examples:</strong> Overlapping zebras and partly
occluded horses illustrate how the visible mask area can be much smaller
than the bounding box. Extreme values may also indicate annotation
errors, but are not necessarily incorrect.</li>

<li><strong>Thin structures:</strong> Birds have the highest proportion
of masks with thin regions (67%), but this drops to 14% when small masks
are excluded. Many birds are so small that much of the animal is only
a few pixels wide. For larger animals, such as horses, thin regions
often correspond to legs or other narrow body parts.</li>

<li><strong>Resizing:</strong> Thin structures may disappear when masks
are downsampled, especially with nearest-neighbor interpolation.
Both small objects and fine structures therefore require attention
during preprocessing.</li>
</ul>

<p><strong>Key insight:</strong> Mask geometry depends on both object
size and shape. Comparing all masks with sufficiently large masks
helps distinguish small-object effects from genuinely thin structures.</p>

</details>
"""))

### 🔴 Where Are the Animal Pixels?

Segmentation masks reveal not only how much of an image an animal occupies, but also where its pixels are located. By averaging masks across images, we can identify spatial patterns in the dataset.

**Before running the code:**
- Predict where animal pixels are most likely to occur in an image. Do you expect differences between classes?

**Explore:**
- Examine the heatmaps showing the average mask for each animal class. All masks are resized to a common grid before averaging.
- Compare the spatial distributions and the average proportion of animal pixels across classes.

**Observe and interpret:**
- Where are animal pixels concentrated, and how do the patterns differ between classes?
- What might explain these patterns?
- How representative are these spatial distributions of images captured by fixed cameras at the Wildwood Farm Park?

In [ ]:
# @title 🔎 Explore Spatial Distributions of Animal Pixels { display-mode: "form" }

# Mean mask per class on a common square: the share of the class's photos in which each spot is animal.
HEAT_RES = 24 if SMOKE else 48  # coarser grid with few photos (N-014)
_class_of = images.set_index("id")["class"]
heat = {c: np.zeros((HEAT_RES, HEAT_RES)) for c in CLASSES}
for image_id, rle in photo_rle.items():
    small = cv2.resize(decode(rle).astype(np.float32), (HEAT_RES, HEAT_RES), interpolation=cv2.INTER_AREA)
    heat[_class_of[image_id]] += small

fig, axes = plt.subplots(1, len(CLASSES), figsize=(2.3 * len(CLASSES), 2.8))
for ax, c in zip(axes, CLASSES, strict=True):
    n_photos = max(int((_class_of == c).sum()), 1)
    cmap = mpl.colors.LinearSegmentedColormap.from_list(c, ["white", CLASS_COLORS[c]])
    ax.imshow(heat[c] / n_photos, cmap=cmap, vmin=0, vmax=1, extent=[0, 1, 1, 0])
    ax.set_title(f"{c} · {heat[c].sum() / n_photos / HEAT_RES**2:.0%} of a photo", fontsize=9,
                 color=CLASS_COLORS[c])
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.grid(False)
fig.suptitle("Where the animal pixels are (same scale for every class: white = never, full colour = always)",
             color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:

# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display

display(HTML("""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px; color:#374151;">
<summary><strong>Sample Answer</strong></summary>
<ul>
  <li><strong>Observation:</strong> Animal pixels are concentrated near
      the image center across all classes. Cats and dogs occupy
      relatively large central regions, while birds have a much
      lower pixel frequency throughout the image.</li>
  <li><strong>Interpretation:</strong> These patterns reflect differences
      in object size, framing, and typical image composition.
      They indicate a spatial bias in the dataset.</li>
  <li><strong>Implication:</strong> Animals recorded by fixed cameras
      at the Wildwood Farm Park may appear anywhere in the frame.
      Models trained on centrally positioned animals could exploit
      this bias and generalize less reliably to such images.</li>
</ul>
<p><strong>Limitation:</strong> The heatmaps show how frequently each
pixel location belongs to an animal, not where individual animals
always appear. Averaging also hides variation between images.</p>
</details>
"""))

### 🔴 Mask Quality: Finding Annotation Errors

Segmentation masks can contain annotation errors that are difficult to identify by inspecting images individually. In this activity, we deliberately introduce five types of errors into copies of existing COCO masks. The modified annotations are stored in `park_masks`; the original `masks` remain unchanged.

Your challenge is to identify suspicious annotations using automated quality checks.

**Before running the code:**
- Examine five types of annotation errors and five automated quality checks.
- Predict which checks could detect each error. Can every error be detected reliably?

**Explore:**
- Compare the simulated errors in Picture A with the quality checks in Picture B.
- Predict which checks will flag each error type.
- Run the automated checks and inspect the strongest suspects.
- Reveal the answer key to evaluate your predictions.

**Observe and interpret:**
- Which checks detect which types of errors, and why?
- Which errors are difficult to detect automatically?
- Why might a valid COCO mask be flagged as suspicious?

In [ ]:
# @title Simulate Mask Annotation Errors { display-mode: "form" }

HUNT9_KINDS = ["another animal", "chunk missing", "shifted", "swollen", "crude polygon"]
SHIFT_PX = 15


def _paste(mask: np.ndarray, height: int, width: int) -> np.ndarray:
    """`mask` placed at the same coordinates on an empty canvas of another photo's size."""
    out = np.zeros((height, width), np.uint8)
    h, w = min(height, mask.shape[0]), min(width, mask.shape[1])
    out[:h, :w] = mask[:h, :w]
    return out


def _crude_polygon(mask: np.ndarray, max_vertices: int = 6) -> list[float]:
    """A polygon of at most `max_vertices` points around the mask (its convex hull, simplified)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(np.vstack(contours))
    eps = 0.02 * cv2.arcLength(hull, True)
    poly = cv2.approxPolyDP(hull, eps, True)
    while len(poly) > max_vertices:
        eps *= 1.5
        poly = cv2.approxPolyDP(hull, eps, True)
    return [float(v) for v in poly.reshape(-1)]


def plant_hunt9(animals_df: pd.DataFrame, table: pd.DataFrame,
                seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (park_masks, key): the masks of `animals_df` with one ugly mask of each of HUNT9_KINDS.

    Each kind spoils a different photo's mask, picked in a seeded order among masks large enough to judge,
    and the mask is changed through pycocotools (decode → OpenCV → encode). With no mask fitting a kind's
    condition (a small subset), the first free one is used, so five masks are always planted.
    """
    rng = random.Random(seed)
    size = table.set_index("id")[["width", "height"]]
    pool = animals_df[animals_df["mask_area"] >= MIN_JUDGED_AREA].set_index("id")
    order = sorted(pool.index)
    rng.shuffle(order)
    used, new_rle, key = set(), {}, []

    def fits(kind: str, ann_id: int):
        """The spoiled mask as a 0/1 array (or an RLE), plus a note, or None when this mask does not suit."""
        a = pool.loc[ann_id]
        x, y, w, h = a["bbox"]
        W, H = int(size.at[a["image_id"], "width"]), int(size.at[a["image_id"], "height"])
        if kind == "another animal":  # prefer a neighbour in the same photo: a duplicated mask, as in COCO
            others = pool[pool["image_id"] != a["image_id"]].index
            same = pool[(pool["image_id"] == a["image_id"]) & (pool.index != ann_id)].index
            for b in [*sorted(same), *[i for i in order if i in others]]:
                donor = _paste(decode(pool.at[b, "rle"]), H, W)
                if donor.any() and box_iou_dice(cv2.boundingRect(donor), tuple(a["bbox"]))[0] < 0.3:
                    where = "next to it" if b in same else "from another photo"
                    return donor, f"the mask of a {pool.at[b, 'class']} {where}"
            return None
        mask = decode(a["rle"])
        if kind == "chunk missing":
            dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
            r = float(dist.max())
            if r < max(15, 0.15 * a["mask_area"] ** 0.5):
                return None
            cy, cx = np.unravel_index(int(dist.argmax()), dist.shape)
            out = mask.copy()
            cv2.circle(out, (int(cx), int(cy)), int(0.8 * r), 0, -1)
            return out, f"a disc of radius {int(0.8 * r)} px cut out of the body"
        if kind == "shifted":
            if x + w + SHIFT_PX > W or y + h + SHIFT_PX > H or max(w, h) > 300:
                return None
            move = np.float32([[1, 0, SHIFT_PX], [0, 1, SHIFT_PX]])
            moved = cv2.warpAffine(mask, move, (W, H), flags=cv2.INTER_NEAREST)
            return moved, f"moved {SHIFT_PX} px right and down"
        if kind == "swollen":
            d = max(6, round(0.08 * (w * h) ** 0.5))
            if x < d or y < d or x + w + d > W or y + h + d > H:
                return None
            disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * d + 1, 2 * d + 1))
            return cv2.dilate(mask, disc), f"grown by {d} px all round"
        if kind == "crude polygon":
            if a["solidity"] > 0.85:  # pick an animal with clear dents, so the shortcut shows
                return None
            poly = _crude_polygon(mask)
            return to_rle([poly], H, W), f"{len(poly) // 2} clicked points"
        raise ValueError(kind)

    for kind in HUNT9_KINDS:
        free = [i for i in order if pool.at[i, "image_id"] not in used and i not in new_rle]
        spoiled = next(((i, out) for i in free if (out := fits(kind, i)) is not None), None)
        if spoiled is None:  # nothing fits the condition: give the first free mask another photo's mask
            i, j = free[0], free[1]
            height, width = decode(pool.at[i, "rle"]).shape
            pasted = _paste(decode(pool.at[j, "rle"]), height, width)
            spoiled = (i, (pasted, "another photo's mask (fallback)"))
        ann_id, (out, note) = spoiled
        new_rle[ann_id] = out if isinstance(out, dict) else encode(out)
        used.add(pool.at[ann_id, "image_id"])
        key.append({"kind": kind, "ann_id": int(ann_id), "image_id": int(pool.at[ann_id, "image_id"]),
                    "class": pool.at[ann_id, "class"], "how": note})
    park = animals_df[["id", "image_id", "class", "bbox", "iscrowd"]].copy()
    park["rle"] = [new_rle.get(i, r) for i, r in zip(animals_df["id"], animals_df["rle"], strict=True)]
    park["mask_area"] = [int(mask_utils.area(r)) for r in park["rle"]]
    return park, pd.DataFrame(key)


park_masks, HUNT9_KEY = plant_hunt9(animals, images, SEED)
print(f"park_masks: {len(park_masks)} animal masks on {park_masks['image_id'].nunique()} photos. "
      "Five of them are ugly. Find them.")

 **Five simulated annotation errors** The following figure illustrates five possible annotation errors using a toy horse mask. Grey represents the original mask and bounding box (dashed); yellow represents the modified annotation.

Look for differences in mask position, shape, coverage, and correspondence with the original horse.

In [ ]:

# @title 🔎 Explore Five Types of Mask Annotation Errors { display-mode: "form" }

def toy_horse(height: int = 60, width: int = 84) -> np.ndarray:
    """A toy horse mask with an oval body and four legs."""
    yy, xx = np.mgrid[0:height, 0:width]
    horse = ((xx - 40) / 24) ** 2 + ((yy - 32) / 13) ** 2 <= 1
    for lx in (24, 32, 46, 54):
        horse |= (xx >= lx) & (xx < lx + 5) & (yy >= 38) & (yy < 56)
    return horse.astype(np.uint8)


horse = toy_horse()
kx, ky, kw, kh = cv2.boundingRect(horse)
yy, xx = np.mgrid[0:horse.shape[0], 0:horse.shape[1]]

poly = np.array(_crude_polygon(horse), np.int32).reshape(-1, 1, 2)
bite = ((xx - 40) ** 2 + (yy - 30) ** 2 <= 36).astype(np.uint8)
other = (((xx - 66) / 13) ** 2 + ((yy - 18) / 9) ** 2 <= 1).astype(np.uint8)

kinds = {
    "Swollen\n(drawn too generously)": cv2.dilate(horse, np.ones((7, 7), np.uint8)),
    "Shifted\n(correct shape, moved)": np.roll(horse, (4, 9), axis=(0, 1)),
    "Chunk missing\n(a piece cut out)": horse & (1 - bite),
    "Wrong animal\n(another mask pasted)": other,
    "Crude polygon\n(a few quick clicks)": cv2.fillPoly(np.zeros_like(horse), [poly], 1),
}

GREY = "#9CA3AF"
YELLOW = "#FFDA00"

outline, _ = cv2.findContours(horse, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

fig, axes = plt.subplots(1, 5, figsize=(13, 2.7))

for ax, (kind, spoiled) in zip(axes, kinds.items(), strict=True):
    # Original mask in grey
    base = np.zeros((*horse.shape, 4))
    base[horse > 0] = (*mpl.colors.to_rgb(GREY), 0.35)
    ax.imshow(base)

    # Simulated annotation in yellow
    overlay = np.zeros((*horse.shape, 4))
    overlay[spoiled > 0] = (*mpl.colors.to_rgb(YELLOW), 0.75)
    ax.imshow(overlay)

    # Original outline
    for contour in outline:
        ax.plot(
            contour[:, 0, 0], contour[:, 0, 1],
            color=GREY, lw=1.5
        )

    # Original bounding box
    ax.add_patch(mpl.patches.Rectangle(
        (kx - 0.5, ky - 0.5), kw, kh,
        fill=False, lw=1.5, ls="--", edgecolor=GREY
    ))

    ax.set_xlim(-0.5, horse.shape[1] - 0.5)
    ax.set_ylim(horse.shape[0] - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(kind, fontsize=9)

fig.suptitle(
    "Five simulated annotation errors",
    color=INK
)
fig.tight_layout()
display(fig)
plt.close(fig)

**Five automated mask quality checks.** The next figure compares a valid toy horse mask (top row) with a deliberately modified mask (bottom row). Each check measures a different property and flags potentially suspicious annotations.

| Check | What it measures | Flagged when |
|---|---|---|
| **Mask–box agreement** | IOU between the annotated box and the tightest box around the mask | IoU <escape><</escape> 0.90 |
| **Fill ratio** | Mask area divided by bounding box area | Outside 0.15–0.95 |
| **Connected components** | Number of separate mask regions | More than 3 |
| **Holes** | Enclosed background regions within the mask | At least 1 |
| **Solidity** | Mask area divided by its convex hull area | Above 0.97 |

A *convex hull* is the smallest convex shape enclosing the mask, similar to stretching a rubber band around it.

These thresholds are heuristics: a flagged mask is worth inspecting, but is not necessarily incorrect.


In [ ]:

# @title 🔎 Explore Five Mask Quality Checks { display-mode: "form" }

# Compare an original toy horse with a deliberately modified version.
H, W = 60, 84
yy, xx = np.mgrid[0:H, 0:W]

original = toy_horse(H, W)
hole = (xx - 40) ** 2 + (yy - 28) ** 2 <= 16
piece = (xx >= 68) & (xx < 76) & (yy >= 30) & (yy < 38)
modified = ((original.astype(bool) | piece) & ~hole).astype(np.uint8)

tx, ty, tw, th = cv2.boundingRect(modified)
rows = {
    "Original": (original, cv2.boundingRect(original)),
    "Modified": (modified, (tx + 5, ty + 3, tw, th))
}

CHECK_RULES = [
    ("1 · Mask–box agreement", "suspect below 0.9", "box_iou",
     lambda v: v < 0.9, "IoU {:.2f}".format),
    ("2 · Fill", "suspect outside 0.15–0.95", "fill",
     lambda v: not 0.15 <= v <= 0.95, "fill {:.2f}".format),
    ("3 · Pieces", "suspect above 3", "pieces",
     lambda v: v > 3, lambda v: f"{v} piece" + "s" * (v != 1)),
    ("4 · Holes", "suspect with any hole", "holes",
     lambda v: v > 0, lambda v: f"{v} hole" + "s" * (v != 1)),
    ("5 · Solidity", "suspect above 0.97", "solidity",
     lambda v: v > 0.97, "solidity {:.2f}".format),
]

UNDER = {
    0: "solid: tightest box around mask\ndashed: annotated box",
    4: "dashed: convex hull"
}

GREY = "#9CA3AF"
DARK_GREY = "#4B5563"
YELLOW = "#FFDA00"

def mask_layer(mask, color, alpha):
    layer = np.zeros((H, W, 4))
    layer[mask > 0] = (*mpl.colors.to_rgb(color), alpha)
    return layer

fig, axes = plt.subplots(2, 5, figsize=(13, 6.2))

for r, (row_name, (mask, box)) in enumerate(rows.items()):
    shape = mask_shape(mask, box)
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    hull = cv2.convexHull(np.vstack(contours))[:, 0, :]

    n_parts, parts = cv2.connectedComponents(mask, connectivity=8)
    main = max(range(1, n_parts), key=lambda k: int((parts == k).sum()))
    own = cv2.boundingRect(mask)

    for k, (name, rule, key, is_suspect, fmt) in enumerate(CHECK_RULES):
        ax = axes[r, k]

        # Original reference in grey, evaluated mask in yellow
        ax.imshow(mask_layer(original, GREY, 0.35))
        ax.imshow(mask_layer(mask, YELLOW, 0.8))

        if k == 0:
            ax.add_patch(mpl.patches.Rectangle(
                (own[0] - 0.5, own[1] - 0.5), own[2], own[3],
                fill=False, lw=2, edgecolor=DARK_GREY
            ))

        if k in (0, 1):
            ax.add_patch(mpl.patches.Rectangle(
                (box[0] - 0.5, box[1] - 0.5), box[2], box[3],
                fill=False, lw=2, ls="--", edgecolor=GREY
            ))

        if k == 2:
            ax.imshow(mask_layer(
                (parts > 0) & (parts != main), DARK_GREY, 1.0
            ))

        if k == 3 and row_name == "Modified":
            ax.imshow(mask_layer(hole, DARK_GREY, 1.0))

        if k == 4:
            ax.add_patch(mpl.patches.Polygon(
                hull, closed=True, fill=False, lw=2,
                ls="--", edgecolor=DARK_GREY
            ))

        value = shape[key]
        bad = is_suspect(value)

        ax.text(
            0.02, 0.98,
            "SUSPECT" if bad else "normal",
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=13, fontweight="bold",
            color=DARK_GREY
        )

        ax.text(
            0.98, 0.97, fmt(value),
            transform=ax.transAxes,
            ha="right", va="top",
            fontsize=9, color=DARK_GREY
        )

        if r == 0:
            ax.set_title(name, fontsize=12, color=INK, pad=10)
        else:
            under = rule + (f"\n\n{UNDER[k]}" if k in UNDER else "")
            ax.set_xlabel(under, fontsize=9, color=INK)

        ax.set_xlim(-0.5, W - 0.5)
        ax.set_ylim(H + 3.5, -14.5)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

    axes[r, 0].set_ylabel(row_name, fontsize=12, color=INK)

fig.suptitle(
    "Five mask quality checks: original (top) vs. modified (bottom)",
    color=INK, fontsize=13
)
fig.tight_layout()
display(fig)
plt.close(fig)

**Predict and investigate**

Match each of the five annotation errors in Picture A to the quality check(s) in Picture B that you expect to flag it. Write five pairs in the form *error type → check number(s)*. An error may trigger multiple checks, and some checks may detect none.

Then run the automated investigation:

1. **Counts:** Compare how many masks each check flags.
2. **Suspect gallery:** Inspect the highlighted mask in each image. Faded masks represent other animals. Predict whether each suspect is a simulated error or an original COCO annotation.
3. **Answer key:** Reveal the planted errors and compare the results with your predictions.

In [ ]:
# @title 🔎 Detect Suspicious Segmentation Masks { display-mode: "form" }

BOX_IOU_MIN = 0.9  # check 1: suspect when the mask's own box overlaps the annotated box less than this
FILL_RANGE = (0.15, 0.95)  # suspect when mask area / box area is outside this range
MAX_PIECES = 3  # suspect with more separate pieces than this
MAX_HOLES = 0  # suspect with more holes than this
SOLIDITY_MAX = 0.97  # suspect when the mask fills more than this share of its rubber band (no dents at all)

_judge = park_masks[park_masks["mask_area"] >= MIN_JUDGED_AREA].reset_index(drop=True)
park_shape = pd.concat([_judge, pd.DataFrame([mask_shape(decode(r), b)
                                              for r, b in zip(_judge["rle"], _judge["bbox"], strict=True)])],
                       axis=1)


def run_checks(shape: pd.DataFrame, box_iou_min: float) -> dict[str, pd.Series]:
    """The five checks as {name: True where a mask is a suspect}; only check 1's threshold is a parameter.

    e.g. run_checks(park_shape, 0.9)["holes"] → a True/False Series, one value per judged mask
    """
    return {
        "box IoU": shape["box_iou"] < box_iou_min,
        "fill": (shape["fill"] < FILL_RANGE[0]) | (shape["fill"] > FILL_RANGE[1]),
        "pieces": shape["pieces"] > MAX_PIECES,
        "holes": shape["holes"] > MAX_HOLES,
        "solidity": shape["solidity"] > SOLIDITY_MAX,
    }


CHECKS = run_checks(park_shape, BOX_IOU_MIN)
park_shape["checks"] = [", ".join(name for name, hit in CHECKS.items() if hit.iloc[i])
                        for i in range(len(park_shape))]
mask_suspects = park_shape[park_shape["checks"] != ""].copy()
mask_suspects["n_checks"] = mask_suspects["checks"].str.count(",") + 1
mask_suspects = mask_suspects.sort_values(["n_checks", "box_iou", "id"], ascending=[False, True, True])

print(f"{len(_judge)} masks judged · {len(mask_suspects)} suspects "
      f"({len(mask_suspects) / max(len(_judge), 1):.1%}). Suspects per check:")
for name, hit in CHECKS.items():
    print(f"  {name:9s} {int(hit.sum())}")
_cols = ["id", "image_id", "class", "checks", "box_iou", "fill", "pieces", "holes", "solidity"]
display(mask_suspects[_cols].head(12).round(2).set_index("id"))

N_SUSPECTS_SHOW = 4 if SMOKE else 8  # suspect photos to show (N-014)
_show = mask_suspects.drop_duplicates("image_id").head(N_SUSPECTS_SHOW).set_index("image_id")


def _suspect_caption(r) -> str:
    s = _show.loc[r["id"]]
    return (f"photo {r['id']} · {s['class']} · {s['checks']}\n"
            f"box IoU {s['box_iou']:.2f} · fill {s['fill']:.2f}")


show_masks(images.set_index("id").loc[list(_show.index)].reset_index(), park_masks,
           f"the {len(_show)} strongest suspects: bright mask + solid box = suspect, faint = other animals",
           caption=_suspect_caption, highlight=dict(zip(_show.index, _show["id"], strict=True)))

In [ ]:

# @title 💡 Sample Answer { display-mode: "form" }

from IPython.display import HTML, display
from html import escape
import io
import base64

# Identify which planted errors were detected.
_checks_by_id = park_shape.set_index("id")["checks"]

HUNT9_KEY["checks"] = HUNT9_KEY["ann_id"].map(_checks_by_id).fillna("")
HUNT9_KEY["found"] = HUNT9_KEY["checks"] != ""

hunt9_found = int(HUNT9_KEY["found"].sum())

RESULTS["hunt9"] = {
    "planted": len(HUNT9_KEY),
    "found": hunt9_found,
}
RESULTS["mask_suspects_real"] = int(
    (~mask_suspects["id"].isin(HUNT9_KEY["ann_id"])).sum()
)

# Prepare the detection summary.
lines = [
    f"{'✓' if k['found'] else '✗'} {k['kind']}: "
    f"{k['checks'] or 'no check fired'}"
    for _, k in HUNT9_KEY.iterrows()
]

lines.append(
    f"Detected {hunt9_found} of {len(HUNT9_KEY)} planted errors; "
    f"{RESULTS['mask_suspects_real']} original COCO annotations "
    "were also flagged."
)

# Compare original and modified annotations.
GREY = "#9CA3AF"
YELLOW = "#FFDA00"

_by_image = images.set_index("id")

fig, axes = plt.subplots(
    2,
    len(HUNT9_KEY),
    figsize=(3.2 * len(HUNT9_KEY), 6.2),
    squeeze=False,
)

for col, (_, k) in enumerate(HUNT9_KEY.iterrows()):
    photo = _by_image.loc[k["image_id"]]
    thumb = load_thumb(photo["path"], (320, 320))

    sx = thumb.width / photo["width"]
    sy = thumb.height / photo["height"]

    for row, source in enumerate([masks, park_masks]):
        ax = axes[row, col]
        color = GREY if row == 0 else YELLOW

        annotation = source.loc[
            source["id"] == k["ann_id"]
        ].iloc[0]

        full = decode(annotation["rle"])
        small = cv2.resize(
            full,
            thumb.size,
            interpolation=cv2.INTER_NEAREST,
        ) > 0

        overlay = np.zeros((thumb.height, thumb.width, 4))
        overlay[small] = (*mpl.colors.to_rgb(color), 0.6)

        ax.imshow(thumb)
        ax.imshow(overlay)

        x, y, w, h = annotation["bbox"]
        ax.add_patch(
            mpl.patches.Rectangle(
                (x * sx, y * sy),
                w * sx,
                h * sy,
                fill=False,
                lw=1.5,
                ls="--",
                edgecolor=color,
            )
        )

        frame(ax, color)

        ax.set_title(
            f"{k['class']} · original"
            if row == 0
            else f"{k['kind']}\n{k['checks'] or 'no check fired'}",
            fontsize=9,
            color=INK,
        )

fig.suptitle(
    "Five planted errors: original masks (top) vs. modified masks (bottom)",
    color=INK,
    fontsize=13,
)
fig.tight_layout()

# Convert the figure to an image without displaying it separately.
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=130, bbox_inches="tight")
plt.close(fig)

img = base64.b64encode(buf.getvalue()).decode("utf-8")
items = "".join(f"<li>{escape(line)}</li>" for line in lines)

# Display the complete answer inside one collapsible grey box.
display(HTML(f"""
<details style="background:#F5F5F5; border:1px solid #9CA3AF;
                border-radius:6px; padding:12px 16px;
                color:#374151; line-height:1.5;">

<summary style="cursor:pointer;">
  <strong>Sample Answer</strong>
</summary>

<p><strong>Which errors were detected?</strong></p>
<ul>{items}</ul>

<img src="data:image/png;base64,{img}"
     style="width:100%; height:auto; margin:12px 0;">

<p><strong>Why did these checks detect the errors?</strong></p>
<ul>
  <li><strong>Wrong animal → box IoU:</strong>
      The pasted mask belongs to a different animal and no longer
      aligns with the original bounding box.</li>
  <li><strong>Shifted mask → box IoU:</strong>
      Moving the mask reduces its overlap with the annotated box.</li>
  <li><strong>Swollen mask → box IoU, possibly fill:</strong>
      Enlarging the mask changes its bounding box and may increase
      its fill ratio.</li>
  <li><strong>Missing region → holes:</strong>
      The simulated cut-out creates an enclosed background region
      inside the mask.</li>
  <li><strong>Crude polygon → solidity:</strong>
      Simplifying the shape removes concavities, making the mask
      more similar to its convex hull.</li>
</ul>

<p><strong>What about the other masks?</strong></p>
<ul>
  <li>The connected-components check may detect none of the planted
      errors, but can flag animals whose visible regions are separated
      by occlusion.</li>
  <li>Some flagged original COCO masks are valid but unusual,
      for example because of occlusion or complex shapes.</li>
  <li>Others may contain genuine annotation errors, such as inaccurate
      boundaries or missing regions. Not every flagged original mask
      is therefore a false alarm.</li>
  <li>Some incorrect masks may pass all checks. Detection results
      also depend on the thresholds used.</li>
</ul>

<p><strong>Key insight:</strong> Automated quality checks identify
potential geometric inconsistencies, not necessarily incorrect
annotations. Combining checks helps prioritize visual inspection,
but cannot replace it.</p>

</details>
"""))

## Continue here (SUS)

## 10. Your project dataset

In [ ]:
# @title Checklist Dataset Mini-Project
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · What makes a good dataset</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Does it fit the question?</strong> The images, the classes and the setting must match the job you want the
  model to do.<br><em>Park example:</em> the alert needs a dog seen from a fixed camera in the sheep paddock, not a
  dog in a holiday snapshot taken from two metres away.</li>
<li style="margin:2px 0;"><strong>Enough samples</strong>, <strong>2–10 classes</strong> and <strong>about 500×500 px</strong>: the numbers to check, explained below.</li>
<li style="margin:2px 0;"><strong>Classes, distributions, imbalance, bias.</strong> Know them, and take them into account in training (for
  example with a weighted loss).<br>
  <em>Park example:</em> if 70 % of the photos are sheep and 3 % are dogs, the model sees few dogs to learn from,
  and the dog is the animal the alert exists for (section 4).</li>
<li style="margin:2px 0;"><strong>Variability matches the target population</strong>, and <strong>labels are present and validated</strong>: every image has
  a label, and someone checked a sample of them.<br>
  <em>Park example:</em> a photo of a zebra labelled "horse" teaches the model the wrong thing, and nobody notices
  until the zebra headcount is off.</li>
<li style="margin:2px 0;"><strong>Objects are identifiable and separable by eye.</strong></li>
<li style="margin:2px 0;"><strong>What do experts and the literature say</strong> about this data and this task?</li>
<li style="margin:2px 0;"><strong>Get a qualitative and a quantitative sense of the data</strong> (Karpathy, 2019): look at many images, then
  measure.</li>
</ul>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">The reasons and park examples behind each point</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Enough samples:</strong> a few hundred to a few thousand images, depending on how many classes you have.
  <em>Park example:</em> 100 photos per animal (600 in all) is a fair start; 12 zebra photos is not.</li>
<li style="margin:2px 0;"><strong>2–10 classes:</strong> more classes need more photos each, and similar classes get confused.
  <em>Park example:</em> six animals is fine; splitting sheep and horses into 40 breeds would not be.</li>
<li style="margin:2px 0;"><strong>About 500×500 px:</strong> check the input size of the models you plan to use.
  <em>Park example:</em> a bird across the pond may cover only 20×20 px of a 640×480 photo; shrink the photo to
  224×224 and it is a handful of pixels.</li>
<li style="margin:2px 0;"><strong>Variability:</strong> lighting, viewpoints, backgrounds and seasons should look like the ones
  the model will meet.
  <em>Park example:</em> take pictures of the park in winter and in summer, in sun and at dusk. A model that only
  saw sunny July afternoons will meet snow and low light in January.</li>
<li style="margin:2px 0;"><strong>Separable by eye:</strong> if you cannot tell them apart, the model will struggle too.
  <em>Park example:</em> a bird that is a dark speck over the pond, or sheep standing so close that their outlines
  merge into one woolly shape.</li>
<li style="margin:2px 0;"><strong>Experts and literature:</strong> before trusting COCO's masks, read the audit that found some of them poor; it
  tells you what to look for.</li>
<li style="margin:2px 0;"><strong>Qualitative and quantitative:</strong> look at many images and see how big the animals are and how much of the
  photo they cover. Is it the same for each class? Can you already see imbalances just by looking?</li>
</ul>
</details>
</div>""")

In [ ]:
# @title 🔗 Resources: Image Datasets and Annotation Tools { display-mode: "form" }

from IPython.display import HTML, display

display(HTML(r"""
<div style="background:#F3F4F6;color:#4B5563;
            border-left:6px solid #4B5563;border-radius:6px;
            padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<div style="font-weight:bold;">🔗 Resources: Image Datasets and Annotation Tools</div>

<p style="margin:6px 0 0 0;">
Where to find existing image datasets and tools for annotating your own images.
</p>

<p style="margin:10px 0 0 0;"><strong>Find a dataset</strong></p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><a href="https://www.kaggle.com/datasets" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Kaggle datasets</a>: a huge community collection, good for a first look.</li>
<li style="margin:2px 0;"><a href="https://datasetsearch.research.google.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Google Dataset Search</a>: one search box over many repositories.</li>
<li style="margin:2px 0;"><a href="https://www.v7labs.com/open-datasets" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">V7 open datasets</a>: a curated list of open computer-vision data
  (plus <a href="https://www.v7labs.com/blog" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">their blog</a> with overviews of well-known datasets).</li>
<li style="margin:2px 0;"><a href="https://universe.roboflow.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Roboflow Universe</a>: community datasets, many already labeled with boxes or masks.</li>
<li style="margin:2px 0;"><a href="https://lila.science" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">LILA</a>: wildlife and camera-trap images, the closest match to the park's cameras.</li>
<li style="margin:2px 0;"><a href="https://pytorch.org/vision/stable/datasets.html" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">torchvision datasets</a>: classic datasets available directly through PyTorch.</li>
<li style="margin:2px 0;"><a href="https://cocodataset.org" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">COCO</a>: a general-purpose dataset; our play dataset is a subset of it.</li>
</ul>

<p style="margin:10px 0 0 0;"><strong>Label your own images</strong></p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><a href="https://supervisely.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Supervisely</a>: a labeling platform for boxes, polygons, and masks.</li>
<li style="margin:2px 0;"><a href="https://www.v7labs.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">V7</a>: browser-based image annotation tools.</li>
<li style="margin:2px 0;"><a href="https://roboflow.com/annotate" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Roboflow Annotate</a>: browser-based labeling with export to common formats.</li>
<li style="margin:2px 0;"><a href="https://labelstud.io" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Label Studio</a>: open-source annotation software that can run locally.</li>
</ul>

</div>
"""))

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Run the checklist of sections 3 to 6 on your own folder-per-class dataset, and answer the guiding questions
for it the way we answered them for the park.</p>
</div>""")

Your project has its own dataset, and it deserves the same twenty minutes of looking before anybody trains
on it. This section runs the checklist on **any folder-per-class dataset**: one sub-folder per class, the
images inside. The sub-folder names become the class names.

```
my_dataset/                ← MY_DATASET_DIR points here (or at my_dataset.zip)
├── daisy/      img_001.jpg, img_002.png, …
├── rose/       …
└── tulip/      …
```

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · Point the notebook at your dataset (you change one line)</div>
<p style="margin:6px 0 0 0;">Colab cannot see your laptop's folders, so the dataset first goes where Colab can read it. Pick <strong>one</strong> of
the three options, then do the last step.</p>
<p style="margin:6px 0 0 0;"><strong>Option A · Upload it to this Colab session</strong> (fastest; gone when the session ends)</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Zip the dataset folder</strong> on your laptop: Mac, right-click the folder → <em>Compress</em>; Windows,
   right-click → <em>Send to</em> → <em>Compressed (zipped) folder</em>. Colab uploads files, not folders.</li>
<li style="margin:2px 0;">Click the 📁 <strong>Files</strong> icon in Colab's left bar and <strong>drag the zip into it</strong> (or use the upload icon at
   the top of the pane). Wait until the progress circle at the bottom left disappears.</li>
<li style="margin:2px 0;"><strong>Right-click the zip → Copy path.</strong> It looks like <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/my_dataset.zip</code>; the notebook unzips it.</li>
</ol>
<p style="margin:6px 0 0 0;">⚠️ Colab deletes uploaded files when the session ends (closing the tab for a while, or <em>Disconnect and
delete runtime</em>): next time, upload the zip again.</p>
<p style="margin:6px 0 0 0;"><strong>Option B · Use Google Drive</strong> (stays between sessions; slower the first time)</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Put the dataset on your Drive</strong> at <a href="https://drive.google.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">drive.google.com</a>: <em>New → Folder upload</em>
   for the folder, or <em>File upload</em> for a zip.</li>
<li style="margin:2px 0;"><strong>Connect Drive to Colab:</strong> in the 📁 Files pane, click the <strong>Mount Drive</strong> icon (a folder with the
   Drive logo) and allow access in the window that opens. A <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">drive/</code> folder appears in the pane.</li>
<li style="margin:2px 0;">Open <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">drive/MyDrive/</code>, find the folder whose sub-folders are your classes (or the zip), <strong>right-click →
   Copy path.</strong> It looks like <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/drive/MyDrive/project/my_dataset</code>.</li>
</ol>
<p style="margin:6px 0 0 0;"><strong>Option C · Paste a Google Drive link to a zip</strong> (no mounting, no browsing)</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">In Drive, right-click the <strong>zip</strong> → <em>Share</em> → <em>Copy link</em>. It looks like
   <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">https://drive.google.com/file/d/1I9D…/view?usp=drive_link</code>.</li>
<li style="margin:2px 0;">Paste the whole link as the path. If the zip is shared as <em>Anyone with the link</em>, it downloads straight
   away; if it is private, Colab asks you to sign in with the Google account that can open it.</li>
</ol>
<p style="margin:6px 0 0 0;">Links to a Drive <em>folder</em> do not work: zip the folder first, or use option B.</p>
<p style="margin:6px 0 0 0;"><strong>Last step, all options:</strong> paste the path between the quotes in the next cell, click into that cell, and
run everything from there: <strong>Runtime → Run cell and below</strong>. If the path is not right, the cell after it
stops with a plain message that says what to change.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">✅ <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/my_dataset.zip</code> (A), <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/drive/MyDrive/project/my_dataset</code> (B) or
  <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">https://drive.google.com/file/d/…/view</code> (C): the folder that holds <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">daisy/</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">rose/</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">tulip/</code>, or a zip
  of it.</li>
<li style="margin:2px 0;">❌ <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/my_dataset/daisy</code>: one class folder, one level too deep.</li>
<li style="margin:2px 0;">❌ a folder that holds <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">train/</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">val/</code>: point at <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/train</code> instead (then at <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/val</code>, if you like).</li>
</ul>
<p style="margin:6px 0 0 0;">Leave it at <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">None</code> to see the whole section on a small park sample first.</p>
</div>""")

In [ ]:
# ✏️ YOUR TURN
MY_DATASET_DIR = None  # paste your path between quotes, e.g. "/content/my_dataset.zip"

In [ ]:
# @title Dataset reader: finds your class folders, or says exactly what to change { display-mode: "form" }
import re
import urllib.error
import urllib.request
import zipfile

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
SPLIT_NAMES = {"train", "training", "val", "valid", "validation", "test", "testing", "dev"}
MY_MAX_IMAGES = 2000  # larger datasets are sampled down to this many photos, stratified by class


def _visible(folder: Path) -> list[Path]:
    """The entries of `folder` that are not hidden or system files (.DS_Store, __MACOSX, …)."""
    return sorted(f for f in folder.iterdir() if not f.name.startswith((".", "_")))


def _is_image(f: Path) -> bool:
    return f.is_file() and f.suffix.lower() in IMAGE_SUFFIXES


def _contents(folder: Path, limit: int = 8) -> str:
    """One line per entry: 'name/ (12 images)' for folders, the name for files."""
    lines = []
    for f in _visible(folder)[:limit]:
        n = sum(_is_image(g) for g in f.rglob("*")) if f.is_dir() else None
        lines.append(f"    {f.name}/  ({n} images)" if f.is_dir() else f"    {f.name}")
    more = len(_visible(folder)) - limit
    return "\n".join(lines + ([f"    … and {more} more"] if more > 0 else [])) or "    (empty)"


def drive_file_id(url: str) -> str:
    """The file id in a Google Drive share link; stops with advice for a folder link or another site.

    e.g. drive_file_id("https://drive.google.com/file/d/1I9De…/view?usp=drive_link") → "1I9De…"
    """
    if "drive.google.com/drive/folders" in url:
        raise SystemExit("❌ MY_DATASET_DIR: this is a link to a Drive *folder*. Zip the folder, upload the "
                         "zip to Drive and share the zip's link, or use option B (Mount Drive, Copy path).")
    found = re.search(r"/file/d/([\w-]+)", url) or re.search(r"[?&]id=([\w-]+)", url)
    if "drive.google.com" not in url or not found:
        raise SystemExit("❌ MY_DATASET_DIR: links work only for a Google Drive file, like "
                         "https://drive.google.com/file/d/<id>/view. For anything else, download it and use "
                         "option A.")
    return found[1]


def _drive_public_download(file_id: str, dest: Path) -> bool:
    """Download a file shared as 'Anyone with the link'. False if Drive answers with a page, not the file."""
    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            first = r.read(4)
            if first[:2] != b"PK":  # not a zip: the sign-in page (private file) or a 'not found' page
                return False
            part = dest.with_suffix(".part")  # renamed only when complete, so a cut download is not reused
            with open(part, "wb") as f:
                f.write(first)
                while chunk := r.read(1 << 20):
                    f.write(chunk)
                    if f.tell() % (25 << 20) < (1 << 20):
                        print(f"\r  downloading {dest.name}: {f.tell() / 1e6:.0f} MB", end="", flush=True)
            part.rename(dest)
        print(f"\r  downloaded {dest.name}: {dest.stat().st_size / 1e6:.0f} MB")
        return True
    except urllib.error.HTTPError:  # 404: a wrong id, or a file that is gone
        return False
    except OSError as err:  # the connection dropped: say so, and start over next time
        dest.with_suffix(".part").unlink(missing_ok=True)
        raise SystemExit(f"❌ the download from Drive stopped halfway ({err}). Run the cell again: it starts "
                         "over.") from None


def _drive_signed_in_download(file_id: str, folder: Path) -> Path:
    """Colab only: sign in with Google and download a private file through the Drive API."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    print("ℹ️ the file is not shared publicly: sign in with the Google account that can open it.")
    auth.authenticate_user()
    drive = build("drive", "v3")
    meta = drive.files().get(fileId=file_id, fields="name,mimeType", supportsAllDrives=True).execute()
    if meta["mimeType"] == "application/vnd.google-apps.folder":
        drive_file_id("drive.google.com/drive/folders/")  # stops with the folder advice
    dest = folder / meta["name"]
    part = dest.with_suffix(".part")  # renamed only when complete, so a cut download is not reused
    with open(part, "wb") as f:
        download = MediaIoBaseDownload(f, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        done = False
        while not done:
            status, done = download.next_chunk()
            print(f"\r  downloading {dest.name}: {status.progress():.0%}", end="", flush=True)
    part.rename(dest)
    print()
    return dest


def fetch_drive_link(url: str) -> Path:
    """Download the zip behind a Google Drive share link (once per session) and return its local path."""
    file_id = drive_file_id(url)
    folder = DATA_DIR / "cache" / "my_upload" / f"drive_{file_id}"
    done = sorted(folder.glob("*.zip")) if folder.exists() else []
    if done:
        return done[0].resolve()
    folder.mkdir(parents=True, exist_ok=True)
    if _drive_public_download(file_id, folder / "dataset.zip"):
        return (folder / "dataset.zip").resolve()
    if IN_COLAB:
        return _drive_signed_in_download(file_id, folder).resolve()
    raise SystemExit("❌ MY_DATASET_DIR: Drive did not hand out this file without a sign-in, so it is "
                     "private (or the link is wrong). In Colab the notebook asks you to sign in; elsewhere, "
                     "open the file in Drive → Share → General access → 'Anyone with the link'.")


def find_dataset(path) -> Path:
    """The folder that holds the class folders. Stops with a plain message saying what to change if the
    path cannot be used; unzips a .zip and steps into a lone wrapper folder on its own.

    e.g. find_dataset("/content/flowers.zip") → Path("data/cache/my_upload/flowers/flowers")
    """
    text = str(path).strip().strip("'\"")
    p = fetch_drive_link(text) if text.startswith(("http://", "https://")) else Path(text).expanduser()
    if str(p).startswith("/content/drive") and IN_COLAB and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive  # Step 1, Drive route: connect it on first use

        drive.mount("/content/drive")
    if p.is_file() and p.suffix.lower() == ".zip":
        dest = (DATA_DIR / "cache" / "my_upload" / p.stem).resolve()  # absolute, so messages are pastable
        if not dest.exists():
            with zipfile.ZipFile(p) as z:
                z.extractall(dest)
        print(f"ℹ️ unzipped {p.name} into {dest}")
        p = dest
    if not p.exists():
        parent = next(q for q in p.parents if q.exists())
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} does not exist (a typo, or the upload is not there).\n"
                         f"   The nearest folder that exists is {parent}, which holds:\n{_contents(parent)}\n"
                         "   Copy the path from the Files pane: right-click the folder → Copy path.")
    if p.is_file():
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} is a file. Point at the folder that holds your class "
                         f"folders, or at a .zip. It sits in {p.parent}, which holds:\n{_contents(p.parent)}")
    while True:  # a zip often wraps everything in one more folder: step into it
        subdirs = [d for d in _visible(p) if d.is_dir()]
        loose = [f for f in _visible(p) if _is_image(f)]
        if len(subdirs) == 1 and not loose and any(d.is_dir() for d in _visible(subdirs[0])):
            print(f"ℹ️ {p.name}/ holds only the folder {subdirs[0].name}/: using that.")
            p = subdirs[0]
            continue
        break
    names = {d.name.lower() for d in subdirs}
    if subdirs and names <= SPLIT_NAMES:
        first = p / sorted(d.name for d in subdirs)[0]
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} holds split folders ({', '.join(sorted(names))}), "
                         f"not class folders. Point at one of them, e.g. {first}, and run the section "
                         "again for the other.")
    if not subdirs and loose:
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} holds {len(loose)} images but no class folders. If this "
                         f"is one class folder, point one level up, at {p.parent}. If your project has no "
                         "classes (detection, segmentation), move the images into one folder inside, e.g. "
                         f"{p / 'all'}.")
    if not any(_is_image(f) for f in p.rglob("*")):
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} holds no image files "
                         f"({', '.join(sorted(IMAGE_SUFFIXES))}). "
                         f"It holds:\n{_contents(p)}")
    return p


def list_folder_per_class(root) -> pd.DataFrame:
    """One row per image file under `root`: columns path, file, class (the top-level folder it sits in).

    e.g. a row: {"path": "/content/flowers/rose/img_7.jpg", "file": "rose/img_7.jpg", "class": "rose"}
    """
    rows = []
    for class_dir in (d for d in _visible(Path(root)) if d.is_dir()):
        for f in sorted(class_dir.rglob("*")):
            if _is_image(f) and not f.name.startswith("."):
                rows.append({"path": str(f), "file": str(f.relative_to(root)), "class": class_dir.name})
    return pd.DataFrame(rows, columns=["path", "file", "class"])


def folder_notes(root: Path, files: pd.DataFrame) -> list[str]:
    """Warnings about what the reader skipped or what makes the checks weak (one line each)."""
    root = Path(root)
    notes = []
    loose = [f.name for f in _visible(root) if _is_image(f)]
    if loose:
        notes.append(f"⚠️ {len(loose)} images sit directly in {root.name}/, outside any class folder: "
                     "skipped.")
    others = [f for d in _visible(root) if d.is_dir() for f in d.rglob("*")
              if f.is_file() and not _is_image(f) and not f.name.startswith(".")]
    skipped = Counter(f.suffix.lower() or "(no extension)" for f in others)
    if skipped:
        notes.append("⚠️ skipped files that are not images: "
                     + ", ".join(f"{n} × {ext}" for ext, n in skipped.most_common()))
    empty = [d.name for d in _visible(root) if d.is_dir() and d.name not in set(files["class"])]
    if empty:
        notes.append(f"⚠️ class folders with no images: {', '.join(empty)}.")
    if files["class"].nunique() == 1:
        notes.append("⚠️ only one class folder: check 1 has nothing to compare (fine for a detection "
                     "project).")
    return notes

The next cell reads your folder (or, with `None`, builds the park sample: 20 cleaned photos per class plus
**three planted problems**, an exact copy of a cat, a mirrored dog and a grayscale bird). It prints what it
found and shows three images per class: **check that these are your images and your class names** before
reading anything else.

In [ ]:
# @title Read your folder (or build the sample), and show what was found { display-mode: "form" }
MY_SAMPLE_DIR = DATA_DIR / "cache" / "my_dataset"
MY_SAMPLE_PER_CLASS = 4 if SMOKE else 20  # photos per class in the sample (N-014: fewer in smoke mode)


def build_my_sample(dest: Path, per_class: int, seed: int) -> Path:
    """Folder-per-class copy of a few clean Play Dataset photos, plus an exact copy, a mirror, a grayscale."""
    shutil.rmtree(dest, ignore_errors=True)  # rebuilt from scratch: re-running gives the same sample (N-013)
    rng = random.Random(seed)
    pool = clean_images[clean_images["flags"] == ""]
    picked = {}
    for c in CLASSES:
        of_class = sorted(pool.loc[pool["class"] == c, "path"])
        picked[c] = rng.sample(of_class, min(per_class, len(of_class)))
        (dest / c).mkdir(parents=True, exist_ok=True)
        for p in picked[c]:
            shutil.copyfile(p, dest / c / Path(p).name)
    first = {c: Path(picked[c][0]) for c in CLASSES if picked[c]}
    shutil.copyfile(first["cat"], dest / "cat" / f"{first['cat'].stem}_copy.jpg")
    with Image.open(first["dog"]) as im:
        ImageOps.mirror(im.convert("RGB")).save(dest / "dog" / f"{first['dog'].stem}_b.jpg", quality=92)
    with Image.open(first["bird"]) as im:
        im.convert("L").save(dest / "bird" / first["bird"].name, quality=92)
    return dest


if MY_DATASET_DIR is not None and not SMOKE:
    my_source, my_root = "yours", find_dataset(MY_DATASET_DIR)
else:
    my_source, my_root = "sample", build_my_sample(MY_SAMPLE_DIR, MY_SAMPLE_PER_CLASS, SEED)
    print("MY_DATASET_DIR is None: running on the park sample. Paste your path above to run on yours.")
my_files = list_folder_per_class(my_root)
for _note in folder_notes(my_root, my_files):
    print(_note)

if len(my_files) > MY_MAX_IMAGES:
    _n_all = len(my_files)
    my_files = pd.concat([g.sample(max(1, round(len(g) * MY_MAX_IMAGES / _n_all)), random_state=SEED)
                          for _, g in my_files.groupby("class")])
    print(f"⚠️ {_n_all} images found: the checks run on a stratified sample of {len(my_files)}.")
my_files = my_files.sort_values(["class", "file"]).reset_index(drop=True)
MY_CLASSES = sorted(my_files["class"].unique())
# Park classes keep their colour; any other class name borrows one from the same palette (L-003).
_palette = list(CLASS_COLORS.values())
MY_COLORS = {c: CLASS_COLORS.get(c, _palette[i % len(_palette)]) for i, c in enumerate(MY_CLASSES)}
print(f"✅ source: {my_source} · {my_root} · {len(my_files)} images in {len(MY_CLASSES)} class folders: "
      + ", ".join(f"{c} ({n})" for c, n in my_files["class"].value_counts().reindex(MY_CLASSES).items()))

_shown = MY_CLASSES[:8]  # a first look: three images per class (at most eight classes)
fig, axes = plt.subplots(len(_shown), 3, figsize=(7, 2.2 * len(_shown)), squeeze=False)
for r, c in enumerate(_shown):
    _of_class = my_files[my_files["class"] == c].head(3)
    for k in range(3):
        ax = axes[r, k]
        ax.axis("off")
        if k < len(_of_class):
            try:
                ax.imshow(load_thumb(_of_class.iloc[k]["path"], 192))
            except Exception:  # an unreadable file: check 2 lists it
                ax.text(0.5, 0.5, "cannot be opened", ha="center", va="center", transform=ax.transAxes)
            _name = Path(_of_class.iloc[k]["file"]).name
            ax.set_title(f"{c} · {_name if len(_name) <= 24 else _name[:20] + '…'}", fontsize=8)
fig.suptitle("A first look: are these your images and your classes?", color=INK)
fig.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Detection or segmentation project, and big datasets</div>
<p style="margin:6px 0 0 0;">Put all images in one folder inside <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">MY_DATASET_DIR</code> (e.g. <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/project/all/</code>). The class counts then say
nothing, but sizes, modes, copies and the split still do (the box and mask checks are sections 7 and 8).
Images deeper inside a class folder count, other files are listed and skipped. Datasets above
<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">MY_MAX_IMAGES</code> photos are sampled down, keeping the class shares, so the checks stay at a minute or two.</p>
</div>""")

Four checks run next and end in one report card. Check 1 counts the images per folder, as section 4 did for
the park: the **ratio** is the
largest class divided by the smallest (in the park about 130 cats against 50 sheep, 2 : 1 to 3 : 1).

In [ ]:
# @title Check 1: images per class, and the largest : smallest ratio { display-mode: "form" }
my_counts = my_files["class"].value_counts().reindex(MY_CLASSES)
fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(MY_CLASSES) + 2), 3.2))
ax.bar(MY_CLASSES, my_counts.to_numpy(), color=[MY_COLORS[c] for c in MY_CLASSES])
ax.set_ylabel("images")
ax.set_title(f"Images per class folder ({len(my_files)} in total)")
ax.tick_params(axis="x", rotation=30 if len(MY_CLASSES) > 6 else 0)
plt.tight_layout()
plt.show()
my_imbalance = float(my_counts.max() / my_counts.min())
print(f"largest {my_counts.idxmax()} ({my_counts.max()}) · smallest {my_counts.idxmin()} ({my_counts.min()}) "
      f"· ratio {my_imbalance:.1f} : 1")

💡 Look at the ratio: up to 3 : 1 is common; from 10 : 1 accuracy hides the small class (section 4).

Check 2 opens every file with `is_readable`, then reads its size (width × height) and its colour mode (`RGB`
stores three colour channels, `L` one gray channel).

In [ ]:
# @title Check 2: readable? size and colour mode of every file { display-mode: "form" }
# (section 3's measure_image on a small thumbnail)
_my_rows = []
for _, r in my_files.iterrows():
    ok = is_readable(r["path"])
    stats = measure_image(r["path"], thumb=64)[0] if ok else {}
    _my_rows.append({"file": r["file"], "class": r["class"], "readable": ok, "width": stats.get("width"),
                  "height": stats.get("height"), "mode": stats.get("mode", "unreadable")})
my_stats = pd.DataFrame(_my_rows)
_ok = my_stats[my_stats["readable"]]

fig, ax = plt.subplots(figsize=(5.5, 4))
for c in MY_CLASSES:
    sub = _ok[_ok["class"] == c]
    ax.scatter(sub["width"], sub["height"], s=18, alpha=0.7, color=MY_COLORS[c], label=c)
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("Image sizes per class")
ax.legend(frameon=False, fontsize=8, ncol=2 if len(MY_CLASSES) > 6 else 1)
plt.tight_layout()
plt.show()
print(f"unreadable files: {int((~my_stats['readable']).sum())}")
print("colour modes:", " · ".join(f"{m} {n}" for m, n in my_stats["mode"].value_counts().items()))
print(f"distinct sizes: {_ok.groupby(['width', 'height']).ngroups} · "
      f"long side from {int(_ok[['width', 'height']].max(axis=1).min())} "
      f"to {int(_ok[['width', 'height']].max(axis=1).max())} px")
display(my_stats[my_stats["mode"] != "RGB"].head(10))

💡 Look for a second cluster of sizes or any mode that is not `RGB`: each is a second source to decide on.

Check 3 is section 5's copy hunt: exact copies by file hash, and near, flipped and rotated copies by the
**distance** between perceptual hashes (differing bits out of 64). Pairs at or below your `HASH_THRESHOLD`
from section 5 count as copies.

In [ ]:
# @title Check 3: exact, recompressed, flipped and rotated copies { display-mode: "form" }
# (all pairs, in blocks to keep memory small)
_readable = my_stats.index[my_stats["readable"]].to_numpy()
_my_paths = my_files.loc[_readable, "path"].tolist()
_md5 = [file_md5(p) for p in _my_paths]
_variant_bits = {v: [] for v in VARIANT_NAMES}
for p in _my_paths:
    for v, im in variants(load_thumb(p, THUMB_FOR_HASH)).items():
        _variant_bits[v].append(phash(im))
_variant_bits = {v: hash_bits(h) for v, h in _variant_bits.items()}
_plain = _variant_bits["original"]
_n = len(_my_paths)
_best = np.full((_n, _n), 65, dtype=np.int16)  # smallest distance over the variants, and which variant
_best_v = np.zeros((_n, _n), dtype=np.int8)
for k, v in enumerate(VARIANT_NAMES):
    for start in range(0, _n, 256):
        d = hash_distances(_variant_bits[v][start:start + 256], _plain)
        better = d < _best[start:start + 256]
        _best[start:start + 256][better] = d[better]
        _best_v[start:start + 256][better] = k

_pairs = []
for i, j in zip(*np.nonzero(np.triu(_best <= HASH_THRESHOLD, k=1)), strict=True):
    a, b = _readable[i], _readable[j]
    kind = "exact" if _md5[i] == _md5[j] else KIND_OF_VARIANT[VARIANT_NAMES[_best_v[i, j]]]
    _pairs.append({"a": my_files.at[a, "file"], "b": my_files.at[b, "file"], "kind": kind,
                   "distance": int(_best[i, j]),
                   "same class": my_files.at[a, "class"] == my_files.at[b, "class"],
                   "ia": a, "ib": b})
my_copies = pd.DataFrame(_pairs, columns=["a", "b", "kind", "distance", "same class", "ia", "ib"])
print(f"{len(my_copies)} copy pairs at HASH_THRESHOLD = {HASH_THRESHOLD}:",
      " · ".join(f"{k} {n}" for k, n in my_copies["kind"].value_counts().items()) or "none")
print(f"pairs whose two files sit in different class folders (one label is wrong): "
      f"{int((~my_copies['same class']).sum())}")
display(my_copies.drop(columns=["ia", "ib"]).head(10))

_show = my_copies.head(4)
if len(_show):
    fig, axes = plt.subplots(len(_show), 2, figsize=(5, 2.4 * len(_show)), squeeze=False)
    for row, (_, pr) in zip(axes, _show.iterrows(), strict=True):
        for ax, idx in zip(row, [pr["ia"], pr["ib"]], strict=True):
            ax.imshow(load_thumb(my_files.at[idx, "path"], 256))
            ax.set_title(f"{my_files.at[idx, 'file']}\n{pr['kind']}", fontsize=8)
            frame(ax, MY_COLORS[my_files.at[idx, "class"]])
    plt.tight_layout()
    plt.show()

💡 Look at the pairs across class folders first: one of the two labels is wrong.

Check 4 suggests a split: it groups the copies, then makes a stratified (every class in both halves, in the
same shares), group-aware split.

In [ ]:
# @title Check 4: a split suggestion, copies grouped, stratified by class { display-mode: "form" }
# Copies form groups that stay on one side; the split is stratified by class.
import warnings

from sklearn.model_selection import StratifiedGroupKFold

_parent = list(range(len(my_files)))  # union-find: every copy pair joins its two files' groups


def _find(i: int) -> int:
    while _parent[i] != i:
        _parent[i] = _parent[_parent[i]]
        i = _parent[i]
    return i


for _, pr in my_copies.iterrows():
    _parent[_find(pr["ia"])] = _find(pr["ib"])
my_groups = np.array([_find(i) for i in range(len(my_files))])

_smallest = int(my_counts.min())
_n_splits = min(5, max(2, _smallest))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a class with fewer photos than folds only triggers a warning
    _splitter = StratifiedGroupKFold(n_splits=_n_splits, shuffle=True, random_state=SEED)
    _train_idx, _test_idx = next(_splitter.split(my_files, my_files["class"], groups=my_groups))
my_split = my_files[["file", "class"]].copy()
my_split["split"] = "train"
my_split.loc[_test_idx, "split"] = "test"
_split_copies = sum(my_split.at[pr["ia"], "split"] != my_split.at[pr["ib"], "split"]
                    for _, pr in my_copies.iterrows())
display(pd.crosstab(my_split["class"], my_split["split"]).reindex(MY_CLASSES))
_test_small = int((my_split["split"] == "test").groupby(my_split["class"]).sum().min())
print(f"test share {len(_test_idx) / len(my_files):.0%} · copy pairs across train/test: {_split_copies}")
if _test_small < 10:
    print(f"💡 the smallest class has only {_test_small} test images: prefer stratified {_n_splits}-fold "
          "cross-validation with the same groups, and report the mean and the spread over the folds.")
else:
    print("💡 a stratified 80/20 hold-out with these groups is enough; keep the test half for the end.")
print("If photos share a camera, a session, a burst or a patient, add that to the groups (section 6).")
_split_csv = DATA_DIR / "cache" / "my_split.csv"
my_split.to_csv(_split_csv, index=False)
print(f"split saved to {_split_csv}")

💡 Look at "copy pairs across train/test": it must be 0. The split is written to `data/cache/my_split.csv`,
and **Colab deletes that folder when the session ends**: download it now (Files pane → right-click the
file → Download) or copy it to your Drive, or it is gone next week.

The checklist in one place, to paste into your project notes:

In [ ]:
# @title The report card for this dataset { display-mode: "form" }
_n_unreadable = int((~my_stats["readable"]).sum())
_n_not_rgb = int(((my_stats["mode"] != "RGB") & my_stats["readable"]).sum())
my_report = [
    ("✅" if my_imbalance < 3 else "⚠️", f"classes: {len(MY_CLASSES)}, imbalance {my_imbalance:.1f} : 1"),
    ("✅" if _n_unreadable == 0 else "⚠️", f"unreadable files: {_n_unreadable}"),
    ("✅" if _n_not_rgb == 0 else "⚠️", f"files not stored as RGB: {_n_not_rgb}"),
    ("✅" if len(my_copies) == 0 else "⚠️", f"copy pairs: {len(my_copies)} "
     f"({int((~my_copies['same class']).sum())} across class folders)"),
    ("✅" if _split_copies == 0 else "⚠️", f"split: {len(_test_idx)} test / {len(_train_idx)} train, "
     f"copy pairs across the split: {_split_copies}"),
]
for mark, line in my_report:
    print(mark, line)
RESULTS["swap_in_ok"] = bool(len(my_files) > 0 and len(my_stats) == len(my_split) == len(my_files))
RESULTS["swap_in"] = {"source": my_source, "n_images": int(len(my_files)), "n_classes": len(MY_CLASSES),
                      "copies": int(len(my_copies)), "not_rgb": _n_not_rgb}

### After the lab · Read your results and answer the guiding questions

Everything above runs in class. This block is for you to do afterwards, on your own dataset: it explains how
to read each check and ends with the seven guiding questions.

**Q25.** 🟢 On the sample the folders are cat, dog, bird, horse, sheep and zebra with 20 photos each, plus
three planted problems (an exact copy of a cat, a mirrored dog, a grayscale bird). Which check caught each
one? Then run the checks on your own dataset and compare.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q25, on the sample)</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Check 1:</strong> every folder holds the same number of photos, plus the planted copies in cat and dog, so
  the ratio is close to 1 : 1. That is a property of the sample, not of the park.</li>
<li style="margin:2px 0;"><strong>Check 2:</strong> all sizes fall in the Play Dataset's few clusters (long side 640 or 500 px). The grayscale
  bird shows up as one <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">L</code> file.</li>
<li style="margin:2px 0;"><strong>Check 3:</strong> the exact copy of the cat and the mirrored dog show up as an <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">exact</code> and a <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">flipped</code> pair,
  both inside one folder. The split keeps each pair on one side.</li>
<li style="margin:2px 0;">If a check missed one of them, look at the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">HASH_THRESHOLD</code> you set in section 5.</li>
</ul>
</details>
</div>""")

💡 How to read the ratio:

- **2 or 3 to 1:** common and harmless if you watch the small class's recall.
- **10 to 1 and beyond:** accuracy hides the small class completely (section 4's ⚠️ Pitfall).
- **Always ask:** is the *expensive* class the small one?

💡 How to read the sizes and modes:

- **One tight cluster of sizes:** one source.
- **A second cluster, or one class sitting apart:** a second source, and the model may learn the source
  instead of the animal.
- **Every file that is not RGB** is listed in check 2: decide whether to convert it or drop it.

💡 Every copy pair is a decision for a human:

- **Keep one file of each pair.**
- **Look twice at pairs from different class folders:** one of the two labels is wrong.
- **Many pairs on near-identical shots** (a fixed camera, a microscope) are genuinely different photos:
  lower `HASH_THRESHOLD` in section 5 and rerun.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · The split is only a suggestion</div>
<p style="margin:6px 0 0 0;">The split comes from what the folders show. Only you know what else ties photos together, such as one
camera, one day, one patient; section 6 showed that a split which ignores such groups reports a better score
than the model will earn.</p>
</div>""")

#### The guiding questions

These questions are the notebook in short. Each one has the park's answer, and an empty line for yours.

**Q26.** 🟢 Double-click the next cell and answer each guiding question for your project dataset, using
the output of the checklist above and what you know about where the data comes from.

1. **What is the task, and which mistake is the expensive one?**
   - *The park:* classification of who is in front of the camera; the expensive mistake is a missed dog
     in the sheep paddock, so the metric is recall on dog (section 4).
   - *Your dataset:*
2. **Do the images look like what the model will see in use?**
   - *The park:* only partly. COCO has close-ups, sofas and people; a fixed paddock camera sees small
     animals on grass (section 2).
   - *Your dataset:*
3. **How balanced are the classes, and is the expensive class a small one?**
   - *The park:* cat and dog about 130 photos each, sheep about 50, roughly 2 : 1 to 3 : 1. Dog is not
     rare, but accuracy would still hide a weak recall on dog (section 4).
   - *Your dataset:*
4. **Which sizes, aspect ratios and colour modes are there, and what must be resized or converted?**
   - *The park:* nearly every photo has a long side of 640 or 500 px, in landscape and portrait; files
     that are not RGB get converted (section 3).
   - *Your dataset:*
5. **Are there copies, flipped or rotated copies, unreadable or odd files, or doubtful labels?**
   - *The park:* a few copies, removed before splitting (section 5's "removed N, kept M"); photos with
     several animals make the label ambiguous (section 4).
   - *Your dataset:*
6. **What ties photos together, and which split keeps those groups apart?**
   - *The park:* COCO has no groups, so a stratified split after deduplication; the trail cameras have
     bursts and locations, so they are split by camera (section 6).
   - *Your dataset:*
7. **What must be fixed before training?**
   - *The park:* drop the copies, convert the files that are not RGB, decide the label of multi-animal
     photos, and resize without distorting the aspect ratio (section 3's 🔀 Callout).
   - *Your dataset:*

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you get a new folder-per-class dataset</strong> → <strong>do</strong> run the four checks (classes, sizes and modes,
  copies, split) before anything else; they take a few minutes.</li>
<li style="margin:2px 0;"><strong>When a check lists files</strong> (not RGB, copy pairs, pairs across class folders) → <strong>do</strong> decide per file:
  convert, keep one, or fix the label.</li>
<li style="margin:2px 0;"><strong>When you leave</strong> → <strong>do</strong> download <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">data/cache/my_split.csv</code> (Colab deletes it) and bring it, with your
  answers to the guiding questions,
  to next week's training.</li>
</ul>
</div>""")